# NB-12_salidas_cnf_anual_y_cofog

Process Flow SAS: **Salidas CNF anual y COFOG** — `ProcessFlowContainer-XcLdWCRvPvWY7qFu`

In [ ]:
# ========= Parámetros =========
# Variables macro del SAS original. El .egp NO las define (venían del
# entorno SAS): su valor sale de la entrevista B4 o de
# project_config.yaml → run.macro_params, o se inyecta acá
# (celda 'parameters' de papermill).

Year = None  # &Year — nadie declaró su valor

faltantes = [n for n, v in {"Year": Year}.items() if v is None]
if faltantes:
    raise ValueError(f"Parámetros sin valor: {faltantes}")

In [ ]:
# ========= Celda 1: Configuración =========
import pandas as pd
import numpy as np
import os
import sqlalchemy
import datetime
from pathlib import Path
from sqlalchemy import text

# Conexión a BD — la define el orquestador al ejecutar
engine = sqlalchemy.create_engine(os.environ["SASMIG_DB_URL"], pool_pre_ping=True)
# Sesión de BD del notebook — espejo de la sesión WORK de SAS: las
# tablas temporales #tmp viven en ESTA conexión y mueren al cerrar el
# kernel. AUTOCOMMIT: cada statement commitea, como los pasos de SAS.
work_conn = engine.connect().execution_options(isolation_level="AUTOCOMMIT")

# Logging liviano de resultados — aprobado en la entrevista (Fase 4)
_LOG_PATH = Path("log") / "NB-12_salidas_cnf_anual_y_cofog.log"
_LOG_PATH.parent.mkdir(parents=True, exist_ok=True)
def _log(label, value=None):
    """Una línea por celda: imprime y persiste. Jamás rompe la corrida."""
    try:
        if hasattr(value, "shape"):
            detail = f"{value.shape[0]} filas x {value.shape[1]} cols"
        elif isinstance(value, int):
            detail = f"{value} filas"
        elif value is None:
            detail = "ok"
        else:
            detail = str(value)
        line = f"[{datetime.datetime.now():%Y-%m-%d %H:%M:%S}] {label}: {detail}"
        print(line)
        with open(_LOG_PATH, "a", encoding="utf-8") as fh:
            fh.write(line + "\n")
    except Exception:
        pass  # el log nunca puede tumbar el notebook
with open(_LOG_PATH, "a", encoding="utf-8") as _fh:
    _fh.write(f"\n=== corrida {datetime.datetime.now():%Y-%m-%d %H:%M:%S} ===\n")


## 1- TCons Mixtas_M_CGR

Redistribuye la ejecución presupuestaria de transacciones mixtas (M) del año según la estructura de muestra y de expansión, generando los montos desagregados y agregados para los reportes CNF/COFOG

*confianza: medium · verificador: approve · SAS: PROC SQL multi-paso: filtro por año, LEFT JOINs múltiples con agregación ponderada, UNION ALL de muestra+expandido, DELETE de ajuste y agregaciones de salida (mensual/anual)*

In [ ]:
# ========= 1- TCons Mixtas_M_CGR =========
# /*FILTRA DATOS DE EJECUCION CGR DEL AÑO*/
work_conn.execute(text("DROP TABLE IF EXISTS #ejec_cgr_y"))
sql_ejec_cgr_y = f"""
SELECT *
INTO #ejec_cgr_y
FROM GOBGENER.dbo.EJECUCION
WHERE AÑO = {int(Year)}
"""
res = work_conn.execute(text(sql_ejec_cgr_y))
# DATA step DROP=: columnas de presupuesto/percibido que no se usan en este nodo
work_conn.execute(text(
    "ALTER TABLE #ejec_cgr_y DROP COLUMN "
    "PPTO_INICIAL, PPTO_ACTUAL, PERCIB_PAG, POR_PERCIB_PAG, "
    "N_ITEM, N_ASIGNACION, C_SUBASIGNACION, N_SUBASIGNACION"
))
_log("#ejec_cgr_y", res.rowcount)


In [ ]:
# /* CREA TABLA DE REDISTRIBUCION DE MUESTRA DE TRANSACCIONES MIXTAS (M) SEGUN ESTRUCTURA, PARA EJECUCION AÑO t */
work_conn.execute(text("DROP TABLE IF EXISTS #ejec_muestra_mixtas_m"))
sql_ejec_muestra_mixtas_m = """
SELECT
    t2.SUBSECTOR, t2.MONEDA, t2.AÑO, t2.MES, t1.ENTIDAD AS C_ENTIDAD,
    t2.C_PARTIDA, t2.N_PARTIDA, t2.C_CAPITULO, t2.N_CAPITULO,
    t2.C_PROGRAMA, t2.N_PROGRAMA, t2.C_TIPO_CUENTA, t2.C_CUENTA, t2.N_CUENTA,
    t2.C_ITEM, t2.C_ASIGNACION, t2.C_ANALITICO, t2.N_ANALITICO,
    t1.C_SCN, t1.C_CAGENTE, t1.C_CUP, t2.DEVENG,
    SUM(t2.DEVENG / 1000000.0 * t1.POND) AS DEVENGO
INTO #ejec_muestra_mixtas_m
FROM GOBGENER.dbo.EST_MIXTAS_M t1
LEFT JOIN #ejec_cgr_y t2
    ON t2.C_ENTIDAD = t1.ENTIDAD
   AND t2.C_ASIGNACION = t1.C_ASIGNACION
   AND t2.C_ANALITICO = t1.C_ANALITICO
   AND t2.C_CUENTA = t1.C_CUENTA
   AND t2.C_ITEM = t1.C_ITEM
   AND t2.MES = t1.MES
WHERE t2.C_CUENTA IN ('24', '33')
  AND t2.C_ITEM IN ('03', '01', '06', '08', '09')
  AND t1.ENTIDAD NOT IN (5601, 10201)
  AND t2.DEVENG IS NOT NULL
GROUP BY
    t2.SUBSECTOR, t2.MONEDA, t2.AÑO, t2.MES, t1.ENTIDAD,
    t2.C_PARTIDA, t2.N_PARTIDA, t2.C_CAPITULO, t2.N_CAPITULO,
    t2.C_PROGRAMA, t2.N_PROGRAMA, t2.C_TIPO_CUENTA, t2.C_CUENTA, t2.N_CUENTA,
    t2.C_ITEM, t2.C_ASIGNACION, t2.C_ANALITICO, t2.N_ANALITICO,
    t1.C_SCN, t1.C_CAGENTE, t1.C_CUP, t2.DEVENG
"""
res = work_conn.execute(text(sql_ejec_muestra_mixtas_m))
# DATA step DROP=DEVENG: se descarta el monto original, queda solo el ponderado
work_conn.execute(text("ALTER TABLE #ejec_muestra_mixtas_m DROP COLUMN DEVENG"))
_log("#ejec_muestra_mixtas_m", res.rowcount)


In [ ]:
# /* CREA TABLA DE ANALITICOS DE MIXTAS (M) A EXPANDIR (FUERA DE MUESTRA), SEGUN EJECUCION AÑO t */
work_conn.execute(text("DROP TABLE IF EXISTS #cruce_ejec_exp_mixtas_m"))
sql_cruce_ejec_exp_mixtas_m = """
SELECT
    t2.SUBSECTOR, t2.MONEDA, t2.AÑO, t2.MES, t2.C_ENTIDAD,
    t2.C_PARTIDA, t2.N_PARTIDA, t2.C_CAPITULO, t2.N_CAPITULO,
    t2.C_PROGRAMA, t2.N_PROGRAMA, t2.C_TIPO_CUENTA, t2.N_CUENTA,
    t2.N_ANALITICO, t2.DEVENG,
    t1.C_CUENTA, t1.C_ITEM, t1.C_ASIGNACION, t1.C_ANALITICO, t1.Clasif_Tcons
INTO #cruce_ejec_exp_mixtas_m
FROM GOBGENER.dbo.ANALIT_TCONS t1
LEFT JOIN #ejec_cgr_y t2
    ON t2.C_ENTIDAD = t1.ENTIDAD
   AND t2.C_ASIGNACION = t1.C_ASIGNACION
   AND t2.C_ANALITICO = t1.C_ANALITICO
   AND t2.C_CUENTA = t1.C_CUENTA
   AND t2.C_ITEM = t1.C_ITEM
WHERE t1.M_MUESTRA = 0
  AND t2.C_ENTIDAD NOT IN (5601, 10201)
  AND t1.Clasif_Tcons = 'M'
  AND t2.DEVENG IS NOT NULL
"""
res = work_conn.execute(text(sql_cruce_ejec_exp_mixtas_m))
_log("#cruce_ejec_exp_mixtas_m", res.rowcount)


In [ ]:
# /* CREA TABLA DE REDISTRIBUCION EXPANDIDA SEGUN ESTRUCTURA DE MUESTRA, PARA RESTO DE TRANSACCIONES MIXTAS (M),
#    PARA EJECUCION AÑO t */
work_conn.execute(text("DROP TABLE IF EXISTS #ejec_exp_mixtas_m_prelim"))
sql_ejec_exp_mixtas_m_prelim = """
SELECT
    t2.SUBSECTOR, t2.MONEDA, t2.AÑO, t2.MES, t2.C_ENTIDAD,
    t2.C_PARTIDA, t2.N_PARTIDA, t2.C_CAPITULO, t2.N_CAPITULO,
    t2.C_PROGRAMA, t2.N_PROGRAMA, t2.C_TIPO_CUENTA, t2.C_CUENTA, t2.N_CUENTA,
    t2.C_ITEM, t2.C_ASIGNACION, t2.C_ANALITICO, t2.N_ANALITICO, t2.DEVENG,
    t2.Clasif_Tcons, t1.C_SCN, t1.C_CAGENTE, t1.C_CUP,
    SUM(t2.DEVENG) AS DATO_OR,
    SUM(t2.DEVENG / 1000000.0 * t1.POND * t3.DATO) AS DEVENGO
INTO #ejec_exp_mixtas_m_prelim
FROM GOBGENER.dbo.EST_EXP_MIXTAS_M t1
LEFT JOIN #cruce_ejec_exp_mixtas_m t2
    ON t2.C_CUENTA = t1.C_CUENTA
   AND t2.C_ITEM = t1.C_ITEM
   AND t2.C_ASIGNACION = t1.C_ASIGNACION
   AND t2.C_ANALITICO = t1.C_ANALITICO
   AND t2.MES = t1.MES
   AND t2.C_ENTIDAD = t1.ENTIDAD
LEFT JOIN #prom_mes t3
    ON t2.MONEDA = t3.MONEDA
   AND t2.MES = t3.MES
   AND t2.AÑO = t3.AÑO
WHERE t2.DEVENG IS NOT NULL
GROUP BY
    t2.SUBSECTOR, t2.MONEDA, t2.AÑO, t2.MES, t2.C_ENTIDAD,
    t2.C_PARTIDA, t2.N_PARTIDA, t2.C_CAPITULO, t2.N_CAPITULO,
    t2.C_PROGRAMA, t2.N_PROGRAMA, t2.C_TIPO_CUENTA, t2.C_CUENTA, t2.N_CUENTA,
    t2.C_ITEM, t2.C_ASIGNACION, t2.C_ANALITICO, t2.N_ANALITICO, t2.DEVENG,
    t2.Clasif_Tcons, t1.C_SCN, t1.C_CAGENTE, t1.C_CUP
"""
res = work_conn.execute(text(sql_ejec_exp_mixtas_m_prelim))
_log("#ejec_exp_mixtas_m_prelim", res.rowcount)


In [ ]:
work_conn.execute(text("DROP TABLE IF EXISTS #ejec_exp_mixtas_m"))
sql_ejec_exp_mixtas_m = """
SELECT
    SUBSECTOR, MONEDA, AÑO, MES, C_ENTIDAD, C_PARTIDA, N_PARTIDA, C_CAPITULO,
    N_CAPITULO, C_PROGRAMA, N_PROGRAMA, C_TIPO_CUENTA, C_CUENTA, N_CUENTA,
    C_ITEM, C_ASIGNACION, C_ANALITICO, N_ANALITICO, C_SCN, C_CAGENTE, C_CUP,
    DEVENGO
INTO #ejec_exp_mixtas_m
FROM #ejec_exp_mixtas_m_prelim
"""
res = work_conn.execute(text(sql_ejec_exp_mixtas_m))
_log("#ejec_exp_mixtas_m", res.rowcount)


In [ ]:
# /* Crea tabla con la ejecución total de transacciones Mixtas (M), Muestra + Resto */
work_conn.execute(text("DROP TABLE IF EXISTS #ejec_mixtas_m_total"))
sql_ejec_mixtas_m_total = """
SELECT * INTO #ejec_mixtas_m_total
FROM (
    SELECT * FROM #ejec_muestra_mixtas_m
    UNION ALL
    SELECT * FROM #ejec_exp_mixtas_m
) u
"""
res = work_conn.execute(text(sql_ejec_mixtas_m_total))
_log("#ejec_mixtas_m_total", res.rowcount)


In [ ]:
# /* Ajuste por imputación de D63 en pandemia (2020) */
res = work_conn.execute(text(
    "DELETE FROM #ejec_mixtas_m_total "
    "WHERE C_ENTIDAD = 911 AND C_CUENTA = '24' AND C_ITEM = '03' "
    "AND C_ASIGNACION = '002' AND C_ANALITICO = '03002000' "
    "AND AÑO = 2020 AND MES > 4"
))
_log("DELETE #ejec_mixtas_m_total (ajuste D63 2020)", res.rowcount)


In [ ]:
# /* SALIDA de EJECUCION TOTAL DE TRANSACCIONES MIXTAS (M) para COFOG y ANALISIS DESAGREGADO CNF anual */
# /*Desagregado mes*/
work_conn.execute(text("DROP TABLE IF EXISTS #ejec_e_mixt_aper"))
sql_ejec_e_mixt_aper = """
SELECT
    t1.SUBSECTOR, t1.MONEDA, t1.AÑO,
    CASE
        WHEN t1.MES IN (1, 2, 3) THEN 1
        WHEN t1.MES IN (4, 5, 6) THEN 2
        WHEN t1.MES IN (7, 8, 9) THEN 3
        ELSE 4
    END AS TRIM,
    t1.MES, t1.C_ENTIDAD, t1.C_PARTIDA, t1.C_CAPITULO, t1.C_PROGRAMA,
    t1.C_TIPO_CUENTA, t1.C_CUENTA, t1.C_ITEM, t1.C_ASIGNACION, t1.C_ANALITICO,
    t1.N_ANALITICO, t1.C_SCN, t1.C_CAGENTE, t1.C_CUP,
    t3.ACTIVIDAD, t3.NIVEL, t4.C_COFOG_1, t4.N_COFOG_1,
    SUM(t1.DEVENGO) AS Dato
INTO #ejec_e_mixt_aper
FROM #ejec_mixtas_m_total t1
LEFT JOIN GOBGENER.dbo.CR18_APERTURA t3 ON t1.C_ENTIDAD = t3.ENTIDAD
LEFT JOIN GOBGENER.dbo.COFOG t4
    ON t1.C_ENTIDAD = t4.ENTIDAD
   AND t1.C_PARTIDA = t4.C_PARTIDA
   AND t1.C_CAPITULO = t4.C_CAPITULO
   AND t1.C_PROGRAMA = t4.C_PROGRAMA
WHERE t1.C_SCN NOT IN ('F22', 'F29', 'F3', 'F32', 'F41', 'F42', 'F5', 'F612', 'F7', 'F71', 'F79', 'K10', 'K11', 'K9')
GROUP BY
    t1.SUBSECTOR, t1.MONEDA, t1.AÑO,
    CASE
        WHEN t1.MES IN (1, 2, 3) THEN 1
        WHEN t1.MES IN (4, 5, 6) THEN 2
        WHEN t1.MES IN (7, 8, 9) THEN 3
        ELSE 4
    END,
    t1.MES, t1.C_ENTIDAD, t1.C_PARTIDA, t1.C_CAPITULO, t1.C_PROGRAMA,
    t1.C_TIPO_CUENTA, t1.C_CUENTA, t1.C_ITEM, t1.C_ASIGNACION, t1.C_ANALITICO,
    t1.N_ANALITICO, t1.C_SCN, t1.C_CAGENTE, t1.C_CUP,
    t3.ACTIVIDAD, t3.NIVEL, t4.C_COFOG_1, t4.N_COFOG_1
"""
res = work_conn.execute(text(sql_ejec_e_mixt_aper))
_log("#ejec_e_mixt_aper", res.rowcount)


In [ ]:
# /*SALIDA DESAGREGADA ANUAL PARA COFOG*/
work_conn.execute(text("DROP TABLE IF EXISTS #ejec_e_mixt_aper_an"))
sql_ejec_e_mixt_aper_an = """
SELECT
    t1.SUBSECTOR, t1.MONEDA, t1.AÑO, t1.C_ENTIDAD, t1.C_PARTIDA, t1.C_CAPITULO,
    t1.C_PROGRAMA, t1.C_TIPO_CUENTA, t1.C_CUENTA, t1.C_ITEM, t1.C_ASIGNACION,
    t1.C_ANALITICO, t1.N_ANALITICO, t1.C_SCN, t1.C_CAGENTE, t1.C_CUP,
    t1.ACTIVIDAD, t1.NIVEL, t1.C_COFOG_1, t1.N_COFOG_1,
    SUM(t1.Dato) AS Dato
INTO #ejec_e_mixt_aper_an
FROM #ejec_e_mixt_aper t1
GROUP BY
    t1.SUBSECTOR, t1.MONEDA, t1.AÑO, t1.C_ENTIDAD, t1.C_PARTIDA, t1.C_CAPITULO,
    t1.C_PROGRAMA, t1.C_TIPO_CUENTA, t1.C_CUENTA, t1.C_ITEM, t1.C_ASIGNACION,
    t1.C_ANALITICO, t1.N_ANALITICO, t1.C_SCN, t1.C_CAGENTE, t1.C_CUP,
    t1.ACTIVIDAD, t1.NIVEL, t1.C_COFOG_1, t1.N_COFOG_1
"""
res = work_conn.execute(text(sql_ejec_e_mixt_aper_an))
_log("#ejec_e_mixt_aper_an", res.rowcount)


In [ ]:
# /* SALIDA de EJECUCION TOTAL DE TRANSACCIONES MIXTAS (M) para CNF anual, AGREGADO*/
work_conn.execute(text("DROP TABLE IF EXISTS #ejec_e_mixt"))
sql_ejec_e_mixt = """
SELECT
    t1.SUBSECTOR, t1.MONEDA, t1.AÑO, t1.TRIM, t1.MES, t1.C_TIPO_CUENTA,
    t1.C_CUENTA, t1.C_ITEM, t1.C_SCN, t1.C_CAGENTE, t1.C_CUP,
    SUM(t1.Dato) AS Dato, t1.ACTIVIDAD, t1.NIVEL
INTO #ejec_e_mixt
FROM #ejec_e_mixt_aper t1
GROUP BY
    t1.SUBSECTOR, t1.MONEDA, t1.AÑO, t1.TRIM, t1.MES, t1.C_TIPO_CUENTA,
    t1.C_CUENTA, t1.C_ITEM, t1.C_SCN, t1.C_CAGENTE, t1.ACTIVIDAD, t1.NIVEL, t1.C_CUP
"""
res = work_conn.execute(text(sql_ejec_e_mixt))
_log("#ejec_e_mixt", res.rowcount)


In [ ]:
# /* SALIDA de EJECUCION TOTAL DE TRANSACCIONES MIXTAS (M) para CNF anual, AGREGADA */
work_conn.execute(text("DROP TABLE IF EXISTS #ejec_mixtas_m_an"))
sql_ejec_mixtas_m_an = """
SELECT
    t1.SUBSECTOR, t1.MONEDA, t1.AÑO, t1.C_TIPO_CUENTA, t1.C_CUENTA, t1.C_ITEM,
    t1.C_SCN, t1.C_CAGENTE, t1.C_CUP, t1.ACTIVIDAD, t1.NIVEL,
    SUM(t1.Dato) AS Dato
INTO #ejec_mixtas_m_an
FROM #ejec_e_mixt_aper t1
GROUP BY
    t1.SUBSECTOR, t1.MONEDA, t1.AÑO, t1.C_TIPO_CUENTA, t1.C_CUENTA, t1.C_ITEM,
    t1.C_SCN, t1.C_CAGENTE, t1.C_CUP, t1.ACTIVIDAD, t1.NIVEL
"""
res = work_conn.execute(text(sql_ejec_mixtas_m_an))
_log("#ejec_mixtas_m_an", res.rowcount)


In [ ]:
# PROC SQL DROP TABLE: limpia las tablas intermedias de la sesión (no las finales)
for t in ["#ejec_muestra_mixtas_m", "#cruce_ejec_exp_mixtas_m", "#ejec_exp_mixtas_m_prelim",
          "#ejec_exp_mixtas_m", "#ejec_mixtas_m_total"]:
    work_conn.execute(text(f"DROP TABLE IF EXISTS {t}"))


## 2- TCons MGCG_CGR

Redistribuye y agrega las transacciones MGCG (muestra + resto expandido) por entidad, partida, ítem, cuenta y clasificación SCN/COFOG, para el análisis desagregado y anual del CNF del año de ejecución

*confianza: medium · verificador: approve · SAS: PROC SQL: cadena de CREATE TABLE con LEFT JOIN, SUM ponderado (POND) y GROUP BY, más DATA steps de DROP de columnas, UNION y limpieza final con DROP TABLE*

In [ ]:
# ========= 2- TCons MGCG_CGR =========
# /* CREA TABLA DE REDISTRIBUCION DE MUESTRA DE TRANSACCIONES MGCG SEGUN ESTRUCTURA, PARA EJECUCION AÑO t */
# DATA EJEC_MUESTRA_MGCG; SET EJEC_MUESTRA_MGCG (DROP=DEVENG); RUN; -> se fusiona acá: se omite el DEVENG
# crudo del SELECT y se conserva solo el DEVENGO agregado
work_conn.execute(text("DROP TABLE IF EXISTS #ejec_muestra_mgcg"))
sql_ejec_muestra_mgcg = """
SELECT
    t2.SUBSECTOR,
    t2.MONEDA,
    t2.MES,
    t2.AÑO,
    t1.ENTIDAD AS C_ENTIDAD,
    t2.C_PARTIDA,
    t2.N_PARTIDA,
    t2.C_CAPITULO,
    t2.N_CAPITULO,
    t2.C_PROGRAMA,
    t2.N_PROGRAMA,
    t2.C_TIPO_CUENTA,
    t2.C_CUENTA,
    t2.N_CUENTA,
    t2.C_ITEM,
    t2.C_ASIGNACION,
    t2.C_ANALITICO,
    t2.N_ANALITICO,
    t1.C_SCN,
    t1.C_CAGENTE,
    SUM(t2.DEVENG / 1000000.0 * t1.POND) AS DEVENGO
INTO #ejec_muestra_mgcg
FROM GOBGENER.dbo.EST_MGCG t1
LEFT JOIN #ejec_cgr_y t2
    ON t2.C_ENTIDAD = t1.ENTIDAD
   AND t2.C_ASIGNACION = t1.C_ASIGNACION
   AND t2.C_ANALITICO = t1.C_ANALITICO
   AND t2.C_CUENTA = t1.C_CUENTA
   AND t2.C_ITEM = t1.C_ITEM
   AND t2.MES = t1.MES
WHERE t2.C_CUENTA IN ('24') AND t2.C_ITEM IN ('03') AND t1.ENTIDAD NOT IN (5601, 10201) AND t2.DEVENG IS NOT NULL
GROUP BY t2.SUBSECTOR, t2.MONEDA, t2.MES, t2.AÑO, t1.ENTIDAD,
    t2.C_PARTIDA, t2.N_PARTIDA, t2.C_CAPITULO, t2.N_CAPITULO, t2.C_PROGRAMA, t2.N_PROGRAMA,
    t2.C_TIPO_CUENTA, t2.C_CUENTA, t2.N_CUENTA, t2.C_ITEM, t2.C_ASIGNACION, t2.C_ANALITICO, t2.N_ANALITICO,
    t1.C_SCN, t1.C_CAGENTE, t2.DEVENG
"""
work_conn.execute(text(sql_ejec_muestra_mgcg))


In [ ]:
# /* CREA TABLA DE ANALITICOS DE MGCG A EXPANDIR (FUERA DE MUESTRA), SEGUN EJECUCION AÑO t */
# Nombres duplicados entre t1 (ANALIT_TCONS) y t2 (#ejec_cgr_y) para C_CUENTA/C_ITEM/C_ASIGNACION/C_ANALITICO:
# se conserva la versión de t1, igual que el resultado final de PROC SQL (el join + WHERE garantiza que coincidan)
work_conn.execute(text("DROP TABLE IF EXISTS #cruce_ejec_exp_mgcg"))
sql_cruce_ejec_exp_mgcg = """
SELECT
    t2.SUBSECTOR,
    t2.MONEDA,
    t2.AÑO,
    t2.MES,
    t2.C_ENTIDAD,
    t2.C_PARTIDA,
    t2.N_PARTIDA,
    t2.C_CAPITULO,
    t2.N_CAPITULO,
    t2.C_PROGRAMA,
    t2.N_PROGRAMA,
    t2.C_TIPO_CUENTA,
    t2.N_CUENTA,
    t2.N_ANALITICO,
    t2.DEVENG,
    t1.C_CUENTA,
    t1.C_ITEM,
    t1.C_ASIGNACION,
    t1.C_ANALITICO,
    t1.Clasif_Tcons
INTO #cruce_ejec_exp_mgcg
FROM GOBGENER.dbo.ANALIT_TCONS t1
LEFT JOIN #ejec_cgr_y t2
    ON t2.C_ENTIDAD = t1.ENTIDAD
   AND t2.C_ASIGNACION = t1.C_ASIGNACION
   AND t2.C_ANALITICO = t1.C_ANALITICO
   AND t2.C_CUENTA = t1.C_CUENTA
   AND t2.C_ITEM = t1.C_ITEM
WHERE t1.M_MUESTRA = 0 AND t1.Clasif_Tcons = 'MGCG' AND t2.C_ENTIDAD NOT IN (5601, 10201) AND t2.DEVENG IS NOT NULL
"""
work_conn.execute(text(sql_cruce_ejec_exp_mgcg))


In [ ]:
# /* CREA TABLA DE REDISTRIBUCION EXPANDIDA SEGUN ESTRUCTURA DE MUESTRA, PARA RESTO DE TRANSACCIONES MGCG,
# PARA EJECUCION AÑO t */
work_conn.execute(text("DROP TABLE IF EXISTS #ejec_exp_mgcg_prelim"))
sql_ejec_exp_mgcg_prelim = """
SELECT
    t2.SUBSECTOR,
    t2.MONEDA,
    t2.AÑO,
    t2.MES,
    t2.C_ENTIDAD,
    t2.C_PARTIDA,
    t2.N_PARTIDA,
    t2.C_CAPITULO,
    t2.N_CAPITULO,
    t2.C_PROGRAMA,
    t2.N_PROGRAMA,
    t2.C_TIPO_CUENTA,
    t2.C_CUENTA,
    t2.N_CUENTA,
    t2.C_ITEM,
    t2.C_ASIGNACION,
    t2.C_ANALITICO,
    t2.N_ANALITICO,
    t2.DEVENG,
    t2.Clasif_Tcons,
    t1.C_SCN,
    t1.C_CAGENTE,
    SUM(t2.DEVENG) AS DATO_OR,
    SUM(t2.DEVENG / 1000000.0 * t1.POND * t3.DATO) AS DEVENGO
INTO #ejec_exp_mgcg_prelim
FROM GOBGENER.dbo.EST_EXP_MGCG t1
LEFT JOIN #cruce_ejec_exp_mgcg t2
    ON t2.C_CUENTA = t1.C_CUENTA AND t2.C_ITEM = t1.C_ITEM AND t2.MES = t1.MES
LEFT JOIN #prom_mes t3
    ON t2.MONEDA = t3.MONEDA AND t2.MES = t3.MES AND t2.AÑO = t3.AÑO
WHERE t2.DEVENG IS NOT NULL
GROUP BY t2.SUBSECTOR, t2.MONEDA, t2.AÑO, t2.MES, t2.C_ENTIDAD, t2.C_PARTIDA, t2.N_PARTIDA,
    t2.C_CAPITULO, t2.N_CAPITULO, t2.C_PROGRAMA, t2.N_PROGRAMA, t2.C_TIPO_CUENTA, t2.C_CUENTA,
    t2.N_CUENTA, t2.C_ITEM, t2.C_ASIGNACION, t2.C_ANALITICO, t2.N_ANALITICO, t2.DEVENG, t2.Clasif_Tcons,
    t1.C_SCN, t1.C_CAGENTE
"""
work_conn.execute(text(sql_ejec_exp_mgcg_prelim))


In [ ]:
# DATA EJEC_EXP_MGCG; SET EJEC_EXP_MGCG_prelim (DROP=DEVENG DATO_OR Clasif_Tcons); RUN;
work_conn.execute(text("DROP TABLE IF EXISTS #ejec_exp_mgcg"))
sql_ejec_exp_mgcg = """
SELECT
    SUBSECTOR, MONEDA, AÑO, MES, C_ENTIDAD, C_PARTIDA, N_PARTIDA, C_CAPITULO, N_CAPITULO,
    C_PROGRAMA, N_PROGRAMA, C_TIPO_CUENTA, C_CUENTA, N_CUENTA, C_ITEM, C_ASIGNACION, C_ANALITICO,
    N_ANALITICO, C_SCN, C_CAGENTE, DEVENGO
INTO #ejec_exp_mgcg
FROM #ejec_exp_mgcg_prelim
"""
work_conn.execute(text(sql_ejec_exp_mgcg))


In [ ]:
# /* Crea tabla con la ejecución total de transacciones MGCG, Muestra + Resto */
work_conn.execute(text("DROP TABLE IF EXISTS #ejec_mgcg_total"))
sql_ejec_mgcg_total = """
SELECT SUBSECTOR, MONEDA, AÑO, MES, C_ENTIDAD, C_PARTIDA, N_PARTIDA, C_CAPITULO, N_CAPITULO,
    C_PROGRAMA, N_PROGRAMA, C_TIPO_CUENTA, C_CUENTA, N_CUENTA, C_ITEM, C_ASIGNACION, C_ANALITICO,
    N_ANALITICO, C_SCN, C_CAGENTE, DEVENGO
INTO #ejec_mgcg_total
FROM #ejec_muestra_mgcg
UNION ALL
SELECT SUBSECTOR, MONEDA, AÑO, MES, C_ENTIDAD, C_PARTIDA, N_PARTIDA, C_CAPITULO, N_CAPITULO,
    C_PROGRAMA, N_PROGRAMA, C_TIPO_CUENTA, C_CUENTA, N_CUENTA, C_ITEM, C_ASIGNACION, C_ANALITICO,
    N_ANALITICO, C_SCN, C_CAGENTE, DEVENGO
FROM #ejec_exp_mgcg
"""
work_conn.execute(text(sql_ejec_mgcg_total))


In [ ]:
# /* SALIDA de EJECUCION TOTAL DE TRANSACCIONES MGCG para COFOG y ANALISIS DESAGREGADO CNF anual */
# /*Desagregado mes*/
work_conn.execute(text("DROP TABLE IF EXISTS #ejec_e_mgcg_aper"))
sql_ejec_e_mgcg_aper = """
SELECT
    t1.SUBSECTOR,
    t1.MONEDA,
    t1.AÑO,
    CASE WHEN t1.MES IN (1, 2, 3) THEN 1
         WHEN t1.MES IN (4, 5, 6) THEN 2
         WHEN t1.MES IN (7, 8, 9) THEN 3
         ELSE 4 END AS [TRIM],
    t1.MES,
    t1.C_ENTIDAD,
    t1.C_PARTIDA,
    t1.C_CAPITULO,
    t1.C_PROGRAMA,
    t1.C_TIPO_CUENTA,
    t1.C_CUENTA,
    t1.C_ITEM,
    t1.C_ASIGNACION,
    t1.C_ANALITICO,
    t1.N_ANALITICO,
    t1.C_SCN,
    t1.C_CAGENTE,
    t3.ACTIVIDAD,
    t3.NIVEL,
    t4.C_COFOG_1,
    t4.N_COFOG_1,
    SUM(t1.DEVENGO) AS DATO
INTO #ejec_e_mgcg_aper
FROM #ejec_mgcg_total t1
LEFT JOIN GOBGENER.dbo.CR18_APERTURA t3 ON t1.C_ENTIDAD = t3.ENTIDAD
LEFT JOIN GOBGENER.dbo.COFOG t4
    ON t1.C_ENTIDAD = t4.ENTIDAD AND t1.C_PARTIDA = t4.C_PARTIDA
   AND t1.C_CAPITULO = t4.C_CAPITULO AND t1.C_PROGRAMA = t4.C_PROGRAMA
WHERE t1.C_SCN NOT IN ('F22','F29','F3','F32','F41','F42','F5','F612','F7','F71','F79','K10','K11','K9')
GROUP BY t1.SUBSECTOR, t1.MONEDA, t1.AÑO,
    CASE WHEN t1.MES IN (1, 2, 3) THEN 1
         WHEN t1.MES IN (4, 5, 6) THEN 2
         WHEN t1.MES IN (7, 8, 9) THEN 3
         ELSE 4 END,
    t1.MES, t1.C_ENTIDAD, t1.C_PARTIDA, t1.C_CAPITULO, t1.C_PROGRAMA, t1.C_TIPO_CUENTA, t1.C_CUENTA,
    t1.C_ITEM, t1.C_ASIGNACION, t1.C_ANALITICO, t1.N_ANALITICO, t1.C_SCN, t1.C_CAGENTE,
    t3.ACTIVIDAD, t3.NIVEL, t4.C_COFOG_1, t4.N_COFOG_1
"""
work_conn.execute(text(sql_ejec_e_mgcg_aper))


In [ ]:
# /*Agregado año, SALIDA COFOG ANALISIS DESAGREGADO CNF anual */
work_conn.execute(text("DROP TABLE IF EXISTS #ejec_e_mgcg_aper_an"))
sql_ejec_e_mgcg_aper_an = """
SELECT
    t1.SUBSECTOR, t1.MONEDA, t1.AÑO, t1.C_ENTIDAD, t1.C_PARTIDA, t1.C_CAPITULO, t1.C_PROGRAMA,
    t1.C_TIPO_CUENTA, t1.C_CUENTA, t1.C_ITEM, t1.C_ASIGNACION, t1.C_ANALITICO, t1.N_ANALITICO,
    t1.C_SCN, t1.C_CAGENTE, t1.ACTIVIDAD, t1.NIVEL, t1.C_COFOG_1, t1.N_COFOG_1,
    SUM(t1.DATO) AS DATO
INTO #ejec_e_mgcg_aper_an
FROM #ejec_e_mgcg_aper t1
GROUP BY t1.SUBSECTOR, t1.MONEDA, t1.AÑO, t1.C_ENTIDAD, t1.C_PARTIDA, t1.C_CAPITULO, t1.C_PROGRAMA,
    t1.C_TIPO_CUENTA, t1.C_CUENTA, t1.C_ITEM, t1.C_ASIGNACION, t1.C_ANALITICO, t1.N_ANALITICO,
    t1.C_SCN, t1.C_CAGENTE, t1.ACTIVIDAD, t1.NIVEL, t1.C_COFOG_1, t1.N_COFOG_1
"""
work_conn.execute(text(sql_ejec_e_mgcg_aper_an))


In [ ]:
# /* SALIDA de EJECUCION TOTAL DE TRANSACCIONES MGCG para CNF anual */
work_conn.execute(text("DROP TABLE IF EXISTS #ejec_e_mgcg"))
sql_ejec_e_mgcg = """
SELECT
    t1.SUBSECTOR, t1.MONEDA, t1.AÑO, t1.[TRIM], t1.MES, t1.C_TIPO_CUENTA, t1.C_CUENTA, t1.C_ITEM,
    t1.C_SCN, t1.C_CAGENTE, SUM(t1.DATO) AS DATO, t1.ACTIVIDAD, t1.NIVEL
INTO #ejec_e_mgcg
FROM #ejec_e_mgcg_aper t1
GROUP BY t1.SUBSECTOR, t1.MONEDA, t1.AÑO, t1.[TRIM], t1.MES, t1.C_TIPO_CUENTA, t1.C_CUENTA, t1.C_ITEM,
    t1.C_SCN, t1.C_CAGENTE, t1.ACTIVIDAD, t1.NIVEL
"""
work_conn.execute(text(sql_ejec_e_mgcg))


In [ ]:
# /* SALIDA de EJECUCION TOTAL DE TRANSACCIONES MGCG para CNF anual */
work_conn.execute(text("DROP TABLE IF EXISTS #ejec_mgcg_an"))
sql_ejec_mgcg_an = """
SELECT
    t1.SUBSECTOR, t1.MONEDA, t1.AÑO, t1.C_TIPO_CUENTA, t1.C_CUENTA, t1.C_ITEM, t1.C_SCN, t1.C_CAGENTE,
    t1.ACTIVIDAD, t1.NIVEL, SUM(t1.DATO) AS DATO
INTO #ejec_mgcg_an
FROM #ejec_e_mgcg_aper t1
GROUP BY t1.SUBSECTOR, t1.MONEDA, t1.AÑO, t1.C_TIPO_CUENTA, t1.C_CUENTA, t1.C_ITEM, t1.C_SCN, t1.C_CAGENTE,
    t1.ACTIVIDAD, t1.NIVEL
"""
work_conn.execute(text(sql_ejec_mgcg_an))


In [ ]:
# PROC SQL; DROP TABLE EJEC_MUESTRA_MGCG,CRUCE_EJEC_EXP_MGCG,EJEC_EXP_MGCG_prelim,EJEC_EXP_MGCG,EJEC_MGCG_TOTAL; QUIT;
for _t in ["#ejec_muestra_mgcg", "#cruce_ejec_exp_mgcg", "#ejec_exp_mgcg_prelim", "#ejec_exp_mgcg", "#ejec_mgcg_total"]:
    work_conn.execute(text(f"DROP TABLE IF EXISTS {_t}"))


## 3- TCons MM_CGR

Calcula el devengo de transacciones MM del año (muestra directa + expansión del resto según ponderadores y factor de crecimiento), excluyendo entidades 5601/10201, y lo consolida por entidad, cuenta, clasificación COFOG y periodo (mensual, trimestral y anual) para el reporte CNF

*confianza: medium · verificador: approve · SAS: PROC SQL: redistribución de muestra y expansión de transacciones MM vía LEFT JOIN/GROUP BY sobre tablas temporales de sesión, concatenación (SET) y agregaciones sucesivas por mes/trimestre/año*

In [ ]:
# ========= 3- TCons MM_CGR =========
# /* CREA TABLA DE REDISTRIBUCION DE MUESTRA DE TRANSACCIONES MM SEGUN ESTRUCTURA, PARA EJECUCION AÑO t */
work_conn.execute(text("DROP TABLE IF EXISTS #ejec_muestra_mm"))
sql_ejec_muestra_mm = """
SELECT
    t2.SUBSECTOR,
    t2.MONEDA,
    t2.MES,
    t2.AÑO,
    t1.ENTIDAD AS C_ENTIDAD,
    t2.C_PARTIDA,
    t2.N_PARTIDA,
    t2.C_CAPITULO,
    t2.N_CAPITULO,
    t2.C_PROGRAMA,
    t2.N_PROGRAMA,
    t2.C_TIPO_CUENTA,
    t2.C_CUENTA,
    t2.N_CUENTA,
    t2.C_ITEM,
    t2.C_ASIGNACION,
    t2.C_ANALITICO,
    t2.N_ANALITICO,
    t2.DEVENG,
    t1.C_SCN,
    t1.C_CAGENTE,
    SUM(t2.DEVENG / 1000000.0 * t1.POND) AS DEVENGO
INTO #ejec_muestra_mm
FROM GOBGENER.dbo.EST_MM t1
LEFT JOIN #ejec_cgr_y t2
    ON t2.C_ENTIDAD = t1.ENTIDAD
   AND t2.C_ASIGNACION = t1.C_ASIGNACION
   AND t2.C_ANALITICO = t1.C_ANALITICO
   AND t2.C_CUENTA = t1.C_CUENTA
   AND t2.C_ITEM = t1.C_ITEM
   AND t2.MES = t1.MES
WHERE t2.C_CUENTA IN ('24', '33')
  AND t2.C_ITEM IN ('01', '03', '08', '09')
  AND t2.DEVENG IS NOT NULL
  AND t1.ENTIDAD NOT IN (5601, 10201)
GROUP BY
    t2.SUBSECTOR, t2.MONEDA, t2.MES, t2.AÑO, t1.ENTIDAD,
    t2.C_PARTIDA, t2.N_PARTIDA, t2.C_CAPITULO, t2.N_CAPITULO,
    t2.C_PROGRAMA, t2.N_PROGRAMA, t2.C_TIPO_CUENTA, t2.C_CUENTA,
    t2.N_CUENTA, t2.C_ITEM, t2.C_ASIGNACION, t2.C_ANALITICO,
    t2.N_ANALITICO, t1.C_SCN, t1.C_CAGENTE, t2.DEVENG
"""
work_conn.execute(text(sql_ejec_muestra_mm))
# DATA EJEC_MUESTRA_MM; SET EJEC_MUESTRA_MM(DROP=DEVENG); -> descarta la columna fuente ya usada en el cálculo
work_conn.execute(text("ALTER TABLE #ejec_muestra_mm DROP COLUMN DEVENG"))


In [ ]:
# /* CREA TABLA DE ANALITICOS DE MM A EXPANDIR (FUERA DE MUESTRA), SEGUN EJECUCION AÑO t */
work_conn.execute(text("DROP TABLE IF EXISTS #cruce_ejec_exp_mm"))
# t1.C_CUENTA/C_ITEM/C_ASIGNACION/C_ANALITICO coinciden con los de t2 por la condición de
# match (igualdad en el JOIN); se omiten de la selección para evitar nombres de columna
# duplicados (SQL Server no lo permite, a diferencia de PROC SQL). ENTIDAD y Clasif_Tcons
# sí son columnas nuevas que agrega ANALIT_TCONS
sql_cruce_ejec_exp_mm = """
SELECT
    t2.*,
    t1.ENTIDAD,
    t1.Clasif_Tcons
INTO #cruce_ejec_exp_mm
FROM GOBGENER.dbo.ANALIT_TCONS t1
LEFT JOIN #ejec_cgr_y t2
    ON t2.C_ENTIDAD = t1.ENTIDAD
   AND t2.C_ASIGNACION = t1.C_ASIGNACION
   AND t2.C_ANALITICO = t1.C_ANALITICO
   AND t2.C_CUENTA = t1.C_CUENTA
   AND t2.C_ITEM = t1.C_ITEM
WHERE t1.M_MUESTRA = 0
  AND t1.Clasif_Tcons = 'MM'
  AND t1.ENTIDAD NOT IN (5601, 10201)
  AND t2.DEVENG IS NOT NULL
"""
work_conn.execute(text(sql_cruce_ejec_exp_mm))


In [ ]:
# /* CREA TABLA DE REDISTRIBUCION EXPANDIDA SEGUN ESTRUCTURA DE MUESTRA, PARA RESTO DE
#    TRANSACCIONES MM, PARA EJECUCION AÑO t */
work_conn.execute(text("DROP TABLE IF EXISTS #ejec_exp_mm_prelim"))
sql_ejec_exp_mm_prelim = """
SELECT
    t2.SUBSECTOR,
    t2.MONEDA,
    t2.AÑO,
    t2.MES,
    t2.C_ENTIDAD,
    t2.C_PARTIDA,
    t2.N_PARTIDA,
    t2.C_CAPITULO,
    t2.N_CAPITULO,
    t2.C_PROGRAMA,
    t2.N_PROGRAMA,
    t2.C_TIPO_CUENTA,
    t2.C_CUENTA,
    t2.N_CUENTA,
    t2.C_ITEM,
    t2.C_ASIGNACION,
    t2.C_ANALITICO,
    t2.N_ANALITICO,
    t2.DEVENG,
    t2.Clasif_Tcons,
    t1.C_SCN,
    t1.C_CAGENTE,
    SUM(t2.DEVENG / 1000000.0 * t1.POND * t3.DATO) AS DEVENGO
INTO #ejec_exp_mm_prelim
FROM GOBGENER.dbo.EST_EXP_MM t1
LEFT JOIN #cruce_ejec_exp_mm t2
    ON t2.C_CUENTA = t1.C_CUENTA
   AND t2.C_ITEM = t1.C_ITEM
   AND t2.MES = t1.MES
   AND t1.ENTIDAD = t2.C_ENTIDAD
   AND t1.C_ASIGNACION = t2.C_ASIGNACION
   AND t1.C_ANALITICO = t2.C_ANALITICO
LEFT JOIN #prom_mes t3
    ON t2.MONEDA = t3.MONEDA
   AND t2.MES = t3.MES
   AND t2.AÑO = t3.AÑO
WHERE t2.DEVENG IS NOT NULL
GROUP BY
    t2.SUBSECTOR, t2.MONEDA, t2.AÑO, t2.MES, t2.C_ENTIDAD, t2.C_PARTIDA,
    t2.N_PARTIDA, t2.C_CAPITULO, t2.N_CAPITULO, t2.C_PROGRAMA, t2.N_PROGRAMA,
    t2.C_TIPO_CUENTA, t2.C_CUENTA, t2.N_CUENTA, t2.C_ITEM, t2.C_ASIGNACION,
    t2.C_ANALITICO, t2.N_ANALITICO, t2.DEVENG, t2.Clasif_Tcons, t1.C_SCN, t1.C_CAGENTE
"""
work_conn.execute(text(sql_ejec_exp_mm_prelim))
# DATA EJEC_EXP_MM_prelim; SET EJEC_EXP_MM_prelim(DROP=DEVENG DATO_OR Clasif_Tcons);
# -> DATO_OR nunca se calculó (se descartaba igual); se descartan las columnas fuente
work_conn.execute(text("ALTER TABLE #ejec_exp_mm_prelim DROP COLUMN DEVENG, Clasif_Tcons"))


In [ ]:
# /* Crea tabla con la ejecución total de transacciones MGG, Muestra + Resto */
work_conn.execute(text("DROP TABLE IF EXISTS #ejec_mm_total"))
sql_ejec_mm_total = """
SELECT
    SUBSECTOR, MONEDA, AÑO, MES, C_ENTIDAD, C_PARTIDA, N_PARTIDA, C_CAPITULO,
    N_CAPITULO, C_PROGRAMA, N_PROGRAMA, C_TIPO_CUENTA, C_CUENTA, N_CUENTA,
    C_ITEM, C_ASIGNACION, C_ANALITICO, N_ANALITICO, C_SCN, C_CAGENTE, DEVENGO
INTO #ejec_mm_total
FROM #ejec_exp_mm_prelim
UNION ALL
SELECT
    SUBSECTOR, MONEDA, AÑO, MES, C_ENTIDAD, C_PARTIDA, N_PARTIDA, C_CAPITULO,
    N_CAPITULO, C_PROGRAMA, N_PROGRAMA, C_TIPO_CUENTA, C_CUENTA, N_CUENTA,
    C_ITEM, C_ASIGNACION, C_ANALITICO, N_ANALITICO, C_SCN, C_CAGENTE, DEVENGO
FROM #ejec_muestra_mm
"""
work_conn.execute(text(sql_ejec_mm_total))


In [ ]:
# /* SALIDA de EJECUCION TOTAL DE TRANSACCIONES MM para COFOG y ANALISIS DESAGREGADO
#    CNF anual */
# /*Desagregado mes*/
work_conn.execute(text("DROP TABLE IF EXISTS #ejec_e_mm_aper"))
sql_ejec_e_mm_aper = """
SELECT
    t1.SUBSECTOR,
    t1.MONEDA,
    t1.AÑO,
    CASE
        WHEN t1.MES IN (1, 2, 3) THEN 1
        WHEN t1.MES IN (4, 5, 6) THEN 2
        WHEN t1.MES IN (7, 8, 9) THEN 3
        ELSE 4
    END AS [TRIM],
    t1.MES,
    t1.C_ENTIDAD,
    t1.C_PARTIDA,
    t1.C_CAPITULO,
    t1.C_PROGRAMA,
    t1.C_TIPO_CUENTA,
    t1.C_CUENTA,
    t1.C_ITEM,
    t1.C_ASIGNACION,
    t1.C_ANALITICO,
    t1.N_ANALITICO,
    t1.C_SCN,
    t1.C_CAGENTE,
    t3.ACTIVIDAD,
    t3.NIVEL,
    t4.C_COFOG_1,
    t4.N_COFOG_1,
    SUM(t1.DEVENGO) AS DATO
INTO #ejec_e_mm_aper
FROM #ejec_mm_total t1
LEFT JOIN GOBGENER.dbo.CR18_APERTURA t3 ON t1.C_ENTIDAD = t3.ENTIDAD
LEFT JOIN GOBGENER.dbo.COFOG t4
    ON t1.C_ENTIDAD = t4.ENTIDAD
   AND t1.C_PARTIDA = t4.C_PARTIDA
   AND t1.C_CAPITULO = t4.C_CAPITULO
   AND t1.C_PROGRAMA = t4.C_PROGRAMA
WHERE t1.C_SCN NOT IN ('F22', 'F29', 'F3', 'F32', 'F41', 'F42', 'F5', 'F612', 'F7', 'F71', 'F79', 'K10', 'K11', 'K9')
GROUP BY
    t1.SUBSECTOR, t1.MONEDA, t1.AÑO,
    CASE
        WHEN t1.MES IN (1, 2, 3) THEN 1
        WHEN t1.MES IN (4, 5, 6) THEN 2
        WHEN t1.MES IN (7, 8, 9) THEN 3
        ELSE 4
    END,
    t1.MES, t1.C_ENTIDAD, t1.C_PARTIDA, t1.C_CAPITULO, t1.C_PROGRAMA,
    t1.C_TIPO_CUENTA, t1.C_CUENTA, t1.C_ITEM, t1.C_ASIGNACION, t1.C_ANALITICO,
    t1.N_ANALITICO, t1.C_SCN, t1.C_CAGENTE, t3.ACTIVIDAD, t3.NIVEL, t4.C_COFOG_1, t4.N_COFOG_1
"""
res = work_conn.execute(text(sql_ejec_e_mm_aper))
_log("CREATE #ejec_e_mm_aper", res.rowcount)


In [ ]:
# /*Agregado año*/
work_conn.execute(text("DROP TABLE IF EXISTS #ejec_e_mm_aper_an"))
sql_ejec_e_mm_aper_an = """
SELECT
    t1.SUBSECTOR, t1.MONEDA, t1.AÑO, t1.C_ENTIDAD, t1.C_PARTIDA, t1.C_CAPITULO,
    t1.C_PROGRAMA, t1.C_TIPO_CUENTA, t1.C_CUENTA, t1.C_ITEM, t1.C_ASIGNACION,
    t1.C_ANALITICO, t1.N_ANALITICO, t1.C_SCN, t1.C_CAGENTE, t1.ACTIVIDAD, t1.NIVEL,
    t1.C_COFOG_1, t1.N_COFOG_1,
    SUM(t1.DATO) AS DATO
INTO #ejec_e_mm_aper_an
FROM #ejec_e_mm_aper t1
GROUP BY
    t1.SUBSECTOR, t1.MONEDA, t1.AÑO, t1.C_ENTIDAD, t1.C_PARTIDA, t1.C_CAPITULO,
    t1.C_PROGRAMA, t1.C_TIPO_CUENTA, t1.C_CUENTA, t1.C_ITEM, t1.C_ASIGNACION,
    t1.C_ANALITICO, t1.N_ANALITICO, t1.C_SCN, t1.C_CAGENTE, t1.ACTIVIDAD, t1.NIVEL,
    t1.C_COFOG_1, t1.N_COFOG_1
"""
res = work_conn.execute(text(sql_ejec_e_mm_aper_an))
_log("CREATE #ejec_e_mm_aper_an", res.rowcount)


In [ ]:
# /* SALIDA de EJECUCION TOTAL DE TRANSACCIONES MM para CNF anual */
work_conn.execute(text("DROP TABLE IF EXISTS #ejec_e_mm"))
sql_ejec_e_mm = """
SELECT
    t1.SUBSECTOR, t1.MONEDA, t1.AÑO, t1.[TRIM], t1.MES, t1.C_TIPO_CUENTA,
    t1.C_CUENTA, t1.C_ITEM, t1.C_SCN, t1.C_CAGENTE, t1.ACTIVIDAD, t1.NIVEL,
    SUM(t1.DATO) AS DATO
INTO #ejec_e_mm
FROM #ejec_e_mm_aper t1
GROUP BY
    t1.SUBSECTOR, t1.MONEDA, t1.AÑO, t1.[TRIM], t1.MES, t1.C_TIPO_CUENTA,
    t1.C_CUENTA, t1.C_ITEM, t1.C_SCN, t1.C_CAGENTE, t1.ACTIVIDAD, t1.NIVEL
"""
res = work_conn.execute(text(sql_ejec_e_mm))
_log("CREATE #ejec_e_mm", res.rowcount)


In [ ]:
# /* SALIDA de EJECUCION TOTAL DE TRANSACCIONES MM para CNF anual */
work_conn.execute(text("DROP TABLE IF EXISTS #ejec_mm_an"))
sql_ejec_mm_an = """
SELECT
    t1.SUBSECTOR, t1.MONEDA, t1.AÑO, t1.C_TIPO_CUENTA, t1.C_CUENTA, t1.C_ITEM,
    t1.C_SCN, t1.C_CAGENTE, t1.ACTIVIDAD, t1.NIVEL,
    SUM(t1.DATO) AS DATO
INTO #ejec_mm_an
FROM #ejec_e_mm_aper_an t1
GROUP BY
    t1.SUBSECTOR, t1.MONEDA, t1.AÑO, t1.C_TIPO_CUENTA, t1.C_CUENTA, t1.C_ITEM,
    t1.C_SCN, t1.C_CAGENTE, t1.ACTIVIDAD, t1.NIVEL
"""
res = work_conn.execute(text(sql_ejec_mm_an))
_log("CREATE #ejec_mm_an", res.rowcount)


In [ ]:
# PROC SQL; DROP TABLE EJEC_MUESTRA_MM,CRUCE_EJEC_EXP_MM,EJEC_EXP_MM_prelim,EJEC_EXP_MM,EJEC_MM_TOTAL; QUIT;
# limpieza de tablas intermedias de la sesión, igual que el SAS original
# (EJEC_EXP_MM nunca se creó en este nodo -> el DROP IF EXISTS no hace nada con ese nombre)
for _t in ["#ejec_muestra_mm", "#cruce_ejec_exp_mm", "#ejec_exp_mm_prelim", "#ejec_exp_mm", "#ejec_mm_total"]:
    work_conn.execute(text(f"DROP TABLE IF EXISTS {_t}"))


## 4- TCons MGG_CGR

Redistribuye la muestra de transacciones de gasto en bienes y servicios de consumo (MGG) según su estructura, expande el resto de transacciones fuera de muestra, y consolida la ejecución total mensual y anual por entidad, cuenta, clasificación SCN y función de gobierno (COFOG) del año en curso

*confianza: medium · verificador: approve · SAS: PROC SQL multi-paso: LEFT JOINs contra tablas de estructura (EST_MGG/EST_EXP_MGG/ANALIT_TCONS/CR18_APERTURA/COFOG), agregaciones con CASE recalculado (CALCULATED TRIM), UNION ALL y DROP TABLE final de limpieza*

In [ ]:
# ========= 4- TCons MGG_CGR =========
# /* CREA TABLA DE REDISTRIBUCION DE MUESTRA DE TRANSACCIONES MGG SEGUN ESTRUCTURA, PARA EJECUCION AÑO t */
work_conn.execute(text("DROP TABLE IF EXISTS #ejec_muestra_mgg_prelim"))
sql_ejec_muestra_mgg_prelim = """
SELECT
    t2.SUBSECTOR,
    t2.MONEDA,
    t2.AÑO,
    t2.MES,
    t1.ENTIDAD AS C_ENTIDAD,
    t2.C_PARTIDA,
    t2.N_PARTIDA,
    t2.C_CAPITULO,
    t2.N_CAPITULO,
    t2.C_PROGRAMA,
    t2.N_PROGRAMA,
    t2.C_TIPO_CUENTA,
    t2.C_CUENTA,
    t2.N_CUENTA,
    t2.C_ITEM,
    t2.C_ASIGNACION,
    t2.C_ANALITICO,
    t2.N_ANALITICO,
    t2.DEVENG,
    t1.C_SCN,
    t1.C_CAGENTE,
    SUM(t2.DEVENG / 1000000.0 * t1.POND) AS DEVENGO
INTO #ejec_muestra_mgg_prelim
FROM GOBGENER.dbo.EST_MGG t1
LEFT JOIN #ejec_cgr_y t2
    ON t2.C_ENTIDAD = t1.ENTIDAD
   AND t2.C_ASIGNACION = t1.C_ASIGNACION
   AND t2.C_ANALITICO = t1.C_ANALITICO
   AND t2.C_CUENTA = t1.C_CUENTA
   AND t2.C_ITEM = t1.C_ITEM
   AND t2.MES = t1.MES
WHERE t2.C_CUENTA IN ('24')
  AND t2.C_ITEM IN ('01','03','08','09')
  AND t1.ENTIDAD NOT IN (5601, 10201)
  AND t2.DEVENG IS NOT NULL
GROUP BY
    t2.SUBSECTOR, t2.MONEDA, t2.AÑO, t2.MES, t1.ENTIDAD,
    t2.C_PARTIDA, t2.N_PARTIDA, t2.C_CAPITULO, t2.N_CAPITULO,
    t2.C_PROGRAMA, t2.N_PROGRAMA, t2.C_TIPO_CUENTA, t2.C_CUENTA,
    t2.N_CUENTA, t2.C_ITEM, t2.C_ASIGNACION, t2.C_ANALITICO,
    t2.N_ANALITICO, t1.C_SCN, t1.C_CAGENTE, t2.DEVENG
"""
res = work_conn.execute(text(sql_ejec_muestra_mgg_prelim))
_log("CREATE #ejec_muestra_mgg_prelim", res.rowcount)


In [ ]:
# DATA EJEC_MUESTRA_MGG; SET EJEC_MUESTRA_MGG_prelim (DROP=DEVENG); RUN;
work_conn.execute(text("DROP TABLE IF EXISTS #ejec_muestra_mgg"))
sql_ejec_muestra_mgg = """
SELECT
    SUBSECTOR, MONEDA, AÑO, MES, C_ENTIDAD, C_PARTIDA, N_PARTIDA,
    C_CAPITULO, N_CAPITULO, C_PROGRAMA, N_PROGRAMA, C_TIPO_CUENTA,
    C_CUENTA, N_CUENTA, C_ITEM, C_ASIGNACION, C_ANALITICO, N_ANALITICO,
    C_SCN, C_CAGENTE, DEVENGO
INTO #ejec_muestra_mgg
FROM #ejec_muestra_mgg_prelim
"""
res = work_conn.execute(text(sql_ejec_muestra_mgg))
_log("CREATE #ejec_muestra_mgg", res.rowcount)


In [ ]:
# /* CREA TABLA DE ANALITICOS DE MGG A EXPANDIR (FUERA DE MUESTRA), SEGUN EJECUCION AÑO t */
work_conn.execute(text("DROP TABLE IF EXISTS #cruce_ejec_exp_mgg"))
sql_cruce_ejec_exp_mgg = """
SELECT
    t2.SUBSECTOR,
    t2.MONEDA,
    t2.AÑO,
    t2.MES,
    t2.C_ENTIDAD,
    t2.C_PARTIDA,
    t2.N_PARTIDA,
    t2.C_CAPITULO,
    t2.N_CAPITULO,
    t2.C_PROGRAMA,
    t2.N_PROGRAMA,
    t2.C_TIPO_CUENTA,
    t2.N_CUENTA,
    t2.N_ANALITICO,
    t2.DEVENG,
    t1.C_CUENTA,
    t1.C_ITEM,
    t1.C_ASIGNACION,
    t1.C_ANALITICO,
    t1.Clasif_Tcons
INTO #cruce_ejec_exp_mgg
FROM GOBGENER.dbo.ANALIT_TCONS t1
LEFT JOIN #ejec_cgr_y t2
    ON t2.C_ENTIDAD = t1.ENTIDAD
   AND t2.C_ASIGNACION = t1.C_ASIGNACION
   AND t2.C_ANALITICO = t1.C_ANALITICO
   AND t2.C_CUENTA = t1.C_CUENTA
   AND t2.C_ITEM = t1.C_ITEM
WHERE t1.M_MUESTRA = 0
  AND t1.Clasif_Tcons = 'MGG'
  AND t2.C_ENTIDAD NOT IN (5601, 10201)
  AND t2.DEVENG IS NOT NULL
"""
res = work_conn.execute(text(sql_cruce_ejec_exp_mgg))
_log("CREATE #cruce_ejec_exp_mgg", res.rowcount)


In [ ]:
# /* CREA TABLA DE REDISTRIBUCION EXPANDIDA SEGUN ESTRUCTURA DE MUESTRA, PARA RESTO DE TRANSACCIONES MGG, PARA EJECUCION AÑO t */
work_conn.execute(text("DROP TABLE IF EXISTS #ejec_exp_mgg_preliminar_raw"))
sql_ejec_exp_mgg_preliminar_raw = """
SELECT
    t2.SUBSECTOR,
    t2.MONEDA,
    t2.AÑO,
    t2.MES,
    t2.C_ENTIDAD,
    t2.C_PARTIDA,
    t2.N_PARTIDA,
    t2.C_CAPITULO,
    t2.N_CAPITULO,
    t2.C_PROGRAMA,
    t2.N_PROGRAMA,
    t2.C_TIPO_CUENTA,
    t2.C_CUENTA,
    t2.N_CUENTA,
    t2.C_ITEM,
    t2.C_ASIGNACION,
    t2.C_ANALITICO,
    t2.N_ANALITICO,
    t2.DEVENG,
    t2.Clasif_Tcons,
    t1.C_SCN,
    t1.C_CAGENTE,
    SUM(t2.DEVENG) AS DATO_OR,
    SUM(t2.DEVENG / 1000000.0 * t1.POND * t3.DATO) AS DEVENGO
INTO #ejec_exp_mgg_preliminar_raw
FROM GOBGENER.dbo.EST_EXP_MGG t1
LEFT JOIN #cruce_ejec_exp_mgg t2
    ON t2.C_CUENTA = t1.C_CUENTA
   AND t2.C_ITEM = t1.C_ITEM
   AND t2.MES = t1.MES
LEFT JOIN #prom_mes t3
    ON t2.MONEDA = t3.MONEDA
   AND t2.MES = t3.MES
   AND t2.AÑO = t3.AÑO
WHERE t2.DEVENG IS NOT NULL
GROUP BY
    t2.SUBSECTOR, t2.MONEDA, t2.AÑO, t2.MES, t2.C_ENTIDAD,
    t2.C_PARTIDA, t2.N_PARTIDA, t2.C_CAPITULO, t2.N_CAPITULO,
    t2.C_PROGRAMA, t2.N_PROGRAMA, t2.C_TIPO_CUENTA, t2.C_CUENTA,
    t2.N_CUENTA, t2.C_ITEM, t2.C_ASIGNACION, t2.C_ANALITICO,
    t2.N_ANALITICO, t2.DEVENG, t2.Clasif_Tcons, t1.C_SCN, t1.C_CAGENTE
"""
work_conn.execute(text(sql_ejec_exp_mgg_preliminar_raw))


In [ ]:
# DATA EJEC_EXP_MGG_preliminar; SET EJEC_EXP_MGG_preliminar (DROP=DEVENG DATO_OR Clasif_Tcons); RUN;
work_conn.execute(text("DROP TABLE IF EXISTS #ejec_exp_mgg_preliminar"))
sql_ejec_exp_mgg_preliminar = """
SELECT
    SUBSECTOR, MONEDA, AÑO, MES, C_ENTIDAD, C_PARTIDA, N_PARTIDA,
    C_CAPITULO, N_CAPITULO, C_PROGRAMA, N_PROGRAMA, C_TIPO_CUENTA,
    C_CUENTA, N_CUENTA, C_ITEM, C_ASIGNACION, C_ANALITICO, N_ANALITICO,
    C_SCN, C_CAGENTE, DEVENGO
INTO #ejec_exp_mgg_preliminar
FROM #ejec_exp_mgg_preliminar_raw
"""
res = work_conn.execute(text(sql_ejec_exp_mgg_preliminar))
work_conn.execute(text("DROP TABLE IF EXISTS #ejec_exp_mgg_preliminar_raw"))
_log("CREATE #ejec_exp_mgg_preliminar", res.rowcount)


In [ ]:
# /* Crea tabla con la ejecución total de transacciones MGG, Muestra + Resto */
work_conn.execute(text("DROP TABLE IF EXISTS #ejec_mgg_total"))
sql_ejec_mgg_total = """
SELECT SUBSECTOR, MONEDA, AÑO, MES, C_ENTIDAD, C_PARTIDA, N_PARTIDA,
       C_CAPITULO, N_CAPITULO, C_PROGRAMA, N_PROGRAMA, C_TIPO_CUENTA,
       C_CUENTA, N_CUENTA, C_ITEM, C_ASIGNACION, C_ANALITICO, N_ANALITICO,
       C_SCN, C_CAGENTE, DEVENGO
INTO #ejec_mgg_total
FROM #ejec_muestra_mgg
UNION ALL
SELECT SUBSECTOR, MONEDA, AÑO, MES, C_ENTIDAD, C_PARTIDA, N_PARTIDA,
       C_CAPITULO, N_CAPITULO, C_PROGRAMA, N_PROGRAMA, C_TIPO_CUENTA,
       C_CUENTA, N_CUENTA, C_ITEM, C_ASIGNACION, C_ANALITICO, N_ANALITICO,
       C_SCN, C_CAGENTE, DEVENGO
FROM #ejec_exp_mgg_preliminar
"""
res = work_conn.execute(text(sql_ejec_mgg_total))
_log("CREATE #ejec_mgg_total", res.rowcount)


In [ ]:
# /* SALIDA de EJECUCION TOTAL DE TRANSACCIONES MGG para COFOG y ANALISIS DESAGREGADO CNF anual */
# /* Desagregado mes */
work_conn.execute(text("DROP TABLE IF EXISTS #ejec_e_mgg_aper"))
sql_ejec_e_mgg_aper = """
SELECT
    t1.SUBSECTOR,
    t1.MONEDA,
    t1.AÑO,
    CASE
        WHEN t1.MES IN (1,2,3) THEN 1
        WHEN t1.MES IN (4,5,6) THEN 2
        WHEN t1.MES IN (7,8,9) THEN 3
        ELSE 4
    END AS TRIM,
    t1.MES,
    t1.C_ENTIDAD,
    t1.C_PARTIDA,
    t1.C_CAPITULO,
    t1.C_PROGRAMA,
    t1.C_TIPO_CUENTA,
    t1.C_CUENTA,
    t1.C_ITEM,
    t1.C_ASIGNACION,
    t1.C_ANALITICO,
    t1.N_ANALITICO,
    t1.C_SCN,
    t1.C_CAGENTE,
    t3.ACTIVIDAD,
    t3.NIVEL,
    t4.C_COFOG_1,
    t4.N_COFOG_1,
    SUM(t1.DEVENGO) AS DATO
INTO #ejec_e_mgg_aper
FROM #ejec_mgg_total t1
LEFT JOIN GOBGENER.dbo.CR18_APERTURA t3 ON t1.C_ENTIDAD = t3.ENTIDAD
LEFT JOIN GOBGENER.dbo.COFOG t4
    ON t1.C_ENTIDAD = t4.ENTIDAD
   AND t1.C_PARTIDA = t4.C_PARTIDA
   AND t1.C_CAPITULO = t4.C_CAPITULO
   AND t1.C_PROGRAMA = t4.C_PROGRAMA
WHERE t1.C_SCN NOT IN ('F22','F29','F3','F32','F41','F42','F5','F612','F7','F71','F79','K10','K11','K9')
GROUP BY
    t1.SUBSECTOR, t1.MONEDA, t1.AÑO,
    CASE
        WHEN t1.MES IN (1,2,3) THEN 1
        WHEN t1.MES IN (4,5,6) THEN 2
        WHEN t1.MES IN (7,8,9) THEN 3
        ELSE 4
    END,
    t1.MES, t1.C_ENTIDAD, t1.C_PARTIDA, t1.C_CAPITULO, t1.C_PROGRAMA,
    t1.C_TIPO_CUENTA, t1.C_CUENTA, t1.C_ITEM, t1.C_ASIGNACION,
    t1.C_ANALITICO, t1.N_ANALITICO, t1.C_SCN, t1.C_CAGENTE,
    t3.ACTIVIDAD, t3.NIVEL, t4.C_COFOG_1, t4.N_COFOG_1
"""
res = work_conn.execute(text(sql_ejec_e_mgg_aper))
_log("CREATE #ejec_e_mgg_aper", res.rowcount)


In [ ]:
# /* SALIDA de EJECUCION TOTAL DE TRANSACCIONES MGG para COFOG y ANALISIS DESAGREGADO CNF anual */
# /* Agregado año */
work_conn.execute(text("DROP TABLE IF EXISTS #ejec_e_mgg_aper_an"))
sql_ejec_e_mgg_aper_an = """
SELECT
    t1.SUBSECTOR, t1.MONEDA, t1.AÑO, t1.C_ENTIDAD, t1.C_PARTIDA,
    t1.C_CAPITULO, t1.C_PROGRAMA, t1.C_TIPO_CUENTA, t1.C_CUENTA,
    t1.C_ITEM, t1.C_ASIGNACION, t1.C_ANALITICO, t1.N_ANALITICO,
    t1.C_SCN, t1.C_CAGENTE, t1.ACTIVIDAD, t1.NIVEL, t1.C_COFOG_1,
    t1.N_COFOG_1,
    SUM(t1.DATO) AS DATO
INTO #ejec_e_mgg_aper_an
FROM #ejec_e_mgg_aper t1
GROUP BY
    t1.SUBSECTOR, t1.MONEDA, t1.AÑO, t1.C_ENTIDAD, t1.C_PARTIDA,
    t1.C_CAPITULO, t1.C_PROGRAMA, t1.C_TIPO_CUENTA, t1.C_CUENTA,
    t1.C_ITEM, t1.C_ASIGNACION, t1.C_ANALITICO, t1.N_ANALITICO,
    t1.C_SCN, t1.C_CAGENTE, t1.ACTIVIDAD, t1.NIVEL, t1.C_COFOG_1,
    t1.N_COFOG_1
"""
res = work_conn.execute(text(sql_ejec_e_mgg_aper_an))
_log("CREATE #ejec_e_mgg_aper_an", res.rowcount)


In [ ]:
# /* SALIDA de EJECUCION TOTAL DE TRANSACCIONES MGG para CNF anual */
work_conn.execute(text("DROP TABLE IF EXISTS #ejec_e_mgg"))
sql_ejec_e_mgg = """
SELECT
    t1.SUBSECTOR, t1.MONEDA, t1.AÑO, t1.TRIM, t1.MES,
    t1.C_TIPO_CUENTA, t1.C_CUENTA, t1.C_ITEM, t1.C_SCN, t1.C_CAGENTE,
    t1.ACTIVIDAD, t1.NIVEL,
    SUM(t1.DATO) AS DATO
INTO #ejec_e_mgg
FROM #ejec_e_mgg_aper t1
GROUP BY
    t1.SUBSECTOR, t1.MONEDA, t1.AÑO, t1.TRIM, t1.MES,
    t1.C_TIPO_CUENTA, t1.C_CUENTA, t1.C_ITEM, t1.C_SCN, t1.C_CAGENTE,
    t1.ACTIVIDAD, t1.NIVEL
"""
res = work_conn.execute(text(sql_ejec_e_mgg))
_log("CREATE #ejec_e_mgg", res.rowcount)


In [ ]:
# /* SALIDA de EJECUCION TOTAL DE TRANSACCIONES MGG para CNF anual */
work_conn.execute(text("DROP TABLE IF EXISTS #ejec_mgg_an"))
sql_ejec_mgg_an = """
SELECT
    t1.SUBSECTOR, t1.MONEDA, t1.AÑO, t1.C_TIPO_CUENTA, t1.C_CUENTA,
    t1.C_ITEM, t1.C_SCN, t1.C_CAGENTE, t1.ACTIVIDAD, t1.NIVEL,
    SUM(t1.DATO) AS DATO
INTO #ejec_mgg_an
FROM #ejec_e_mgg_aper t1
GROUP BY
    t1.SUBSECTOR, t1.MONEDA, t1.AÑO, t1.C_TIPO_CUENTA, t1.C_CUENTA,
    t1.C_ITEM, t1.C_SCN, t1.C_CAGENTE, t1.ACTIVIDAD, t1.NIVEL
"""
res = work_conn.execute(text(sql_ejec_mgg_an))
_log("CREATE #ejec_mgg_an", res.rowcount)


In [ ]:
# PROC SQL; DROP TABLE EJEC_MUESTRA_MGG, EJEC_MUESTRA_MGG_prelim, CRUCE_EJEC_EXP_MGG,
# EJEC_EXP_MGG_prelim, EJEC_EXP_MGG_preliminar, EJEC_EXP_MGG, EJEC_MGG_TOTAL; QUIT;
for _t in [
    "#ejec_muestra_mgg",
    "#ejec_muestra_mgg_prelim",
    "#cruce_ejec_exp_mgg",
    "#ejec_exp_mgg_prelim",
    "#ejec_exp_mgg_preliminar",
    "#ejec_exp_mgg",
    "#ejec_mgg_total",
]:
    work_conn.execute(text(f"DROP TABLE IF EXISTS {_t}"))


## SALIDA CNF_anual

Arma la ejecución presupuestaria consolidada no financiera anual, aplicando las reclasificaciones y exclusiones de negocio vigentes (Transantiago, cobre bruto, D63, subvenciones, dividendos CORFO, estandarización de cuentas) antes de unir gobierno central y municipal

*confianza: low · verificador: approve · SAS: PROC SQL: secuencia de CREATE TABLE/UPDATE/DELETE que consolida analíticos predefinidos, variables y diferenciados en una ejecución CNF anual (gobierno central + municipal), con reasignaciones manuales punto a punto*

In [ ]:
# ========= SALIDA CNF_anual =========
# /* Salida para indicadores GC para CNF anual y COFOG */
# /*Analiticos predefinidos */
work_conn.execute(text('DROP TABLE IF EXISTS #ejecucion_a_prelim_an'))
sql_ejecucion_a_prelim_an = f'''
SELECT t1.SUBSECTOR, t1.MONEDA, t1.AÑO,
       CASE WHEN t1.MES IN (1,2,3) THEN 1
            WHEN t1.MES IN (4,5,6) THEN 2
            WHEN t1.MES IN (7,8,9) THEN 3
            ELSE 4 END AS TRIM,
       t1.MES, t1.C_ENTIDAD, t1.C_PARTIDA, t1.C_CAPITULO, t1.C_PROGRAMA,
       t1.C_TIPO_CUENTA, t1.C_CUENTA, t1.C_ITEM, t1.C_ASIGNACION, t1.C_ANALITICO, t1.N_ANALITICO,
       CASE WHEN t2.ANALITICO = 'M' THEN t1.N_ANALITICO ELSE t2.ANALITICO END AS ANALITICO_,
       t2.C_SCN, t2.C_CAGENTE, t2.C_TRANSACCION, t2.T_TRANSACCION, t2.N_CUENTA, t2.C_ACTIVO, t2.C_CUP,
       t4.ACTIVIDAD, t4.NIVEL,
       (t1.DEVENG * t3.DATO / 1000000) AS DEVENGO
INTO #ejecucion_a_prelim_an
FROM GOBGENER.dbo.EJECUCION t1
LEFT JOIN GOBGENER.dbo.CR18_T_SCN_1 t2
    ON t1.C_TIPO_CUENTA = t2.T_CUENTA AND t1.C_CUENTA = t2.C_CUENTA
   AND t1.C_ITEM = t2.C_ITEM AND t1.C_ANALITICO = t2.C_ANALITICO
LEFT JOIN #prom_mes t3
    ON t1.MONEDA = t3.MONEDA AND t1.MES = t3.MES AND t1.AÑO = t3.AÑO
LEFT JOIN GOBGENER.dbo.CR18_APERTURA t4 ON t1.C_ENTIDAD = t4.ENTIDAD
WHERE t1.C_CUENTA NOT IN ('05','13','24','33')
  AND t1.C_ENTIDAD NOT IN (5601,10201)
  AND t1.AÑO = {int(Year)}
  AND t2.C_SCN NOT IN ('F22','F29','F3','F32','F41','F42','F5','F612','F7','F71','F79','K10','K11','K9')
'''
res = work_conn.execute(text(sql_ejecucion_a_prelim_an))
_log('ejecucion_a_prelim_an', res.rowcount)


In [ ]:
res = work_conn.execute(text("DELETE FROM #ejecucion_a_prelim_an WHERE C_CUENTA = '32' AND C_ANALITICO IN ('04001000','04002000')"))
_log('DELETE #ejecucion_a_prelim_an (cuenta 32)', res.rowcount)


In [ ]:
# /*Analiticos predefinidos: AJUSTE COYUNTURA - se elimina de indicador transacción CORFO
# que se desagrega anualmente entre ventas y D45 */
res = work_conn.execute(text(
    "DELETE FROM #ejecucion_a_prelim_an WHERE C_ENTIDAD = 3801 AND C_CUENTA = '06' "
    "AND C_ITEM = '01' AND C_ASIGNACION = '000' AND C_ANALITICO IN ('01000000')"
))
_log('DELETE #ejecucion_a_prelim_an (ajuste coyuntura CORFO)', res.rowcount)


In [ ]:
# /*Analiticos predefinidos: reasignación D39 2% constitucional 2008 para Transantiago */
_filtro_transantiago = (
    "C_ENTIDAD = 19703 AND C_CUENTA = '26' AND C_ITEM = '03' AND C_ASIGNACION = '000' "
    "AND C_ANALITICO IN ('03000000') AND MONEDA IN ('P') AND AÑO = 2008 AND MES IN (10,11,12)"
)
work_conn.execute(text(f"UPDATE #ejecucion_a_prelim_an SET C_CAGENTE = 'S11' WHERE {_filtro_transantiago}"))
work_conn.execute(text(f"UPDATE #ejecucion_a_prelim_an SET C_SCN = 'D39' WHERE {_filtro_transantiago}"))
res = work_conn.execute(text(f"UPDATE #ejecucion_a_prelim_an SET C_TRANSACCION = '4139' WHERE {_filtro_transantiago}"))
_log('UPDATE #ejecucion_a_prelim_an (D39 Transantiago)', res.rowcount)


In [ ]:
# /*Analiticos predefinidos: AJUSTE COBRE BRUTO (Financiamiento Capacidadad Estrategicas
# Defensa Nacional) - reasignacion entre D75 y D42 */
_filtro_cobre_bruto = (
    "C_ENTIDAD = 19701 AND C_CUENTA = '08' AND C_ITEM = '99' AND C_ASIGNACION = '999' "
    "AND C_ANALITICO IN ('99999000') AND MONEDA IN ('D') AND AÑO > 2019"
)
work_conn.execute(text(f"UPDATE #ejecucion_a_prelim_an SET C_SCN = 'D42' WHERE {_filtro_cobre_bruto}"))
work_conn.execute(text(f"UPDATE #ejecucion_a_prelim_an SET C_TRANSACCION = '42222' WHERE {_filtro_cobre_bruto}"))
res = work_conn.execute(text(f"UPDATE #ejecucion_a_prelim_an SET N_CUENTA = 'Cobre bruto' WHERE {_filtro_cobre_bruto}"))
_log('UPDATE #ejecucion_a_prelim_an (ajuste cobre bruto)', res.rowcount)


In [ ]:
work_conn.execute(text('DROP TABLE IF EXISTS #ejecucion_a_an'))
sql_ejecucion_a_an = '''
SELECT t1.SUBSECTOR, t1.MONEDA, t1.AÑO, t1.C_TIPO_CUENTA, t1.C_CUENTA, t1.C_ITEM,
       t1.C_SCN, t1.C_CAGENTE, t1.C_TRANSACCION, t1.T_TRANSACCION, t1.N_CUENTA,
       t1.C_ACTIVO, t1.C_CUP, t1.ACTIVIDAD, t1.NIVEL,
       SUM(t1.DEVENGO) AS Dato
INTO #ejecucion_a_an
FROM #ejecucion_a_prelim_an t1
WHERE t1.C_SCN NOT IN ('F22','F29','F3','F32','F41','F42','F5','F612','F7','F71','F79','K10','K11','K9')
GROUP BY t1.SUBSECTOR, t1.MONEDA, t1.AÑO, t1.C_TIPO_CUENTA, t1.C_CUENTA, t1.C_ITEM,
         t1.C_SCN, t1.C_CAGENTE, t1.C_TRANSACCION, t1.T_TRANSACCION, t1.N_CUENTA,
         t1.C_ACTIVO, t1.C_CUP, t1.ACTIVIDAD, t1.NIVEL
'''
res = work_conn.execute(text(sql_ejecucion_a_an))
_log('ejecucion_a_an', res.rowcount)


In [ ]:
# /*Analiticos variables */
work_conn.execute(text('DROP TABLE IF EXISTS #ejecucion_b_an'))
sql_ejecucion_b_an = f'''
SELECT t1.SUBSECTOR, t1.MONEDA, t1.AÑO, t1.C_TIPO_CUENTA, t1.C_CUENTA, t1.C_ITEM,
       t2.C_SCN, t2.C_CAGENTE, t2.C_TRANSACCION, t2.T_TRANSACCION, t2.N_CUENTA,
       t2.C_ACTIVO, t2.C_CUP, t4.ACTIVIDAD, t4.NIVEL,
       SUM(t1.DEVENG * t3.DATO / 1000000) AS Dato
INTO #ejecucion_b_an
FROM GOBGENER.dbo.EJECUCION t1
LEFT JOIN GOBGENER.dbo.CR18_T_SCN_1 t2
    ON t1.C_TIPO_CUENTA = t2.T_CUENTA AND t1.C_CUENTA = t2.C_CUENTA AND t1.C_ITEM = t2.C_ITEM
   AND t1.C_ASIGNACION = t2.C_ASIGNACION AND t1.C_ANALITICO = t2.C_ANALITICO
LEFT JOIN #prom_mes t3
    ON t1.MONEDA = t3.MONEDA AND t1.MES = t3.MES AND t1.AÑO = t3.AÑO
LEFT JOIN GOBGENER.dbo.CR18_APERTURA t4 ON t1.C_ENTIDAD = t4.ENTIDAD
WHERE t1.C_CUENTA = '32' AND t1.C_ANALITICO IN ('04001000','04002000')
  AND t1.C_ENTIDAD NOT IN (5601,10201)
  AND t1.AÑO = {int(Year)}
  AND t2.C_SCN NOT IN ('F22','F29','F3','F32','F41','F42','F5','F612','F7','F71','F79','K10','K11','K9')
GROUP BY t1.SUBSECTOR, t1.MONEDA, t1.AÑO, t1.C_TIPO_CUENTA, t1.C_CUENTA, t1.C_ITEM,
         t2.C_SCN, t2.C_CAGENTE, t2.C_TRANSACCION, t2.T_TRANSACCION, t2.N_CUENTA,
         t2.C_ACTIVO, t2.C_CUP, t4.ACTIVIDAD, t4.NIVEL
'''
res = work_conn.execute(text(sql_ejecucion_b_an))
_log('ejecucion_b_an', res.rowcount)


In [ ]:
work_conn.execute(text('DROP TABLE IF EXISTS #ejecucion_c_an_prelim'))
sql_ejecucion_c_an_prelim = f'''
SELECT t1.SUBSECTOR, t1.MONEDA, t1.AÑO,
       CASE WHEN t1.MES IN (1,2,3) THEN 1
            WHEN t1.MES IN (4,5,6) THEN 2
            WHEN t1.MES IN (7,8,9) THEN 3
            ELSE 4 END AS TRIM,
       t1.MES, t1.C_ENTIDAD, t1.C_PARTIDA, t1.C_CAPITULO, t1.C_PROGRAMA,
       t1.C_TIPO_CUENTA, t1.C_CUENTA, t1.C_ITEM, t1.C_ASIGNACION, t1.C_ANALITICO, t1.N_ANALITICO,
       t2.C_SCN, t2.C_CAGENTE, t2.C_TRANSACCION, t2.T_TRANSACCION, t2.N_CUENTA,
       t2.C_ACTIVO, t2.C_CUP, t4.ACTIVIDAD, t4.NIVEL,
       (t1.DEVENG * t3.DATO / 1000000) AS Dato
INTO #ejecucion_c_an_prelim
FROM GOBGENER.dbo.EJECUCION t1
LEFT JOIN GOBGENER.dbo.CR18_T_SCN_2 t2
    ON t1.C_PARTIDA = t2.C_PARTIDA AND t1.C_CAPITULO = t2.C_CAPITULO AND t1.C_PROGRAMA = t2.C_PROGRAMA
   AND t1.C_ENTIDAD = t2.ENTIDAD AND t1.C_TIPO_CUENTA = t2.T_CUENTA AND t1.C_CUENTA = t2.C_CUENTA
   AND t1.C_ITEM = t2.C_ITEM AND t1.C_ASIGNACION = t2.C_ASIGNACION AND t1.C_ANALITICO = t2.C_ANALITICO
LEFT JOIN #prom_mes t3
    ON t1.MONEDA = t3.MONEDA AND t1.MES = t3.MES AND t1.AÑO = t3.AÑO
LEFT JOIN GOBGENER.dbo.CR18_APERTURA t4 ON t1.C_ENTIDAD = t4.ENTIDAD
WHERE t1.C_CUENTA IN ('05','13','24','33')
  AND t2.OBS NOT IN ('CR18_difcoy','CR18_difact')
  AND t1.C_ENTIDAD NOT IN (5601,10201)
  AND t2.C_SCN NOT IN ('TC')
  AND t1.AÑO = {int(Year)}
  AND t2.C_SCN NOT IN ('F22','F29','F3','F32','F41','F42','F5','F612','F7','F71','F79','K10','K11','K9')
GROUP BY t1.SUBSECTOR, t1.MONEDA, t1.AÑO,
         CASE WHEN t1.MES IN (1,2,3) THEN 1 WHEN t1.MES IN (4,5,6) THEN 2 WHEN t1.MES IN (7,8,9) THEN 3 ELSE 4 END,
         t1.MES, t1.C_TIPO_CUENTA, t1.C_CUENTA, t1.C_ITEM,
         t2.C_SCN, t2.C_CAGENTE, t2.C_TRANSACCION, t2.T_TRANSACCION, t2.N_CUENTA,
         t2.C_ACTIVO, t2.C_CUP, t4.ACTIVIDAD, t4.NIVEL
'''
res = work_conn.execute(text(sql_ejecucion_c_an_prelim))
_log('ejecucion_c_an_prelim', res.rowcount)


In [ ]:
# /*Analiticos variables: reasignación contragente ayudas IPS */
# /* Bono Ayuda Familiar (18O) */
res = work_conn.execute(text(
    "UPDATE #ejecucion_c_an_prelim SET C_CAGENTE = 'S14' WHERE C_ENTIDAD = 24101 AND C_CUENTA = '24' "
    "AND C_ITEM = '01' AND C_ASIGNACION = '006' AND C_ANALITICO IN ('01006000') AND MONEDA IN ('P') "
    "AND AÑO = 2019 AND MES = 12"
))
_log('UPDATE #ejecucion_c_an_prelim (Bono Ayuda Familiar)', res.rowcount)


In [ ]:
# /*Bono Covid */
res = work_conn.execute(text(
    "UPDATE #ejecucion_c_an_prelim SET C_CAGENTE = 'S14' WHERE C_ENTIDAD = 24101 AND C_CUENTA = '24' "
    "AND C_ITEM = '01' AND C_ASIGNACION = '006' AND C_ANALITICO IN ('01006000') AND MONEDA IN ('P') "
    "AND AÑO = 2020 AND MES = 4"
))
_log('UPDATE #ejecucion_c_an_prelim (Bono Covid)', res.rowcount)


In [ ]:
# /* Reasignación nivel de gobierno para Programa Ampliado de Inmunizaciones */
res = work_conn.execute(text(
    "UPDATE #ejecucion_c_an_prelim SET NIVEL = 'MUNICIPAL' WHERE C_ENTIDAD IN (11105,22701,22750) "
    "AND C_CUENTA = '24' AND C_ITEM = '01' AND C_ASIGNACION = '004'"
))
_log('UPDATE #ejecucion_c_an_prelim (PAI)', res.rowcount)


In [ ]:
# /* Reasignación nivel de D63 municipal */
res = work_conn.execute(text(
    "UPDATE #ejecucion_c_an_prelim SET NIVEL = 'MUNICIPAL' WHERE NIVEL = 'REGIONAL' "
    "AND C_SCN IN ('D63','D63-P2','D63-P') AND C_ITEM = '03' "
    "AND C_ASIGNACION IN ('001','003','006','011','013','015','016','017','018','020','021','100','101')"
))
_log('UPDATE #ejecucion_c_an_prelim (D63 municipal)', res.rowcount)


In [ ]:
# /* Beca Especial Carbon (previo 2020) */
_filtro_beca_carbon = (
    "C_ENTIDAD = 6403 AND C_CUENTA = '24' AND C_ITEM = '01' AND C_ASIGNACION = '270' "
    "AND C_ANALITICO IN ('01270000') AND MONEDA IN ('P') AND AÑO < 2020"
)
work_conn.execute(text(f"UPDATE #ejecucion_c_an_prelim SET C_CAGENTE = 'S14' WHERE {_filtro_beca_carbon}"))
work_conn.execute(text(f"UPDATE #ejecucion_c_an_prelim SET C_SCN = 'D75' WHERE {_filtro_beca_carbon}"))
res = work_conn.execute(text(f"UPDATE #ejecucion_c_an_prelim SET C_TRANSACCION = '4345' WHERE {_filtro_beca_carbon}"))
_log('UPDATE #ejecucion_c_an_prelim (Beca Especial Carbón)', res.rowcount)


In [ ]:
work_conn.execute(text('DROP TABLE IF EXISTS #ejecucion_c_an'))
sql_ejecucion_c_an = '''
SELECT t1.SUBSECTOR, t1.MONEDA, t1.AÑO, t1.C_TIPO_CUENTA, t1.C_CUENTA, t1.C_ITEM,
       t1.C_SCN, t1.C_CAGENTE, t1.C_TRANSACCION, t1.T_TRANSACCION, t1.N_CUENTA,
       t1.C_ACTIVO, t1.C_CUP, t1.ACTIVIDAD, t1.NIVEL,
       SUM(t1.Dato) AS Dato
INTO #ejecucion_c_an
FROM #ejecucion_c_an_prelim t1
WHERE t1.C_SCN NOT IN ('F22','F29','F3','F32','F41','F42','F5','F612','F7','F71','F79','K10','K11','K9')
GROUP BY t1.SUBSECTOR, t1.MONEDA, t1.AÑO, t1.C_TIPO_CUENTA, t1.C_CUENTA, t1.C_ITEM,
         t1.C_SCN, t1.C_CAGENTE, t1.C_TRANSACCION, t1.T_TRANSACCION, t1.N_CUENTA,
         t1.C_ACTIVO, t1.C_CUP, t1.ACTIVIDAD, t1.NIVEL
'''
res = work_conn.execute(text(sql_ejecucion_c_an))
_log('ejecucion_c_an', res.rowcount)


In [ ]:
# /* Transacciones diferenciadas de coyuntura, con clasificacion anual */
work_conn.execute(text('DROP TABLE IF EXISTS #ejecucion_d_an_prelim'))
sql_ejecucion_d_an_prelim = f'''
SELECT t1.SUBSECTOR, t1.MONEDA, t1.C_ENTIDAD, t1.AÑO, t1.C_TIPO_CUENTA, t1.C_CUENTA, t1.C_ITEM,
       t1.C_ASIGNACION, t2.C_SCN, t2.C_CAGENTE, t2.C_TRANSACCION, t2.T_TRANSACCION, t2.N_CUENTA,
       t2.C_ACTIVO, t2.C_CUP, t4.ACTIVIDAD, t4.NIVEL,
       (t1.DEVENG * t3.DATO / 1000000) AS Dato
INTO #ejecucion_d_an_prelim
FROM GOBGENER.dbo.EJECUCION t1
LEFT JOIN GOBGENER.dbo.CR18_T_SCN_2 t2
    ON t1.C_PARTIDA = t2.C_PARTIDA AND t1.C_CAPITULO = t2.C_CAPITULO AND t1.C_PROGRAMA = t2.C_PROGRAMA
   AND t1.C_ENTIDAD = t2.ENTIDAD AND t1.C_TIPO_CUENTA = t2.T_CUENTA AND t1.C_CUENTA = t2.C_CUENTA
   AND t1.C_ITEM = t2.C_ITEM AND t1.C_ASIGNACION = t2.C_ASIGNACION AND t1.C_ANALITICO = t2.C_ANALITICO
LEFT JOIN #prom_mes t3
    ON t1.MONEDA = t3.MONEDA AND t1.MES = t3.MES AND t1.AÑO = t3.AÑO
LEFT JOIN GOBGENER.dbo.CR18_APERTURA t4 ON t1.C_ENTIDAD = t4.ENTIDAD
WHERE t2.OBS IN ('CR18_difcoy')
  AND t1.C_ENTIDAD NOT IN (5601,10201)
  AND t2.C_SCN NOT IN ('TC')
  AND t1.AÑO = {int(Year)}
  AND t2.C_ACTIVO NOT IN ('F22','F29','F3','F32','F41','F42','F5','F612','F7','F71','F79','K10','K11','K9')
'''
res = work_conn.execute(text(sql_ejecucion_d_an_prelim))
_log('ejecucion_d_an_prelim', res.rowcount)


In [ ]:
# /* Reasignación nivel de D63 municipal */
res = work_conn.execute(text(
    "UPDATE #ejecucion_d_an_prelim SET NIVEL = 'MUNICIPAL' WHERE NIVEL = 'REGIONAL' "
    "AND C_SCN IN ('D63','D63-P2','D63-P') AND C_ITEM = '03' "
    "AND C_ASIGNACION IN ('001','003','006','011','013','015','016','017','018','020','021','100','101')"
))
_log('UPDATE #ejecucion_d_an_prelim (D63 municipal)', res.rowcount)


In [ ]:
# /* Glosas comunes GR: Aplicación de clasificación SCN y función según estructura */
work_conn.execute(text('DROP TABLE IF EXISTS #ejecucion_da_an_prelim'))
sql_ejecucion_da_an_prelim = '''
SELECT t1.*
INTO #ejecucion_da_an_prelim
FROM #ejecucion_d_an_prelim t1
WHERE t1.C_SCN IN ('D63-P2','D63-P') AND t1.NIVEL IN ('REGIONAL','MUNICIPAL')
'''
res = work_conn.execute(text(sql_ejecucion_da_an_prelim))
_log('ejecucion_da_an_prelim', res.rowcount)

res = work_conn.execute(text(
    "DELETE FROM #ejecucion_d_an_prelim WHERE C_SCN IN ('D63-P2','D63-P') AND NIVEL IN ('REGIONAL','MUNICIPAL')"
))
_log('DELETE #ejecucion_d_an_prelim (traspasado a DA)', res.rowcount)


In [ ]:
work_conn.execute(text('DROP TABLE IF EXISTS #ejecucion_da_an'))
# Las 10 ramas de C_COFOG_1 ('01.'..'10.') aplican la MISMA fórmula (Dato*POND) del SAS;
# se simplifica el CASE a un IN. C_COFOG_1/POND no están en el GROUP BY original (PROC SQL
# toma el primer registro del grupo); se envuelve en MAX() para cumplir T-SQL, asumiendo
# unicidad por (AÑO, ENTIDAD, ITEM) en EST_GLOSASCOM
sql_ejecucion_da_an = '''
SELECT t2.SUBSECTOR, t2.MONEDA, t2.C_ENTIDAD, t2.AÑO, t2.C_TIPO_CUENTA, t2.C_CUENTA, t2.C_ITEM,
       t1.C_SCN, t2.C_CAGENTE, t2.C_TRANSACCION, t2.T_TRANSACCION, t2.N_CUENTA,
       t2.C_ACTIVO, t2.C_CUP, t2.ACTIVIDAD, t2.NIVEL,
       MAX(CASE WHEN t1.C_COFOG_1 IN ('01.','02.','03.','04.','05.','06.','07.','08.','09.','10.')
                THEN t2.Dato * t1.POND END) AS Dato_p
INTO #ejecucion_da_an
FROM GOBGENER.dbo.EST_GLOSASCOM t1
LEFT JOIN #ejecucion_da_an_prelim t2 ON t2.AÑO = t1.AÑO AND t1.C_ENTIDAD = t2.C_ENTIDAD AND t1.C_ITEM = t2.C_ITEM
WHERE t2.NIVEL IS NOT NULL
GROUP BY t2.SUBSECTOR, t2.MONEDA, t2.AÑO, t2.C_ENTIDAD, t2.C_TIPO_CUENTA, t2.C_CUENTA, t2.C_ITEM,
         t1.C_SCN, t2.C_CAGENTE, t2.C_TRANSACCION, t2.T_TRANSACCION, t2.N_CUENTA,
         t2.C_ACTIVO, t2.C_CUP, t2.ACTIVIDAD, t2.NIVEL
'''
res = work_conn.execute(text(sql_ejecucion_da_an))
_log('ejecucion_da_an', res.rowcount)


In [ ]:
# CREATE TABLE EJECUCION_D_AN_prelim AS SELECT * FROM EJECUCION_D_AN_prelim UNION ALL EJECUCION_DA_AN
# (self-referential): T-SQL no permite leer y sobreescribir la misma tabla con SELECT INTO,
# se materializa en un nombre nuevo que usan los pasos siguientes
work_conn.execute(text('DROP TABLE IF EXISTS #ejecucion_d_an_prelim_final'))
sql_ejecucion_d_an_prelim_final = '''
SELECT SUBSECTOR, MONEDA, AÑO, C_TIPO_CUENTA, C_CUENTA, C_ITEM, C_SCN, C_CAGENTE,
       C_TRANSACCION, T_TRANSACCION, N_CUENTA, C_ACTIVO, C_CUP, ACTIVIDAD, NIVEL, Dato
INTO #ejecucion_d_an_prelim_final
FROM #ejecucion_d_an_prelim
UNION ALL
SELECT SUBSECTOR, MONEDA, AÑO, C_TIPO_CUENTA, C_CUENTA, C_ITEM, C_SCN, C_CAGENTE,
       C_TRANSACCION, T_TRANSACCION, N_CUENTA, C_ACTIVO, C_CUP, ACTIVIDAD, NIVEL, Dato_p AS Dato
FROM #ejecucion_da_an
'''
res = work_conn.execute(text(sql_ejecucion_d_an_prelim_final))
_log('ejecucion_d_an_prelim_final', res.rowcount)


In [ ]:
work_conn.execute(text('DROP TABLE IF EXISTS #ejecucion_d_an'))
sql_ejecucion_d_an = '''
SELECT t1.SUBSECTOR, t1.MONEDA, t1.AÑO, t1.C_TIPO_CUENTA, t1.C_CUENTA, t1.C_ITEM,
       t1.C_SCN, t1.C_CAGENTE, t1.C_TRANSACCION, t1.T_TRANSACCION, t1.N_CUENTA,
       t1.C_ACTIVO, t1.C_CUP, t1.ACTIVIDAD, t1.NIVEL,
       SUM(t1.Dato) AS Dato
INTO #ejecucion_d_an
FROM #ejecucion_d_an_prelim_final t1
WHERE t1.NIVEL IS NOT NULL
GROUP BY t1.SUBSECTOR, t1.MONEDA, t1.AÑO, t1.C_TIPO_CUENTA, t1.C_CUENTA, t1.C_ITEM,
         t1.C_SCN, t1.C_CAGENTE, t1.C_TRANSACCION, t1.T_TRANSACCION, t1.N_CUENTA,
         t1.C_ACTIVO, t1.C_CUP, t1.ACTIVIDAD, t1.NIVEL
'''
res = work_conn.execute(text(sql_ejecucion_d_an))
_log('ejecucion_d_an', res.rowcount)


In [ ]:
# /* Transacciones diferenciadas por actividad */
work_conn.execute(text('DROP TABLE IF EXISTS #ejecucion_f_an'))
sql_ejecucion_f_an = f'''
SELECT t1.SUBSECTOR, t1.MONEDA, t1.AÑO, t1.C_TIPO_CUENTA, t1.C_CUENTA, t1.C_ITEM,
       t2.C_SCN, t2.C_CAGENTE, t2.C_TRANSACCION, t2.T_TRANSACCION, t2.N_CUENTA,
       t2.C_ACTIVO, t2.C_CUP, t2.C_CUP AS ACTIVIDAD, t4.NIVEL,
       SUM(t1.DEVENG * t3.DATO / 1000000) AS Dato
INTO #ejecucion_f_an
FROM GOBGENER.dbo.EJECUCION t1
LEFT JOIN GOBGENER.dbo.CR18_T_SCN_2 t2
    ON t1.C_PARTIDA = t2.C_PARTIDA AND t1.C_CAPITULO = t2.C_CAPITULO AND t1.C_PROGRAMA = t2.C_PROGRAMA
   AND t1.C_ENTIDAD = t2.ENTIDAD AND t1.C_TIPO_CUENTA = t2.T_CUENTA AND t1.C_CUENTA = t2.C_CUENTA
   AND t1.C_ITEM = t2.C_ITEM AND t1.C_ASIGNACION = t2.C_ASIGNACION AND t1.C_ANALITICO = t2.C_ANALITICO
LEFT JOIN #prom_mes t3
    ON t1.MONEDA = t3.MONEDA AND t1.MES = t3.MES AND t1.AÑO = t3.AÑO
LEFT JOIN GOBGENER.dbo.CR18_APERTURA t4 ON t1.C_ENTIDAD = t4.ENTIDAD
WHERE t2.OBS IN ('CR18_difact')
  AND t1.C_ENTIDAD NOT IN (5601,10201)
  AND t2.C_SCN NOT IN ('TC')
  AND t1.AÑO = {int(Year)}
GROUP BY t1.SUBSECTOR, t1.MONEDA, t1.AÑO, t1.C_TIPO_CUENTA, t1.C_CUENTA, t1.C_ITEM,
         t2.C_SCN, t2.C_CAGENTE, t2.C_TRANSACCION, t2.T_TRANSACCION, t2.N_CUENTA,
         t2.C_ACTIVO, t2.C_CUP, t4.NIVEL
'''
res = work_conn.execute(text(sql_ejecucion_f_an))
_log('ejecucion_f_an', res.rowcount)


In [ ]:
# /* TCONS trimestrales */
# EJEC_MIXTAS_M_AN, EJEC_MGCG_AN, EJEC_MM_AN, EJEC_MGG_AN quedaron materializadas como
# tablas temporales de sesión por los nodos que los producen (session_sql)
work_conn.execute(text('DROP TABLE IF EXISTS #ejecucion_h_an'))
sql_ejecucion_h_an = '''
SELECT * INTO #ejecucion_h_an FROM #ejec_mixtas_m_an
UNION ALL
SELECT * FROM #ejec_mgcg_an
UNION ALL
SELECT * FROM #ejec_mm_an
UNION ALL
SELECT * FROM #ejec_mgg_an
'''
res = work_conn.execute(text(sql_ejecucion_h_an))
_log('ejecucion_h_an', res.rowcount)

res = work_conn.execute(text("UPDATE #ejecucion_h_an SET C_ACTIVO = 'TC'"))
_log('UPDATE #ejecucion_h_an (C_ACTIVO=TC)', res.rowcount)


In [ ]:
# /*UNE TABLAS*/
work_conn.execute(text('DROP TABLE IF EXISTS #ejecucion_ext_prev_an'))
sql_ejecucion_ext_prev_an = '''
SELECT * INTO #ejecucion_ext_prev_an FROM #ejecucion_a_an
UNION ALL
SELECT * FROM #ejecucion_b_an
UNION ALL
SELECT * FROM #ejecucion_c_an
UNION ALL
SELECT * FROM #ejecucion_d_an
UNION ALL
SELECT * FROM #ejecucion_f_an
UNION ALL
SELECT * FROM #ejecucion_h_an
'''
res = work_conn.execute(text(sql_ejecucion_ext_prev_an))
_log('ejecucion_ext_prev_an', res.rowcount)


In [ ]:
# DROP TABLE EJECUCION_A_prelim_AN, EJECUCION_A_AN, EJECUCION_B_AN, EJECUCION_C_AN,
# EJECUCION_C_AN_prelim, EJECUCION_D_AN, EJECUCION_D_AN_prelim, EJECUCION_DA_AN,
# EJECUCION_DA_AN_prelim, EJECUCION_F_AN, EJECUCION_H_AN
for _t in [
    '#ejecucion_a_prelim_an', '#ejecucion_a_an', '#ejecucion_b_an', '#ejecucion_c_an',
    '#ejecucion_c_an_prelim', '#ejecucion_d_an', '#ejecucion_d_an_prelim',
    '#ejecucion_d_an_prelim_final', '#ejecucion_da_an', '#ejecucion_da_an_prelim',
    '#ejecucion_f_an', '#ejecucion_h_an',
]:
    work_conn.execute(text(f'DROP TABLE IF EXISTS {_t}'))


In [ ]:
# /*Incorporacion de imputaciones permanentes*/
work_conn.execute(text('DROP TABLE IF EXISTS #cr18_apertura_pacapr'))
sql_cr18_apertura_pacapr = '''
SELECT DISTINCT t1.ACTIVIDAD, t2.C_PARTIDA, t2.C_CAPITULO, t2.C_PROGRAMA, t1.NIVEL
INTO #cr18_apertura_pacapr
FROM GOBGENER.dbo.CR18_APERTURA t1
LEFT JOIN GOBGENER.dbo.ENTIDADES t2 ON t1.ENTIDAD = t2.C_ENTIDAD
'''
res = work_conn.execute(text(sql_cr18_apertura_pacapr))
_log('cr18_apertura_pacapr', res.rowcount)


In [ ]:
work_conn.execute(text('DROP TABLE IF EXISTS #ejecucion_ext_dip_imput_perm_an'))
sql_ejecucion_ext_dip_imput_perm_an = f'''
SELECT t1.SUBSECTOR, t1.MONEDA, t1.AÑO, t1.C_TIPO_CUENTA, t1.C_CUENTA, t1.C_ITEM,
       t1.C_SCN, t1.C_CAGENTE, t1.C_TRANSACCION, t1.T_TRANSACCION, t1.N_CUENTA,
       t1.C_ACTIVO, t1.C_CUP, t2.ACTIVIDAD, t2.NIVEL,
       SUM(t1.Dato) AS Dato
INTO #ejecucion_ext_dip_imput_perm_an
FROM GOBGENER.dbo.IMPUT_GC_DIP_APER t1
LEFT JOIN #cr18_apertura_pacapr t2
    ON t1.C_PARTIDA = t2.C_PARTIDA AND t1.C_CAPITULO = t2.C_CAPITULO AND t1.C_PROGRAMA = t2.C_PROGRAMA
WHERE t1.AÑO = {int(Year)}
  AND t1.C_SCN NOT IN ('F22','F29','F3','F32','F41','F42','F5','F612','F7','F71','F79','K10','K11','K9')
GROUP BY t1.SUBSECTOR, t1.MONEDA, t1.AÑO, t1.C_TIPO_CUENTA, t1.C_CUENTA, t1.C_ITEM,
         t1.C_SCN, t1.C_CAGENTE, t1.C_TRANSACCION, t1.T_TRANSACCION, t1.N_CUENTA,
         t1.C_ACTIVO, t1.C_CUP, t2.ACTIVIDAD, t2.NIVEL
'''
res = work_conn.execute(text(sql_ejecucion_ext_dip_imput_perm_an))
_log('ejecucion_ext_dip_imput_perm_an', res.rowcount)


In [ ]:
work_conn.execute(text('DROP TABLE IF EXISTS #ejecucion_cnf_an'))
sql_ejecucion_cnf_an = '''
SELECT * INTO #ejecucion_cnf_an FROM #ejecucion_ext_prev_an
UNION ALL
SELECT * FROM #ejecucion_ext_dip_imput_perm_an
'''
res = work_conn.execute(text(sql_ejecucion_cnf_an))
_log('ejecucion_cnf_an', res.rowcount)


In [ ]:
# /*Reasignación de P3 para cuentas de subsector */
res = work_conn.execute(text("UPDATE #ejecucion_cnf_an SET C_SCN = 'D73' WHERE C_SCN = 'P3'"))
_log('UPDATE #ejecucion_cnf_an (P3->D73)', res.rowcount)


In [ ]:
# /*Reasignación de nivel regional a central, por metodología de acuerdo a
# caracteristicas actuales de GOREs */
res = work_conn.execute(text("UPDATE #ejecucion_cnf_an SET NIVEL = 'CENTRAL' WHERE NIVEL = 'REGIONAL'"))
_log('UPDATE #ejecucion_cnf_an (REGIONAL->CENTRAL)', res.rowcount)


In [ ]:
# /* Reasignación FBCF realizada por municipalidades pero medida en ejecución GC */
# el SAS compara NIVEL='CENTRAL ' (con espacio final); SQL Server ignora los espacios
# finales en comparaciones de cadena, mismo resultado
res = work_conn.execute(text(
    "UPDATE #ejecucion_cnf_an SET NIVEL = 'MUNICIPAL' WHERE NIVEL = 'CENTRAL' "
    "AND C_SCN = 'P51' AND C_CAGENTE = 'S1313'"
))
_log('UPDATE #ejecucion_cnf_an (FBCF municipal)', res.rowcount)


In [ ]:
# /* Eliminación de transacciones redundantes en consolidación */
_pares_redundantes = [
    ('D73', 'CENTRAL', 'S13'), ('D93', 'CENTRAL', 'S13'),
    ('D73', 'CENTRAL', 'S1312'), ('D93', 'CENTRAL', 'S1312'),
    ('D73', 'CENTRAL', 'S131'), ('D93', 'CENTRAL', 'S131'),
    ('D73', 'PREVISION', 'S1314'), ('D93', 'PREVISION', 'S1314'),
]
_total_borradas = 0
for _scn, _nivel, _cagente in _pares_redundantes:
    res = work_conn.execute(text(
        f"DELETE FROM #ejecucion_cnf_an WHERE C_SCN = '{_scn}' AND NIVEL = '{_nivel}' AND C_CAGENTE = '{_cagente}'"
    ))
    _total_borradas += res.rowcount
_log('DELETE #ejecucion_cnf_an (transacciones redundantes)', _total_borradas)


In [ ]:
for _t in ['#ejecucion_ext_dip_imput_perm_an', '#ejecucion_ext_prev_an']:
    work_conn.execute(text(f'DROP TABLE IF EXISTS {_t}'))


In [ ]:
# /* EJECUCION MUNICIPAL */
# /* Salida Ejecucion Municipal predefinidos */
work_conn.execute(text('DROP TABLE IF EXISTS #ejec_cnf_mun_a_an_a'))
sql_ejec_cnf_mun_a_an_a = f'''
SELECT t1.SUBSECTOR, t1.MONEDA, t1.AÑO, t1.C_TIPO_CUENTA, t1.C_CUENTA,
       SUBSTRING(t1.C_ANALITICO, 1, 2) AS C_ITEM,
       t2.C_SCN, t2.C_CAGENTE, t2.C_TRANSACCION, t2.T_TRANSACCION, t2.N_CUENTA,
       t2.C_ACTIVO, t2.C_CUP,
       CASE WHEN t1.C_T_AREA = 'GM' THEN 'AAPP'
            WHEN t1.C_T_AREA = 'ED' THEN 'EEPP'
            WHEN t1.C_T_AREA = 'SL' THEN 'SSPP' END AS ACTIVIDAD,
       t2.T_TRANSACCION AS NIVEL,
       SUM(t1.DEVENG / 1000000) AS Dato
INTO #ejec_cnf_mun_a_an_a
FROM GOBGENER.dbo.EJECUCION_MUN t1
LEFT JOIN GOBGENER.dbo.CR18_T_SCN_1_MUN t2
    ON t1.C_TIPO_CUENTA = t2.C_TIPO_CUENTA AND t1.C_CUENTA = t2.C_CUENTA AND t1.C_ANALITICO = t2.C_ANALITICO
WHERE t1.C_CUENTA NOT IN ('05','13','24','33')
  AND t1.AÑO = {int(Year)} AND t1.AÑO < 2022
  AND t1.C_T_AREA NOT IN ('CM')
  AND t2.C_SCN NOT IN ('F22','F29','F3','F32','F41','F42','F5','F612','F7','F71','F79','K10','K11','K9')
GROUP BY t1.SUBSECTOR, t1.MONEDA, t1.AÑO, t1.C_TIPO_CUENTA, t1.C_CUENTA,
         SUBSTRING(t1.C_ANALITICO, 1, 2),
         t2.C_SCN, t2.C_CAGENTE, t2.C_TRANSACCION, t2.T_TRANSACCION, t2.N_CUENTA,
         t2.C_ACTIVO, t2.C_CUP,
         CASE WHEN t1.C_T_AREA = 'GM' THEN 'AAPP'
              WHEN t1.C_T_AREA = 'ED' THEN 'EEPP'
              WHEN t1.C_T_AREA = 'SL' THEN 'SSPP' END
'''
res = work_conn.execute(text(sql_ejec_cnf_mun_a_an_a))
_log('ejec_cnf_mun_a_an_a', res.rowcount)


In [ ]:
# /* Salida Ejecucion Municipal variables */
work_conn.execute(text('DROP TABLE IF EXISTS #ejec_cnf_mun_b_an_a'))
sql_ejec_cnf_mun_b_an_a = f'''
SELECT t1.SUBSECTOR, t1.MONEDA, t1.AÑO, t1.C_TIPO_CUENTA, t1.C_CUENTA,
       SUBSTRING(t1.C_ANALITICO, 1, 2) AS C_ITEM,
       t2.C_SCN, t2.C_CAGENTE, t2.C_TRANSACCION, t2.T_TRANSACCION, t2.N_CUENTA,
       t2.C_ACTIVO, t2.C_CUP,
       CASE WHEN t1.C_T_AREA = 'GM' THEN 'AAPP'
            WHEN t1.C_T_AREA = 'ED' THEN 'EEPP'
            WHEN t1.C_T_AREA = 'SL' THEN 'SSPP' END AS ACTIVIDAD,
       t2.T_TRANSACCION AS NIVEL,
       SUM(t1.DEVENG / 1000000) AS Dato
INTO #ejec_cnf_mun_b_an_a
FROM GOBGENER.dbo.EJECUCION_MUN t1
LEFT JOIN GOBGENER.dbo.CR18_T_SCN_2_MUN t2
    ON t1.C_CUENTA = t2.C_CUENTA AND t1.C_ANALITICO = t2.C_ANALITICO AND t1.C_T_AREA = t2.C_T_AREA
WHERE t1.C_CUENTA IN ('05','13','24','33')
  AND t1.AÑO = {int(Year)} AND t1.AÑO < 2022
  AND t1.C_T_AREA NOT IN ('CM')
  AND t2.C_SCN NOT IN ('F22','F29','F3','F32','F41','F42','F5','F612','F7','F71','F79','K10','K11','K9')
GROUP BY t1.SUBSECTOR, t1.MONEDA, t1.AÑO, t1.C_TIPO_CUENTA, t1.C_CUENTA,
         SUBSTRING(t1.C_ANALITICO, 1, 2),
         t2.C_SCN, t2.C_CAGENTE, t2.C_TRANSACCION, t2.T_TRANSACCION, t2.N_CUENTA,
         t2.C_ACTIVO, t2.C_CUP,
         CASE WHEN t1.C_T_AREA = 'GM' THEN 'AAPP'
              WHEN t1.C_T_AREA = 'ED' THEN 'EEPP'
              WHEN t1.C_T_AREA = 'SL' THEN 'SSPP' END
'''
res = work_conn.execute(text(sql_ejec_cnf_mun_b_an_a))
_log('ejec_cnf_mun_b_an_a', res.rowcount)


In [ ]:
# /* EJECUCION MUNICIPAL NUEVA */
# /* Salida Ejecucion Municipal predefinidos */
work_conn.execute(text('DROP TABLE IF EXISTS #ejec_cnf_mun_a_an_b'))
sql_ejec_cnf_mun_a_an_b = f'''
SELECT SUBSTRING(t1.SUBSECTOR, 2, 1) AS SUBSECTOR,
       SUBSTRING(t1.MONEDA, 3, 1) AS MONEDA,
       t1.AÑO,
       CASE WHEN t1.C_T_CUENTA = 1 THEN 'Y' WHEN t1.C_T_CUENTA = 2 THEN 'G' ELSE '' END AS C_TIPO_CUENTA,
       t1.C_CUENTA,
       SUBSTRING(t1.C_ANALITICO, 1, 2) AS C_ITEM,
       t2.C_SCN, t2.C_CAGENTE, t2.C_TRANSACCION, t2.T_TRANSACCION, t2.N_CUENTA,
       t2.C_ACTIVO, t2.C_CUP,
       CASE WHEN t1.C_SERVICIO = 1 THEN 'AAPP'
            WHEN t1.C_SERVICIO = 2 THEN 'EEPP'
            WHEN t1.C_SERVICIO = 3 THEN 'SSPP' END AS ACTIVIDAD,
       t2.T_TRANSACCION AS NIVEL,
       SUM(t1.DEVENGO / 1000000) AS Dato
INTO #ejec_cnf_mun_a_an_b
FROM GOBGENER.dbo.EJECUCION_MUN_P t1
LEFT JOIN GOBGENER.dbo.CR18_T_SCN_1_MUN_P t2
    ON t1.C_T_CUENTA = t2.C_T_CUENTA AND t1.C_CUENTA = t2.C_CUENTA AND t1.C_ANALITICO_2 = t2.C_ANALITICO_2
WHERE t1.C_CUENTA NOT IN ('05','13','24','33')
  AND t1.AÑO = {int(Year)} AND t1.AÑO >= 2022
  AND t1.C_SERVICIO NOT IN (4)
  AND t2.C_SCN NOT IN ('F22','F29','F3','F32','F41','F42','F5','F612','F7','F71','F79','K10','K11','K9')
GROUP BY t1.SUBSECTOR, t1.MONEDA, t1.AÑO,
         CASE WHEN t1.C_T_CUENTA = 1 THEN 'Y' WHEN t1.C_T_CUENTA = 2 THEN 'G' ELSE '' END,
         t1.C_CUENTA,
         SUBSTRING(t1.C_ANALITICO, 1, 2),
         t2.C_SCN, t2.C_CAGENTE, t2.C_TRANSACCION, t2.T_TRANSACCION, t2.N_CUENTA,
         t2.C_ACTIVO, t2.C_CUP,
         CASE WHEN t1.C_SERVICIO = 1 THEN 'AAPP'
              WHEN t1.C_SERVICIO = 2 THEN 'EEPP'
              WHEN t1.C_SERVICIO = 3 THEN 'SSPP' END
'''
res = work_conn.execute(text(sql_ejec_cnf_mun_a_an_b))
_log('ejec_cnf_mun_a_an_b', res.rowcount)


In [ ]:
# /* Salida Ejecucion Municipal variables */
work_conn.execute(text('DROP TABLE IF EXISTS #ejec_cnf_mun_b_an_b'))
sql_ejec_cnf_mun_b_an_b = f'''
SELECT SUBSTRING(t1.SUBSECTOR, 2, 1) AS SUBSECTOR,
       SUBSTRING(t1.MONEDA, 3, 1) AS MONEDA,
       t1.AÑO,
       CASE WHEN t1.C_T_CUENTA = 1 THEN 'Y' WHEN t1.C_T_CUENTA = 2 THEN 'G' ELSE '' END AS C_TIPO_CUENTA,
       t1.C_CUENTA,
       SUBSTRING(t1.C_ANALITICO, 1, 2) AS C_ITEM,
       t2.C_SCN, t2.C_CAGENTE, t2.C_TRANSACCION, t2.T_TRANSACCION, t2.N_CUENTA,
       t2.C_ACTIVO, t2.C_CUP,
       CASE WHEN t1.C_SERVICIO = 1 THEN 'AAPP'
            WHEN t1.C_SERVICIO = 2 THEN 'EEPP'
            WHEN t1.C_SERVICIO = 3 THEN 'SSPP' END AS ACTIVIDAD,
       t2.T_TRANSACCION AS NIVEL,
       SUM(t1.DEVENGO / 1000000) AS Dato
INTO #ejec_cnf_mun_b_an_b
FROM GOBGENER.dbo.EJECUCION_MUN_P t1
LEFT JOIN GOBGENER.dbo.CR18_T_SCN_2_MUN_P t2
    ON t1.C_CUENTA = t2.C_CUENTA AND t1.C_ANALITICO = t2.C_ANALITICO AND t1.C_SERVICIO = t2.C_SERVICIO
WHERE t1.C_CUENTA IN ('05','13','24','33')
  AND t1.AÑO = {int(Year)} AND t1.AÑO >= 2022
  AND t1.C_SERVICIO NOT IN (4)
  AND t2.C_SCN NOT IN ('F22','F29','F3','F32','F41','F42','F5','F612','F7','F71','F79','K10','K11','K9')
GROUP BY t1.SUBSECTOR, t1.MONEDA, t1.AÑO,
         CASE WHEN t1.C_T_CUENTA = 1 THEN 'Y' WHEN t1.C_T_CUENTA = 2 THEN 'G' ELSE '' END,
         t1.C_CUENTA,
         SUBSTRING(t1.C_ANALITICO, 1, 2),
         t2.C_SCN, t2.C_CAGENTE, t2.C_TRANSACCION, t2.T_TRANSACCION, t2.N_CUENTA,
         t2.C_ACTIVO, t2.C_CUP,
         CASE WHEN t1.C_SERVICIO = 1 THEN 'AAPP'
              WHEN t1.C_SERVICIO = 2 THEN 'EEPP'
              WHEN t1.C_SERVICIO = 3 THEN 'SSPP' END
'''
res = work_conn.execute(text(sql_ejec_cnf_mun_b_an_b))
_log('ejec_cnf_mun_b_an_b', res.rowcount)


In [ ]:
# /* Union de ambos tipos de ejecucion municipal */
work_conn.execute(text('DROP TABLE IF EXISTS #ejecucion_cnf_mun_an_a'))
work_conn.execute(text(
    'SELECT * INTO #ejecucion_cnf_mun_an_a FROM #ejec_cnf_mun_a_an_a '
    'UNION ALL SELECT * FROM #ejec_cnf_mun_b_an_a'
))
work_conn.execute(text('DROP TABLE IF EXISTS #ejecucion_cnf_mun_an_b'))
res = work_conn.execute(text(
    'SELECT * INTO #ejecucion_cnf_mun_an_b FROM #ejec_cnf_mun_a_an_b '
    'UNION ALL SELECT * FROM #ejec_cnf_mun_b_an_b'
))
_log('ejecucion_cnf_mun_an_a + ejecucion_cnf_mun_an_b', res.rowcount)


In [ ]:
# data EJECUCION_CNF_MUN_AN; SET EJECUCION_CNF_MUN_AN_A EJECUCION_CNF_MUN_AN_B; RUN;
work_conn.execute(text('DROP TABLE IF EXISTS #ejecucion_cnf_mun_an'))
res = work_conn.execute(text(
    'SELECT * INTO #ejecucion_cnf_mun_an FROM #ejecucion_cnf_mun_an_a '
    'UNION ALL SELECT * FROM #ejecucion_cnf_mun_an_b'
))
_log('ejecucion_cnf_mun_an', res.rowcount)

res = work_conn.execute(text("UPDATE #ejecucion_cnf_mun_an SET NIVEL = 'MUNICIPAL'"))
_log('UPDATE #ejecucion_cnf_mun_an (NIVEL=MUNICIPAL)', res.rowcount)


In [ ]:
# /*Reasignación de P3 para cuentas de subsector */
res = work_conn.execute(text("UPDATE #ejecucion_cnf_mun_an SET C_SCN = 'D73' WHERE C_SCN = 'P3'"))
_log('UPDATE #ejecucion_cnf_mun_an (P3->D73)', res.rowcount)


In [ ]:
# /* Eliminación de transacciones redundantes en consolidación */
work_conn.execute(text("DELETE FROM #ejecucion_cnf_mun_an WHERE C_SCN = 'D73' AND C_CAGENTE = 'S1313'"))
res = work_conn.execute(text("DELETE FROM #ejecucion_cnf_mun_an WHERE C_SCN = 'D93' AND C_CAGENTE = 'S1313'"))
_log('DELETE #ejecucion_cnf_mun_an (redundantes S1313)', res.rowcount)


In [ ]:
# /* Eliminacion de D63 por ya estar estimadas desde el origen (EDO), Gobierno Central */
res = work_conn.execute(text("DELETE FROM #ejecucion_cnf_mun_an WHERE C_SCN = 'D63'"))
_log('DELETE #ejecucion_cnf_mun_an (D63)', res.rowcount)


In [ ]:
# /* Union de ambos tipos de ejecucion, Gob Central + Municipal */
# Igual que antes, SQL Server no permite leer y sobreescribir la misma tabla: el resultado
# consolidado FINAL que reemplaza a WORK.EJECUCION_CNF_AN queda en #ejecucion_cnf_an_final
# (no en #ejecucion_cnf_an, que es un paso intermedio que se dropea más abajo)
work_conn.execute(text('DROP TABLE IF EXISTS #ejecucion_cnf_an_final'))
res = work_conn.execute(text(
    'SELECT * INTO #ejecucion_cnf_an_final FROM #ejecucion_cnf_an '
    'UNION ALL SELECT * FROM #ejecucion_cnf_mun_an'
))
_log('ejecucion_cnf_an_final', res.rowcount)


In [ ]:
# /* Se eliminan dividendos de CORFO porque se tomaran los del sectorialista de dividendos*/
res = work_conn.execute(text("DELETE FROM #ejecucion_cnf_an_final WHERE C_SCN = 'D421' AND ACTIVIDAD = 'CORFO'"))
_log('DELETE #ejecucion_cnf_an_final (CORFO D421)', res.rowcount)


In [ ]:
# /* Se eliminan subvenciones porque se toman las calculadas aparte */
res = work_conn.execute(text("DELETE FROM #ejecucion_cnf_an_final WHERE C_SCN = 'D39'"))
_log('DELETE #ejecucion_cnf_an_final (subvenciones D39)', res.rowcount)


In [ ]:
# /* Se elimina D41 dolar, porque se toma dado de contragente S2 de Balanza de Pagos */
res = work_conn.execute(text(
    "DELETE FROM #ejecucion_cnf_an_final WHERE C_SCN = 'D41' AND C_TIPO_CUENTA = 'Y' AND MONEDA = 'D'"
))
_log('DELETE #ejecucion_cnf_an_final (D41 dólar)', res.rowcount)


In [ ]:
# /* Se eliminan datos de producción dado que se ocupan los publicados */
res = work_conn.execute(text("DELETE FROM #ejecucion_cnf_an_final WHERE C_SCN IN ('D11','D12','P2')"))
_log('DELETE #ejecucion_cnf_an_final (producción publicada)', res.rowcount)


In [ ]:
# /* Estandarizacion de salida para calculo de cuenta */
work_conn.execute(text("UPDATE #ejecucion_cnf_an_final SET C_SCN = 'D11' WHERE C_SCN = 'D12'"))
work_conn.execute(text("UPDATE #ejecucion_cnf_an_final SET C_SCN = 'D61' WHERE C_SCN = 'D6111'"))
work_conn.execute(text("UPDATE #ejecucion_cnf_an_final SET C_SCN = 'D75' WHERE C_SCN = 'D74'"))
work_conn.execute(text("UPDATE #ejecucion_cnf_an_final SET C_SCN = 'D99' WHERE C_SCN IN ('D91','D92','D991')"))
work_conn.execute(text("UPDATE #ejecucion_cnf_an_final SET C_SCN = 'K2' WHERE C_SCN = 'P513'"))
work_conn.execute(text("UPDATE #ejecucion_cnf_an_final SET C_SCN = 'P51' WHERE C_SCN = 'P511'"))
res = work_conn.execute(text("UPDATE #ejecucion_cnf_an_final SET C_SCN = 'D42' WHERE C_SCN = 'D421'"))
_log('UPDATE #ejecucion_cnf_an_final (estandarización C_SCN)', res.rowcount)


In [ ]:
# DROP TABLE EJEC_CNF_MUN_A_AN_A, EJEC_CNF_MUN_B_AN_A, EJECUCION_CNF_MUN_AN, EJEC_CNF_MUN_A_AN_B,
# EJEC_CNF_MUN_B_AN_B, EJEC_CNF_MUN_AN_A, EJEC_CNF_MUN_AN_B
# se agrega también #ejecucion_cnf_an (el consolidado previo a unir con municipal, ya
# superado por #ejecucion_cnf_an_final) para no dejar tablas de sesión huérfanas
for _t in [
    '#ejec_cnf_mun_a_an_a', '#ejec_cnf_mun_b_an_a', '#ejecucion_cnf_mun_an',
    '#ejec_cnf_mun_a_an_b', '#ejec_cnf_mun_b_an_b', '#ejecucion_cnf_mun_an_a',
    '#ejecucion_cnf_mun_an_b', '#ejecucion_cnf_an',
]:
    work_conn.execute(text(f'DROP TABLE IF EXISTS {_t}'))


## APER_CNF_anual

Arma la salida detallada y agregada de ejecución CNF anual (Gobierno Central + Municipal), por entidad/partida/programa/analítico, aplicando reclasificaciones SCN predefinidas (Transantiago, cobre bruto, IPS, becas, glosas comunes GORE) y eliminando transacciones duplicadas, de producción o ya contabilizadas en otras fuentes (dividendos CORFO, subvenciones, D41 dólar).

*confianza: low · verificador: approve · SAS: PROC SQL encadenado: joins con GOBGENER, agregaciones GROUP BY, UPDATE/DELETE de reasignaciones predefinidas y uniones DATA step (SET) sobre tablas WORK temporales, todo contra la conexión de sesión (#tmp)*

In [ ]:
# ========= APER_CNF_anual =========
# /*** Salida para ANALISIS DESAGREGADO de indicadores GC para CNF anual ***/
# /*Adjunta imputación por ajuste en IFE LABORAL 2023 */
work_conn.execute(text("DROP TABLE IF EXISTS #ejecucion"))
sql_ejecucion = f"""
SELECT *
INTO #ejecucion
FROM (
    SELECT * FROM #ejec_cgr_y
    UNION ALL
    SELECT * FROM GOBGENER.dbo.AJUSTE_IFE_LABORAL
) u
WHERE AÑO = {int(Year)}
"""
res = work_conn.execute(text(sql_ejecucion))
_log("CREATE #ejecucion", res.rowcount)


In [ ]:
# /*Analiticos predefinidos */
work_conn.execute(text("DROP TABLE IF EXISTS #ejecucion_a_prelim_aper_an"))
sql_a_prelim = """
SELECT
    MAX(t1.SUBSECTOR) AS SUBSECTOR,
    MAX(t1.MONEDA) AS MONEDA,
    MAX(t1.AÑO) AS AÑO,
    (CASE WHEN t1.MES IN (1,2,3) THEN 1
          WHEN t1.MES IN (4,5,6) THEN 2
          WHEN t1.MES IN (7,8,9) THEN 3
          ELSE 4 END) AS TRIM,
    t1.MES, t1.C_ENTIDAD, t1.C_PARTIDA, t1.C_CAPITULO, t1.C_PROGRAMA, t1.C_TIPO_CUENTA, t1.C_CUENTA,
    t1.C_ITEM, t1.C_ASIGNACION, t1.C_ANALITICO, t1.N_ANALITICO,
    t2.C_SCN, t2.C_CAGENTE, t2.C_TRANSACCION, t2.T_TRANSACCION, t2.N_CUENTA, t2.C_ACTIVO, t2.C_CUP,
    t4.ACTIVIDAD, t4.NIVEL,
    SUM(t1.DEVENG * t3.DATO / 1000000.0) AS DEVENGO
INTO #ejecucion_a_prelim_aper_an
FROM #ejecucion t1
LEFT JOIN GOBGENER.dbo.CR18_T_SCN_1 t2
    ON t1.C_TIPO_CUENTA = t2.T_CUENTA AND t1.C_CUENTA = t2.C_CUENTA AND t1.C_ANALITICO = t2.C_ANALITICO
LEFT JOIN #prom_mes t3
    ON t1.MONEDA = t3.MONEDA AND t1.MES = t3.MES AND t1.AÑO = t3.AÑO
LEFT JOIN GOBGENER.dbo.CR18_APERTURA t4
    ON t1.C_ENTIDAD = t4.ENTIDAD
WHERE t1.C_CUENTA NOT IN ('05','13','24','33') AND t1.C_ENTIDAD NOT IN (5601,10201)
GROUP BY t1.MES, t1.C_ENTIDAD, t1.C_PARTIDA, t1.C_CAPITULO, t1.C_PROGRAMA, t1.C_TIPO_CUENTA, t1.C_CUENTA,
    t1.C_ITEM, t1.C_ASIGNACION, t1.C_ANALITICO, t1.N_ANALITICO, t2.C_SCN, t2.C_CAGENTE, t2.C_TRANSACCION,
    t2.T_TRANSACCION, t2.N_CUENTA, t2.C_ACTIVO, t2.C_CUP, t4.ACTIVIDAD, t4.NIVEL
"""
res = work_conn.execute(text(sql_a_prelim))
_log("CREATE #ejecucion_a_prelim_aper_an", res.rowcount)


In [ ]:
work_conn.execute(text(
    "DELETE FROM #ejecucion_a_prelim_aper_an WHERE C_CUENTA='32' AND C_ANALITICO IN ('04001000','04002000')"
))
# /*Analiticos predefinidos: AJUSTE COYUNTURA - se elimina de indicador transacción CORFO
# que se desagrega anualmente entre ventas y D45 */
res = work_conn.execute(text(
    "DELETE FROM #ejecucion_a_prelim_aper_an "
    "WHERE C_ENTIDAD=3801 AND C_CUENTA='06' AND C_ITEM='01' AND C_ASIGNACION='000' AND C_ANALITICO IN ('01000000')"
))
_log("DELETE #ejecucion_a_prelim_aper_an (duplicados 32 / coyuntura CORFO)", res.rowcount)


In [ ]:
# /*Analiticos predefinidos: reasignación D39 2% constitucional 2008 para Transantiago */
_where_transantiago = (
    "C_ENTIDAD=19703 AND C_CUENTA='26' AND C_ITEM='03' AND C_ASIGNACION='000' "
    "AND C_ANALITICO IN ('03000000') AND MONEDA IN ('P') AND AÑO=2008 AND MES IN (10,11,12)"
)
work_conn.execute(text(f"UPDATE #ejecucion_a_prelim_aper_an SET C_CAGENTE='S11' WHERE {_where_transantiago}"))
work_conn.execute(text(f"UPDATE #ejecucion_a_prelim_aper_an SET C_SCN='D39' WHERE {_where_transantiago}"))
res = work_conn.execute(text(f"UPDATE #ejecucion_a_prelim_aper_an SET C_TRANSACCION='4139' WHERE {_where_transantiago}"))
_log("UPDATE #ejecucion_a_prelim_aper_an (Transantiago D39)", res.rowcount)


In [ ]:
# /*AJUSTE COBRE BRUTO (Financiamiento Capacidades Estrategicas Defensa Nacional) - reasignacion entre D75 y D42 */
_where_cobre_bruto = (
    "C_ENTIDAD=19701 AND C_CUENTA='08' AND C_ITEM='99' AND C_ASIGNACION='999' "
    "AND SUBSTRING(C_ANALITICO,1,5) = '99999' AND MONEDA IN ('D') AND AÑO>2019"
)
work_conn.execute(text(f"UPDATE #ejecucion_a_prelim_aper_an SET C_SCN='D42' WHERE {_where_cobre_bruto}"))
work_conn.execute(text(f"UPDATE #ejecucion_a_prelim_aper_an SET C_TRANSACCION='42222' WHERE {_where_cobre_bruto}"))
res = work_conn.execute(text(f"UPDATE #ejecucion_a_prelim_aper_an SET N_CUENTA='Cobre bruto' WHERE {_where_cobre_bruto}"))
_log("UPDATE #ejecucion_a_prelim_aper_an (Cobre bruto)", res.rowcount)


In [ ]:
# /*Analiticos predefinidos (sigue)*/
work_conn.execute(text("DROP TABLE IF EXISTS #ejecucion_a_aper_an"))
sql_a_aper_an = """
SELECT t1.SUBSECTOR, t1.MONEDA, t1.AÑO, t1.C_ENTIDAD, t1.C_PARTIDA, t1.C_CAPITULO, t1.C_PROGRAMA,
    t1.C_TIPO_CUENTA, t1.C_CUENTA, t1.C_ITEM, t1.C_ASIGNACION, t1.C_ANALITICO, t1.N_ANALITICO,
    t1.C_SCN, t1.C_CAGENTE, t1.C_TRANSACCION, t1.T_TRANSACCION, t1.N_CUENTA, t1.C_ACTIVO, t1.C_CUP,
    t1.ACTIVIDAD, t1.NIVEL,
    SUM(t1.DEVENGO) AS Dato
INTO #ejecucion_a_aper_an
FROM #ejecucion_a_prelim_aper_an t1
WHERE (t1.C_SCN IS NULL OR t1.C_SCN NOT IN ('F22','F29','F3','F32','F41','F42','F5','F612','F7','F71','F79','K10','K11','K9'))
GROUP BY t1.SUBSECTOR, t1.MONEDA, t1.AÑO, t1.C_ENTIDAD, t1.C_PARTIDA, t1.C_CAPITULO, t1.C_PROGRAMA,
    t1.C_TIPO_CUENTA, t1.C_CUENTA, t1.C_ITEM, t1.C_ASIGNACION, t1.C_ANALITICO, t1.N_ANALITICO,
    t1.C_SCN, t1.C_CAGENTE, t1.C_TRANSACCION, t1.T_TRANSACCION, t1.N_CUENTA, t1.C_ACTIVO, t1.C_CUP,
    t1.ACTIVIDAD, t1.NIVEL
"""
res = work_conn.execute(text(sql_a_aper_an))
_log("CREATE #ejecucion_a_aper_an", res.rowcount)


In [ ]:
# /*Analiticos fijos cuenta 32 */
work_conn.execute(text("DROP TABLE IF EXISTS #ejecucion_b_aper_an"))
sql_b_aper_an = """
SELECT t1.SUBSECTOR, t1.MONEDA, t1.AÑO, t1.C_ENTIDAD, t1.C_PARTIDA, t1.C_CAPITULO, t1.C_PROGRAMA,
    t1.C_TIPO_CUENTA, t1.C_CUENTA, t1.C_ITEM, t1.C_ASIGNACION, t1.C_ANALITICO, t1.N_ANALITICO,
    t2.C_SCN, t2.C_CAGENTE, t2.C_TRANSACCION, t2.T_TRANSACCION, t2.N_CUENTA, t2.C_ACTIVO, t2.C_CUP,
    t4.ACTIVIDAD, t4.NIVEL,
    SUM(t1.DEVENG * t3.DATO / 1000000.0) AS Dato
INTO #ejecucion_b_aper_an
FROM #ejecucion t1
LEFT JOIN GOBGENER.dbo.T_SCN_1 t2
    ON t1.C_TIPO_CUENTA = t2.T_CUENTA AND t1.C_CUENTA = t2.C_CUENTA AND t1.C_ITEM = t2.C_ITEM
       AND t1.C_ASIGNACION = t2.C_ASIGNACION AND t1.C_ANALITICO = t2.C_ANALITICO
LEFT JOIN #prom_mes t3
    ON t1.MONEDA = t3.MONEDA AND t1.MES = t3.MES AND t1.AÑO = t3.AÑO
LEFT JOIN GOBGENER.dbo.CR18_APERTURA t4
    ON t1.C_ENTIDAD = t4.ENTIDAD
WHERE t1.C_CUENTA = '32'
    AND (t2.C_SCN IS NULL OR t2.C_SCN NOT IN ('F22','F29','F3','F32','F41','F42','F5','F612','F7','F71','F79','K10','K11','K9'))
    AND t1.C_ANALITICO IN ('04001000','04002000')
    AND t1.C_ENTIDAD NOT IN (5601,10201)
GROUP BY t1.SUBSECTOR, t1.MONEDA, t1.AÑO, t1.C_ENTIDAD, t1.C_PARTIDA, t1.C_CAPITULO, t1.C_PROGRAMA,
    t1.C_TIPO_CUENTA, t1.C_CUENTA, t1.C_ITEM, t1.C_ASIGNACION, t1.C_ANALITICO, t1.N_ANALITICO,
    t2.C_SCN, t2.C_CAGENTE, t2.C_TRANSACCION, t2.T_TRANSACCION, t2.N_CUENTA, t2.C_ACTIVO, t2.C_CUP,
    t4.ACTIVIDAD, t4.NIVEL
"""
res = work_conn.execute(text(sql_b_aper_an))
_log("CREATE #ejecucion_b_aper_an", res.rowcount)


In [ ]:
# /*analíticos variables*/
work_conn.execute(text("DROP TABLE IF EXISTS #ejecucion_c_aper_an_prelim"))
sql_c_prelim = """
SELECT t1.SUBSECTOR, t1.MONEDA, t1.AÑO, t1.MES, t1.C_ENTIDAD, t1.C_PARTIDA, t1.C_CAPITULO, t1.C_PROGRAMA,
    t1.C_TIPO_CUENTA, t1.C_CUENTA, t1.C_ITEM, t1.C_ASIGNACION, t1.C_ANALITICO, t1.N_ANALITICO,
    t2.C_SCN, t2.C_CAGENTE, t2.C_TRANSACCION, t2.T_TRANSACCION, t2.N_CUENTA, t2.C_ACTIVO, t2.C_CUP,
    t4.ACTIVIDAD, t4.NIVEL,
    SUM(t1.DEVENG * t3.DATO / 1000000.0) AS Dato
INTO #ejecucion_c_aper_an_prelim
FROM #ejecucion t1
LEFT JOIN GOBGENER.dbo.CR18_T_SCN_2 t2
    ON t1.C_PARTIDA = t2.C_PARTIDA AND t1.C_CAPITULO = t2.C_CAPITULO AND t1.C_PROGRAMA = t2.C_PROGRAMA
       AND t1.C_ENTIDAD = t2.ENTIDAD AND t1.C_TIPO_CUENTA = t2.T_CUENTA AND t1.C_CUENTA = t2.C_CUENTA
       AND t1.C_ITEM = t2.C_ITEM AND t1.C_ASIGNACION = t2.C_ASIGNACION AND t1.C_ANALITICO = t2.C_ANALITICO
LEFT JOIN #prom_mes t3
    ON t1.MONEDA = t3.MONEDA AND t1.MES = t3.MES AND t1.AÑO = t3.AÑO
LEFT JOIN GOBGENER.dbo.CR18_APERTURA t4
    ON t1.C_ENTIDAD = t4.ENTIDAD
WHERE t1.C_CUENTA IN ('05','13','24','33')
    AND t2.C_SCN NOT IN ('F22','F29','F3','F32','F41','F42','F5','F612','F7','F71','F79','K10','K11','K9','')
    AND t2.OBS NOT IN ('CR18_difcoy','CR18_difact')
    AND t1.C_ENTIDAD NOT IN (5601,10201)
    AND t2.C_SCN NOT IN ('TC')
GROUP BY t1.SUBSECTOR, t1.MONEDA, t1.AÑO, t1.MES, t1.C_ENTIDAD, t1.C_PARTIDA, t1.C_CAPITULO, t1.C_PROGRAMA,
    t1.C_TIPO_CUENTA, t1.C_CUENTA, t1.C_ITEM, t1.C_ASIGNACION, t1.C_ANALITICO, t1.N_ANALITICO,
    t2.C_SCN, t2.C_CAGENTE, t2.C_TRANSACCION, t2.T_TRANSACCION, t2.N_CUENTA, t2.C_ACTIVO, t2.C_CUP,
    t4.ACTIVIDAD, t4.NIVEL
"""
res = work_conn.execute(text(sql_c_prelim))
_log("CREATE #ejecucion_c_aper_an_prelim", res.rowcount)


In [ ]:
# /*Analiticos variables: reasignación contragente ayudas IPS */
# /* Bono Ayuda Familiar (18O) */
work_conn.execute(text(
    "UPDATE #ejecucion_c_aper_an_prelim SET C_CAGENTE='S14' "
    "WHERE C_ENTIDAD=24101 AND C_CUENTA='24' AND C_ITEM='01' AND C_ASIGNACION='006' "
    "AND C_ANALITICO IN ('01006000') AND MONEDA IN ('P') AND AÑO=2019 AND MES=12"
))
# /*Bono Covid */
work_conn.execute(text(
    "UPDATE #ejecucion_c_aper_an_prelim SET C_CAGENTE='S14' "
    "WHERE C_ENTIDAD=24101 AND C_CUENTA='24' AND C_ITEM='01' AND C_ASIGNACION='006' "
    "AND C_ANALITICO IN ('01006000') AND MONEDA IN ('P') AND AÑO=2020 AND MES=4"
))
# /* Reasignación nivel de gobierno para Programa Ampliado de Inmunizaciones */
work_conn.execute(text(
    "UPDATE #ejecucion_c_aper_an_prelim SET NIVEL='MUNICIPAL' "
    "WHERE C_ENTIDAD IN (11105,22701,22750) AND C_CUENTA='24' AND C_ITEM='01' AND C_ASIGNACION='004'"
))
# /* Reasignación nivel de D63 municipal */
work_conn.execute(text(
    "UPDATE #ejecucion_c_aper_an_prelim SET NIVEL='MUNICIPAL' "
    "WHERE NIVEL='REGIONAL' AND C_SCN IN ('D63','D63-P2','D63-P') AND C_ITEM='03' "
    "AND C_ASIGNACION IN ('001','003','006','011','013','015','016','017','018','020','021','100','101')"
))
# /* Beca Especial Carbon (previo 2020) */
_where_beca_carbon = (
    "C_ENTIDAD=6403 AND C_CUENTA='24' AND C_ITEM='01' AND C_ASIGNACION='270' "
    "AND C_ANALITICO IN ('01270000') AND MONEDA IN ('P') AND AÑO<2020"
)
work_conn.execute(text(f"UPDATE #ejecucion_c_aper_an_prelim SET C_CAGENTE='S14' WHERE {_where_beca_carbon}"))
work_conn.execute(text(f"UPDATE #ejecucion_c_aper_an_prelim SET C_SCN='D75' WHERE {_where_beca_carbon}"))
res = work_conn.execute(text(f"UPDATE #ejecucion_c_aper_an_prelim SET C_TRANSACCION='4345' WHERE {_where_beca_carbon}"))
_log("UPDATE #ejecucion_c_aper_an_prelim (reasignaciones IPS/Carbon)", res.rowcount)


In [ ]:
work_conn.execute(text("DROP TABLE IF EXISTS #ejecucion_c_aper_an"))
sql_c_aper_an = """
SELECT t1.SUBSECTOR, t1.MONEDA, t1.AÑO, t1.C_ENTIDAD, t1.C_PARTIDA, t1.C_CAPITULO, t1.C_PROGRAMA,
    t1.C_TIPO_CUENTA, t1.C_CUENTA, t1.C_ITEM, t1.C_ASIGNACION, t1.C_ANALITICO, t1.N_ANALITICO,
    t1.C_SCN, t1.C_CAGENTE, t1.C_TRANSACCION, t1.T_TRANSACCION, t1.N_CUENTA, t1.C_ACTIVO, t1.C_CUP,
    t1.ACTIVIDAD, t1.NIVEL,
    SUM(t1.Dato) AS Dato
INTO #ejecucion_c_aper_an
FROM #ejecucion_c_aper_an_prelim t1
GROUP BY t1.SUBSECTOR, t1.MONEDA, t1.AÑO, t1.C_ENTIDAD, t1.C_PARTIDA, t1.C_CAPITULO, t1.C_PROGRAMA,
    t1.C_TIPO_CUENTA, t1.C_CUENTA, t1.C_ITEM, t1.C_ASIGNACION, t1.C_ANALITICO, t1.N_ANALITICO,
    t1.C_SCN, t1.C_CAGENTE, t1.C_TRANSACCION, t1.T_TRANSACCION, t1.N_CUENTA, t1.C_ACTIVO, t1.C_CUP,
    t1.ACTIVIDAD, t1.NIVEL
"""
res = work_conn.execute(text(sql_c_aper_an))
_log("CREATE #ejecucion_c_aper_an", res.rowcount)


In [ ]:
# /* Transacciones diferenciadas de coyuntura, con clasificacion anual */
# el JOIN con COFOG (t5) no se selecciona en el SAS original pero puede multiplicar filas
# si hay más de un match por (ENTIDAD, C_PARTIDA, C_CAPITULO, C_PROGRAMA); se preserva por fidelidad
work_conn.execute(text("DROP TABLE IF EXISTS #ejecucion_d_aper_an_prelim"))
sql_d_prelim = """
SELECT t1.SUBSECTOR, t1.MONEDA, t1.AÑO,
    (CASE WHEN t1.MES IN (1,2,3) THEN 1
          WHEN t1.MES IN (4,5,6) THEN 2
          WHEN t1.MES IN (7,8,9) THEN 3
          ELSE 4 END) AS TRIM,
    t1.MES, t1.C_ENTIDAD, t1.C_PARTIDA, t1.C_CAPITULO, t1.C_PROGRAMA, t1.C_TIPO_CUENTA, t1.C_CUENTA,
    t1.C_ITEM, t1.C_ASIGNACION, t1.C_ANALITICO, t1.N_ANALITICO,
    t2.C_SCN, t2.C_CAGENTE, t2.C_TRANSACCION, t2.T_TRANSACCION, t2.N_CUENTA, t2.C_ACTIVO, t2.C_CUP,
    t4.ACTIVIDAD, t4.NIVEL,
    (t1.DEVENG * t3.DATO / 1000000.0) AS Dato
INTO #ejecucion_d_aper_an_prelim
FROM #ejecucion t1
LEFT JOIN GOBGENER.dbo.CR18_T_SCN_2 t2
    ON t1.C_PARTIDA = t2.C_PARTIDA AND t1.C_CAPITULO = t2.C_CAPITULO AND t1.C_PROGRAMA = t2.C_PROGRAMA
       AND t1.C_ENTIDAD = t2.ENTIDAD AND t1.C_TIPO_CUENTA = t2.T_CUENTA AND t1.C_CUENTA = t2.C_CUENTA
       AND t1.C_ITEM = t2.C_ITEM AND t1.C_ASIGNACION = t2.C_ASIGNACION AND t1.C_ANALITICO = t2.C_ANALITICO
LEFT JOIN #prom_mes t3
    ON t1.MONEDA = t3.MONEDA AND t1.MES = t3.MES AND t1.AÑO = t3.AÑO
LEFT JOIN GOBGENER.dbo.CR18_APERTURA t4
    ON t1.C_ENTIDAD = t4.ENTIDAD
LEFT JOIN GOBGENER.dbo.COFOG t5
    ON t1.C_ENTIDAD = t5.ENTIDAD AND t1.C_PARTIDA = t5.C_PARTIDA AND t1.C_CAPITULO = t5.C_CAPITULO
       AND t1.C_PROGRAMA = t5.C_PROGRAMA
WHERE t2.OBS IN ('CR18_difcoy') AND t1.C_ENTIDAD NOT IN (5601,10201) AND t2.C_SCN NOT IN ('TC')
    AND t2.C_ACTIVO NOT IN ('F22','F29','F3','F32','F41','F42','F5','F612','F7','F71','F79','K10','K11','K9')
"""
res = work_conn.execute(text(sql_d_prelim))
_log("CREATE #ejecucion_d_aper_an_prelim", res.rowcount)


In [ ]:
# /* Reasignación nivel de D63 municipal */
work_conn.execute(text(
    "UPDATE #ejecucion_d_aper_an_prelim SET NIVEL='MUNICIPAL' "
    "WHERE NIVEL='REGIONAL' AND C_SCN IN ('D63','D63-P2','D63-P') AND C_ITEM='03' "
    "AND C_ASIGNACION IN ('001','003','006','011','013','015','016','017','018','020','021','100','101')"
))

# /* Glosas comunes GR: se separan para aplicarles el ponderador COFOG aparte */
work_conn.execute(text("DROP TABLE IF EXISTS #ejecucion_da_aper_an_prelim"))
work_conn.execute(text(
    "SELECT t1.* INTO #ejecucion_da_aper_an_prelim FROM #ejecucion_d_aper_an_prelim t1 "
    "WHERE t1.C_SCN IN ('D63-P2','D63-P') AND t1.NIVEL IN ('REGIONAL','MUNICIPAL')"
))
res = work_conn.execute(text(
    "DELETE FROM #ejecucion_d_aper_an_prelim WHERE C_SCN IN ('D63-P2','D63-P') AND NIVEL IN ('REGIONAL','MUNICIPAL')"
))
_log("DELETE #ejecucion_d_aper_an_prelim (glosas comunes movidas a DA)", res.rowcount)


In [ ]:
# /*REALIZA APERTURA GLOSAS COFOG DE GLOSAS COMUNES DE GORES*/
# Dato = t2.Dato*t1.POND no está agregado ni en el GROUP BY del SAS original (PROC SQL toma
# el primer registro del grupo); se aproxima con MAX() asumiendo que es constante por grupo
work_conn.execute(text("DROP TABLE IF EXISTS #ejecucion_da_aper_an"))
sql_da = """
SELECT t2.SUBSECTOR, t2.MONEDA, t2.AÑO, t2.TRIM, t2.MES, t2.C_ENTIDAD, t2.C_PARTIDA, t2.C_CAPITULO,
    t2.C_PROGRAMA, t2.C_TIPO_CUENTA, t2.C_CUENTA, t2.C_ITEM, t2.C_ASIGNACION, t2.C_ANALITICO, t2.N_ANALITICO,
    t1.C_SCN, t2.C_CAGENTE, t2.C_TRANSACCION, t2.T_TRANSACCION, t2.N_CUENTA, t2.C_ACTIVO, t2.C_CUP,
    t2.ACTIVIDAD, t2.NIVEL,
    MAX(t2.Dato * t1.POND) AS Dato
INTO #ejecucion_da_aper_an
FROM GOBGENER.dbo.EST_GLOSASCOM t1
LEFT JOIN #ejecucion_da_aper_an_prelim t2
    ON t2.AÑO = t1.AÑO AND t1.C_ENTIDAD = t2.C_ENTIDAD AND t1.C_ITEM = t2.C_ITEM
WHERE t2.NIVEL IS NOT NULL
GROUP BY t2.SUBSECTOR, t2.MONEDA, t2.AÑO, t2.TRIM, t2.MES, t2.C_ENTIDAD, t2.C_PARTIDA, t2.C_CAPITULO,
    t2.C_PROGRAMA, t2.C_TIPO_CUENTA, t2.C_CUENTA, t2.C_ITEM, t2.C_ASIGNACION, t2.C_ANALITICO, t2.N_ANALITICO,
    t1.C_SCN, t2.C_CAGENTE, t2.C_TRANSACCION, t2.T_TRANSACCION, t2.N_CUENTA, t2.C_ACTIVO, t2.C_CUP,
    t2.ACTIVIDAD, t2.NIVEL
"""
res = work_conn.execute(text(sql_da))
_log("CREATE #ejecucion_da_aper_an", res.rowcount)


In [ ]:
# /*UNE TABLAS PROCESO d*/ (DATA step que reescribe EJECUCION_D_aper_AN_prelim con sus propios datos + DA)
work_conn.execute(text("DROP TABLE IF EXISTS #stage_d_prelim"))
work_conn.execute(text("SELECT * INTO #stage_d_prelim FROM #ejecucion_d_aper_an_prelim"))
work_conn.execute(text("DROP TABLE IF EXISTS #ejecucion_d_aper_an_prelim"))
res = work_conn.execute(text(
    "SELECT * INTO #ejecucion_d_aper_an_prelim FROM #stage_d_prelim "
    "UNION ALL SELECT * FROM #ejecucion_da_aper_an"
))
work_conn.execute(text("DROP TABLE IF EXISTS #stage_d_prelim"))
_log("UNION #ejecucion_d_aper_an_prelim + #ejecucion_da_aper_an", res.rowcount)


In [ ]:
# /*RESUMEN ANUAL DE PROCESO D*/
work_conn.execute(text("DROP TABLE IF EXISTS #ejecucion_d_aper_an"))
sql_d_aper_an = """
SELECT t1.SUBSECTOR, t1.MONEDA, t1.AÑO, t1.C_ENTIDAD, t1.C_PARTIDA, t1.C_CAPITULO, t1.C_PROGRAMA,
    t1.C_TIPO_CUENTA, t1.C_CUENTA, t1.C_ITEM, t1.C_ASIGNACION, t1.C_ANALITICO, t1.N_ANALITICO,
    t1.C_SCN, t1.C_CAGENTE, t1.C_TRANSACCION, t1.T_TRANSACCION, t1.N_CUENTA, t1.C_ACTIVO, t1.C_CUP,
    t1.ACTIVIDAD, t1.NIVEL,
    SUM(t1.Dato) AS Dato
INTO #ejecucion_d_aper_an
FROM #ejecucion_d_aper_an_prelim t1
WHERE t1.NIVEL IS NOT NULL
GROUP BY t1.SUBSECTOR, t1.MONEDA, t1.AÑO, t1.C_ENTIDAD, t1.C_PARTIDA, t1.C_CAPITULO, t1.C_PROGRAMA,
    t1.C_TIPO_CUENTA, t1.C_CUENTA, t1.C_ITEM, t1.C_ASIGNACION, t1.C_ANALITICO, t1.N_ANALITICO,
    t1.C_SCN, t1.C_CAGENTE, t1.C_TRANSACCION, t1.T_TRANSACCION, t1.N_CUENTA, t1.C_ACTIVO, t1.C_CUP,
    t1.ACTIVIDAD, t1.NIVEL
"""
res = work_conn.execute(text(sql_d_aper_an))
_log("CREATE #ejecucion_d_aper_an", res.rowcount)


In [ ]:
# /* Transacciones diferenciadas por actividad */
work_conn.execute(text("DROP TABLE IF EXISTS #ejecucion_f_aper_an"))
sql_f = """
SELECT t1.SUBSECTOR, t1.MONEDA, t1.AÑO, t1.C_ENTIDAD, t1.C_PARTIDA, t1.C_CAPITULO, t1.C_PROGRAMA,
    t1.C_TIPO_CUENTA, t1.C_CUENTA, t1.C_ITEM, t1.C_ASIGNACION, t1.C_ANALITICO, t1.N_ANALITICO,
    t2.C_SCN, t2.C_CAGENTE, t2.C_TRANSACCION, t2.T_TRANSACCION, t2.N_CUENTA, t2.C_ACTIVO, t2.C_CUP,
    t2.C_CUP AS ACTIVIDAD, t4.NIVEL,
    SUM(t1.DEVENG * t3.DATO / 1000000.0) AS Dato
INTO #ejecucion_f_aper_an
FROM #ejecucion t1
LEFT JOIN GOBGENER.dbo.CR18_T_SCN_2 t2
    ON t1.C_PARTIDA = t2.C_PARTIDA AND t1.C_CAPITULO = t2.C_CAPITULO AND t1.C_PROGRAMA = t2.C_PROGRAMA
       AND t1.C_ENTIDAD = t2.ENTIDAD AND t1.C_TIPO_CUENTA = t2.T_CUENTA AND t1.C_CUENTA = t2.C_CUENTA
       AND t1.C_ITEM = t2.C_ITEM AND t1.C_ASIGNACION = t2.C_ASIGNACION AND t1.C_ANALITICO = t2.C_ANALITICO
LEFT JOIN #prom_mes t3
    ON t1.MONEDA = t3.MONEDA AND t1.MES = t3.MES AND t1.AÑO = t3.AÑO
LEFT JOIN GOBGENER.dbo.CR18_APERTURA t4
    ON t1.C_ENTIDAD = t4.ENTIDAD
WHERE t2.OBS IN ('CR18_difact') AND t1.C_ENTIDAD NOT IN (5601,10201) AND t2.C_SCN NOT IN ('TC')
    AND t2.C_SCN NOT IN ('F22','F29','F3','F32','F41','F42','F5','F612','F7','F71','F79','K10','K11','K9')
GROUP BY t1.SUBSECTOR, t1.MONEDA, t1.AÑO, t1.C_ENTIDAD, t1.C_PARTIDA, t1.C_CAPITULO, t1.C_PROGRAMA,
    t1.C_TIPO_CUENTA, t1.C_CUENTA, t1.C_ITEM, t1.C_ASIGNACION, t1.C_ANALITICO, t1.N_ANALITICO,
    t2.C_SCN, t2.C_CAGENTE, t2.C_TRANSACCION, t2.T_TRANSACCION, t2.N_CUENTA, t2.C_ACTIVO, t2.C_CUP,
    t4.NIVEL
"""
res = work_conn.execute(text(sql_f))
_log("CREATE #ejecucion_f_aper_an", res.rowcount)


In [ ]:
# /*UNE TABLAS DE TRANSFERENCIAS CONSOLIDABLES DETALLADO ANUAL*/
# se asume igual estructura estándar (23 columnas) en las 4 fuentes dejadas por los nodos de TCons
_cols_estandar = (
    "SUBSECTOR, MONEDA, AÑO, C_ENTIDAD, C_PARTIDA, C_CAPITULO, C_PROGRAMA, C_TIPO_CUENTA, C_CUENTA, "
    "C_ITEM, C_ASIGNACION, C_ANALITICO, N_ANALITICO, C_SCN, C_CAGENTE, C_TRANSACCION, T_TRANSACCION, "
    "N_CUENTA, 'TC' AS C_ACTIVO, C_CUP, ACTIVIDAD, NIVEL, Dato"
)
work_conn.execute(text("DROP TABLE IF EXISTS #ejecucion_e_aper_an"))
sql_e = (
    f"SELECT {_cols_estandar} INTO #ejecucion_e_aper_an FROM #ejec_e_mixt_aper_an "
    f"UNION ALL SELECT {_cols_estandar} FROM #ejec_e_mgcg_aper_an "
    f"UNION ALL SELECT {_cols_estandar} FROM #ejec_e_mm_aper_an "
    f"UNION ALL SELECT {_cols_estandar} FROM #ejec_e_mgg_aper_an"
)
res = work_conn.execute(text(sql_e))
_log("CREATE #ejecucion_e_aper_an", res.rowcount)


In [ ]:
# /*UNE TABLAS SALIDA CNF ANUAL*/ (A,B,C,D,E,F comparten el mismo esquema de salida)
work_conn.execute(text("DROP TABLE IF EXISTS #ejecucion_ext_cgr_aper_prev_an"))
res = work_conn.execute(text(
    "SELECT * INTO #ejecucion_ext_cgr_aper_prev_an FROM #ejecucion_a_aper_an "
    "UNION ALL SELECT * FROM #ejecucion_b_aper_an "
    "UNION ALL SELECT * FROM #ejecucion_c_aper_an "
    "UNION ALL SELECT * FROM #ejecucion_d_aper_an "
    "UNION ALL SELECT * FROM #ejecucion_e_aper_an "
    "UNION ALL SELECT * FROM #ejecucion_f_aper_an"
))
_log("CREATE #ejecucion_ext_cgr_aper_prev_an", res.rowcount)


In [ ]:
# DROP TABLE de intermedias ya consumidas (replica el PROC SQL DROP TABLE del SAS original,
# incluye dos nombres que el SAS nunca llegó a crear: DROP TABLE IF EXISTS no falla por eso)
for _t in ["#ejecucion_a_prelim_aper_an", "#ejecucion_a_aper_an", "#ejecucion_b_aper_an",
           "#ejecucion_c_aper_an", "#ejecucion_c_aper_an_prelim", "#ejecucion_d_aper_an",
           "#ejecucion_d_aper_an_prelim", "#ejecucion_da_aper_an", "#ejecucion_da_aper_an_prelim",
           "#ejecucion_e_aper_an", "#ejecucion_e_aper_prev_an", "#ejecucion_f_aper_an",
           "#ejecucion_ext_dip_aper_prev_an"]:
    work_conn.execute(text(f"DROP TABLE IF EXISTS {_t}"))


In [ ]:
# /*Incorporacion de imputaciones permanentes*/
work_conn.execute(text("DROP TABLE IF EXISTS #cr18_apertura_pacapr"))
res = work_conn.execute(text(
    "SELECT DISTINCT t1.ACTIVIDAD, t2.C_PARTIDA, t2.C_CAPITULO, t2.C_PROGRAMA, t1.NIVEL "
    "INTO #cr18_apertura_pacapr "
    "FROM GOBGENER.dbo.CR18_APERTURA t1 "
    "LEFT JOIN GOBGENER.dbo.ENTIDADES t2 ON t1.ENTIDAD = t2.C_ENTIDAD"
))
_log("CREATE #cr18_apertura_pacapr", res.rowcount)


In [ ]:
work_conn.execute(text("DROP TABLE IF EXISTS #ejec_ext_dip_aper_imput_perm_an"))
sql_imput_perm = f"""
SELECT t1.SUBSECTOR, t1.MONEDA, t1.AÑO, t1.C_ENTIDAD, t1.C_PARTIDA, t1.C_CAPITULO, t1.C_PROGRAMA,
    t1.C_TIPO_CUENTA, t1.C_CUENTA, t1.C_ITEM, t1.C_ASIGNACION, t1.C_ANALITICO, t1.N_ANALITICO,
    t1.C_SCN, t1.C_CAGENTE, t1.C_TRANSACCION, t1.T_TRANSACCION, t1.N_CUENTA, t1.C_ACTIVO, t1.C_CUP,
    t2.ACTIVIDAD, t2.NIVEL,
    SUM(t1.Dato) AS Dato
INTO #ejec_ext_dip_aper_imput_perm_an
FROM GOBGENER.dbo.IMPUT_GC_DIP_APER t1
LEFT JOIN #cr18_apertura_pacapr t2
    ON t1.C_PARTIDA = t2.C_PARTIDA AND t1.C_CAPITULO = t2.C_CAPITULO AND t1.C_PROGRAMA = t2.C_PROGRAMA
WHERE t1.AÑO = {int(Year)}
    AND t1.C_SCN NOT IN ('F22','F29','F3','F32','F41','F42','F5','F612','F7','F71','F79','K10','K11','K9')
GROUP BY t1.SUBSECTOR, t1.MONEDA, t1.AÑO, t1.C_ENTIDAD, t1.C_PARTIDA, t1.C_CAPITULO, t1.C_PROGRAMA,
    t1.C_TIPO_CUENTA, t1.C_CUENTA, t1.C_ITEM, t1.C_ASIGNACION, t1.C_ANALITICO, t1.N_ANALITICO,
    t1.C_SCN, t1.C_CAGENTE, t1.C_TRANSACCION, t1.T_TRANSACCION, t1.N_CUENTA, t1.C_ACTIVO, t1.C_CUP,
    t2.ACTIVIDAD, t2.NIVEL
"""
res = work_conn.execute(text(sql_imput_perm))
_log("CREATE #ejec_ext_dip_aper_imput_perm_an", res.rowcount)


In [ ]:
# /*UNE IMPUTACIONES PERMANENTES A SALIDA*/
work_conn.execute(text("DROP TABLE IF EXISTS #ejecucion_cnf_cgr_aper_an"))
res = work_conn.execute(text(
    "SELECT * INTO #ejecucion_cnf_cgr_aper_an FROM #ejecucion_ext_cgr_aper_prev_an "
    "UNION ALL SELECT * FROM #ejec_ext_dip_aper_imput_perm_an"
))
_log("CREATE #ejecucion_cnf_cgr_aper_an", res.rowcount)


In [ ]:
# El bloque "/*Reasignación de P3 para cuentas de subsector ... UPDATE SET C_SCN='D73' WHERE C_SCN='P3'; QUIT;"
# queda dentro de un comentario SAS que no se cierra donde parece: el primer */ que lo cierra es
# el de la nota siguiente ("Reasignación de nivel regional a central..."). Permanece inactivo en el
# SAS original y no se traduce.

# /*Reasignación de nivel regional a central, por metodología de acuerdo a características actuales de GOREs */
work_conn.execute(text("UPDATE #ejecucion_cnf_cgr_aper_an SET NIVEL='CENTRAL' WHERE NIVEL='REGIONAL'"))

# /* Reasignación FBCF realizada por municipalidades pero medida en ejecución GC */
# 'CENTRAL ' con espacio final preservado literal del SAS original (posible typo del SAS)
work_conn.execute(text(
    "UPDATE #ejecucion_cnf_cgr_aper_an SET NIVEL='MUNICIPAL' "
    "WHERE NIVEL='CENTRAL ' AND C_SCN='P51' AND C_CAGENTE='S1313'"
))

# /* Eliminación de transacciones redundantes en consolidación */
res = None
for _scn, _nivel, _cagente in [
    ("D73", "CENTRAL", "S13"), ("D93", "CENTRAL", "S13"),
    ("D73", "CENTRAL", "S1312"), ("D93", "CENTRAL", "S1312"),
    ("D73", "CENTRAL", "S131"), ("D93", "CENTRAL", "S131"),
    ("D73", "PREVISION", "S1314"), ("D93", "PREVISION", "S1314"),
]:
    res = work_conn.execute(text(
        f"DELETE FROM #ejecucion_cnf_cgr_aper_an WHERE C_SCN='{_scn}' AND NIVEL='{_nivel}' AND C_CAGENTE='{_cagente}'"
    ))
_log("DELETE #ejecucion_cnf_cgr_aper_an (redundantes consolidación, último statement)", res.rowcount)

work_conn.execute(text("DROP TABLE IF EXISTS #ejec_ext_dip_aper_imput_perm_an"))


In [ ]:
# /* EJECUCION MUNICIPAL */
# /* Salida Ejecucion Municipal predefinidos */
work_conn.execute(text("DROP TABLE IF EXISTS #ejec_cnf_mun_a_an_aper"))
sql_mun_a = f"""
SELECT t1.SUBSECTOR, t1.MONEDA, t1.AÑO,
    3 AS C_ENTIDAD, 'MUNI' AS C_PARTIDA, 'MUNI' AS C_CAPITULO, 'MUNI' AS C_PROGRAMA,
    t1.C_TIPO_CUENTA, t1.C_CUENTA,
    SUBSTRING(t1.C_ANALITICO, 1, 2) AS C_ITEM,
    SUBSTRING(t1.C_ANALITICO, 3, 3) AS C_ASIGNACION,
    t1.C_ANALITICO, t2.ANALITICO AS N_ANALITICO,
    t2.C_SCN, t2.C_CAGENTE, t2.C_TRANSACCION, t2.T_TRANSACCION, t2.N_CUENTA, t2.C_ACTIVO, t2.C_CUP,
    (CASE WHEN t1.C_T_AREA = 'GM' THEN 'AAPP'
          WHEN t1.C_T_AREA = 'ED' THEN 'EEPP'
          WHEN t1.C_T_AREA = 'SL' THEN 'SSPP' END) AS ACTIVIDAD,
    'MUNICIPAL' AS NIVEL,
    SUM(t1.DEVENG / 1000000.0) AS Dato
INTO #ejec_cnf_mun_a_an_aper
FROM GOBGENER.dbo.EJECUCION_MUN t1
LEFT JOIN GOBGENER.dbo.CR18_T_SCN_1_MUN t2
    ON t1.C_TIPO_CUENTA = t2.C_TIPO_CUENTA AND t1.C_CUENTA = t2.C_CUENTA AND t1.C_ANALITICO = t2.C_ANALITICO
WHERE t1.C_CUENTA NOT IN ('05','13','24','33')
    AND t1.AÑO = {int(Year)} AND t1.AÑO < 2022
    AND t1.C_T_AREA NOT IN ('CM')
    AND (t2.C_SCN IS NULL OR t2.C_SCN NOT IN ('F22','F29','F3','F32','F41','F42','F5','F612','F7','F71','F79','K10','K11','K9'))
GROUP BY t1.SUBSECTOR, t1.MONEDA, t1.AÑO, t1.C_TIPO_CUENTA, t1.C_CUENTA,
    SUBSTRING(t1.C_ANALITICO, 1, 2), SUBSTRING(t1.C_ANALITICO, 3, 3), t1.C_ANALITICO, t2.ANALITICO,
    t2.C_SCN, t2.C_CAGENTE, t2.C_TRANSACCION, t2.T_TRANSACCION, t2.N_CUENTA, t2.C_ACTIVO, t2.C_CUP,
    (CASE WHEN t1.C_T_AREA = 'GM' THEN 'AAPP'
          WHEN t1.C_T_AREA = 'ED' THEN 'EEPP'
          WHEN t1.C_T_AREA = 'SL' THEN 'SSPP' END)
"""
res = work_conn.execute(text(sql_mun_a))
_log("CREATE #ejec_cnf_mun_a_an_aper", res.rowcount)


In [ ]:
# /* Salida Ejecucion Municipal variables */
work_conn.execute(text("DROP TABLE IF EXISTS #ejec_cnf_mun_b_an_aper"))
sql_mun_b = f"""
SELECT t1.SUBSECTOR, t1.MONEDA, t1.AÑO,
    3 AS C_ENTIDAD, 'MUNI' AS C_PARTIDA, 'MUNI' AS C_CAPITULO, 'MUNI' AS C_PROGRAMA,
    t1.C_TIPO_CUENTA, t1.C_CUENTA,
    SUBSTRING(t1.C_ANALITICO, 1, 2) AS C_ITEM,
    SUBSTRING(t1.C_ANALITICO, 3, 3) AS C_ASIGNACION,
    t1.C_ANALITICO, t1.ANALITICO AS N_ANALITICO,
    t2.C_SCN, t2.C_CAGENTE, t2.C_TRANSACCION, t2.T_TRANSACCION, t2.N_CUENTA, t2.C_ACTIVO, t2.C_CUP,
    (CASE WHEN t1.C_T_AREA = 'GM' THEN 'AAPP'
          WHEN t1.C_T_AREA = 'ED' THEN 'EEPP'
          WHEN t1.C_T_AREA = 'SL' THEN 'SSPP' END) AS ACTIVIDAD,
    'MUNICIPAL' AS NIVEL,
    SUM(t1.DEVENG / 1000000.0) AS Dato
INTO #ejec_cnf_mun_b_an_aper
FROM GOBGENER.dbo.EJECUCION_MUN t1
LEFT JOIN GOBGENER.dbo.CR18_T_SCN_2_MUN t2
    ON t1.C_CUENTA = t2.C_CUENTA AND t1.C_ANALITICO = t2.C_ANALITICO AND t1.C_T_AREA = t2.C_T_AREA
WHERE t1.C_CUENTA IN ('05','13','24','33')
    AND t1.AÑO = {int(Year)} AND t1.AÑO < 2022
    AND t1.C_T_AREA NOT IN ('CM')
    AND (t2.C_SCN IS NULL OR t2.C_SCN NOT IN ('F22','F29','F3','F32','F41','F42','F5','F612','F7','F71','F79','K10','K11','K9'))
GROUP BY t1.SUBSECTOR, t1.MONEDA, t1.AÑO, t1.C_TIPO_CUENTA, t1.C_CUENTA,
    SUBSTRING(t1.C_ANALITICO, 1, 2), SUBSTRING(t1.C_ANALITICO, 3, 3), t1.C_ANALITICO, t1.ANALITICO,
    t2.C_SCN, t2.C_CAGENTE, t2.C_TRANSACCION, t2.T_TRANSACCION, t2.N_CUENTA, t2.C_ACTIVO, t2.C_CUP,
    (CASE WHEN t1.C_T_AREA = 'GM' THEN 'AAPP'
          WHEN t1.C_T_AREA = 'ED' THEN 'EEPP'
          WHEN t1.C_T_AREA = 'SL' THEN 'SSPP' END)
"""
res = work_conn.execute(text(sql_mun_b))
_log("CREATE #ejec_cnf_mun_b_an_aper", res.rowcount)


In [ ]:
# /* EJECUCUIÓN NUEVA MUNICIPAL DESDE 2022 EN ADELANTE*/
work_conn.execute(text("DROP TABLE IF EXISTS #ejec_cnf_mun_a_an_nuevo"))
sql_mun_a_nuevo = f"""
SELECT SUBSTRING(t1.SUBSECTOR, 2, 1) AS SUBSECTOR,
    SUBSTRING(t1.MONEDA, 3, 1) AS MONEDA,
    t1.AÑO,
    3 AS C_ENTIDAD, 'MUNI' AS C_PARTIDA, 'MUNI' AS C_CAPITULO, 'MUNI' AS C_PROGRAMA,
    (CASE WHEN t1.C_T_CUENTA = 1 THEN 'Y' WHEN t1.C_T_CUENTA = 2 THEN 'G' ELSE '' END) AS C_TIPO_CUENTA,
    t1.C_CUENTA, t1.C_ITEM, t1.C_ASIGNACION, t1.C_ANALITICO,
    t2.N_ASIGNACION AS N_ANALITICO,
    t2.C_SCN, t2.C_CAGENTE, t2.C_TRANSACCION, t2.T_TRANSACCION, t2.N_CUENTA, t2.C_ACTIVO, t2.C_CUP,
    (CASE WHEN t1.C_SERVICIO = 1 THEN 'AAPP'
          WHEN t1.C_SERVICIO = 2 THEN 'EEPP'
          WHEN t1.C_SERVICIO = 3 THEN 'SSPP' END) AS ACTIVIDAD,
    'MUNICIPAL' AS NIVEL,
    SUM(t1.DEVENGO / 1000000.0) AS Dato
INTO #ejec_cnf_mun_a_an_nuevo
FROM GOBGENER.dbo.EJECUCION_MUN_P t1
LEFT JOIN GOBGENER.dbo.CR18_T_SCN_1_MUN_P t2
    ON t1.C_CUENTA = t2.C_CUENTA AND t1.C_ANALITICO_2 = t2.C_ANALITICO_2
WHERE t1.C_CUENTA NOT IN ('05','13','24','33')
    AND (CASE WHEN t1.C_T_CUENTA = 1 THEN 'Y' WHEN t1.C_T_CUENTA = 2 THEN 'G' ELSE '' END) = t2.C_TIPO_CUENTA
    AND t1.AÑO = {int(Year)} AND t1.AÑO >= 2022
    AND t1.C_SERVICIO NOT IN (4)
    AND (t2.C_SCN IS NULL OR t2.C_SCN NOT IN ('F22','F29','F3','F32','F41','F42','F5','F612','F7','F71','F79','K10','K11','K9'))
GROUP BY SUBSTRING(t1.SUBSECTOR, 2, 1), SUBSTRING(t1.MONEDA, 3, 1), t1.AÑO, t1.C_T_CUENTA, t1.C_CUENTA,
    t1.C_ITEM, t1.C_ASIGNACION, t1.C_ANALITICO, t2.N_ASIGNACION, t2.C_SCN, t2.C_CAGENTE, t2.C_TRANSACCION,
    t2.T_TRANSACCION, t2.N_CUENTA, t2.C_ACTIVO, t2.C_CUP,
    (CASE WHEN t1.C_SERVICIO = 1 THEN 'AAPP'
          WHEN t1.C_SERVICIO = 2 THEN 'EEPP'
          WHEN t1.C_SERVICIO = 3 THEN 'SSPP' END)
"""
res = work_conn.execute(text(sql_mun_a_nuevo))
_log("CREATE #ejec_cnf_mun_a_an_nuevo", res.rowcount)


In [ ]:
work_conn.execute(text("DROP TABLE IF EXISTS #ejec_cnf_mun_b_an_nuevo"))
sql_mun_b_nuevo = f"""
SELECT SUBSTRING(t1.SUBSECTOR, 2, 1) AS SUBSECTOR,
    SUBSTRING(t1.MONEDA, 3, 1) AS MONEDA,
    t1.AÑO,
    3 AS C_ENTIDAD, 'MUNI' AS C_PARTIDA, 'MUNI' AS C_CAPITULO, 'MUNI' AS C_PROGRAMA,
    (CASE WHEN t1.C_T_CUENTA = 1 THEN 'Y' WHEN t1.C_T_CUENTA = 2 THEN 'G' ELSE '' END) AS C_TIPO_CUENTA,
    t1.C_CUENTA, t1.C_ITEM, t1.C_ASIGNACION, t1.C_ANALITICO,
    t2.N_ASIGNACION AS N_ANALITICO,
    t2.C_SCN, t2.C_CAGENTE, t2.C_TRANSACCION, t2.T_TRANSACCION, t2.N_CUENTA, t2.C_ACTIVO, t2.C_CUP,
    (CASE WHEN t1.C_SERVICIO = 1 THEN 'AAPP'
          WHEN t1.C_SERVICIO = 2 THEN 'EEPP'
          WHEN t1.C_SERVICIO = 3 THEN 'SSPP' END) AS ACTIVIDAD,
    'MUNICIPAL' AS NIVEL,
    SUM(t1.DEVENGO / 1000000.0) AS Dato
INTO #ejec_cnf_mun_b_an_nuevo
FROM GOBGENER.dbo.EJECUCION_MUN_P t1
LEFT JOIN GOBGENER.dbo.CR18_T_SCN_2_MUN_P t2
    ON t1.C_CUENTA = t2.C_CUENTA AND t1.C_ANALITICO = t2.C_ANALITICO AND t1.C_SERVICIO = t2.C_SERVICIO
WHERE t1.C_CUENTA IN ('05','13','24','33')
    AND t1.AÑO = {int(Year)} AND t1.AÑO >= 2022
    AND t1.C_SERVICIO NOT IN (4)
    AND (t2.C_SCN IS NULL OR t2.C_SCN NOT IN ('F22','F29','F3','F32','F41','F42','F5','F612','F7','F71','F79','K10','K11','K9'))
GROUP BY SUBSTRING(t1.SUBSECTOR, 2, 1), SUBSTRING(t1.MONEDA, 3, 1), t1.AÑO, t1.C_T_CUENTA, t1.C_CUENTA,
    t1.C_ITEM, t1.C_ASIGNACION, t1.C_ANALITICO, t2.N_ASIGNACION, t2.C_SCN, t2.C_CAGENTE, t2.C_TRANSACCION,
    t2.T_TRANSACCION, t2.N_CUENTA, t2.C_ACTIVO, t2.C_CUP,
    (CASE WHEN t1.C_SERVICIO = 1 THEN 'AAPP'
          WHEN t1.C_SERVICIO = 2 THEN 'EEPP'
          WHEN t1.C_SERVICIO = 3 THEN 'SSPP' END)
"""
res = work_conn.execute(text(sql_mun_b_nuevo))
_log("CREATE #ejec_cnf_mun_b_an_nuevo", res.rowcount)


In [ ]:
# /* Union de ambos tipos de ejecucion municipal */
work_conn.execute(text("DROP TABLE IF EXISTS #ejecucion_cnf_mun_an_aper"))
work_conn.execute(text(
    "SELECT * INTO #ejecucion_cnf_mun_an_aper FROM #ejec_cnf_mun_a_an_aper "
    "UNION ALL SELECT * FROM #ejec_cnf_mun_b_an_aper "
    "UNION ALL SELECT * FROM #ejec_cnf_mun_a_an_nuevo "
    "UNION ALL SELECT * FROM #ejec_cnf_mun_b_an_nuevo"
))

# /*Reasignación de P3 para cuentas de subsector */
work_conn.execute(text("UPDATE #ejecucion_cnf_mun_an_aper SET C_SCN='D73' WHERE C_SCN='P3'"))

# /* Eliminación de transacciones redundantes en consolidación */
work_conn.execute(text("DELETE FROM #ejecucion_cnf_mun_an_aper WHERE C_SCN='D73' AND C_CAGENTE='S1313'"))
work_conn.execute(text("DELETE FROM #ejecucion_cnf_mun_an_aper WHERE C_SCN='D93' AND C_CAGENTE='S1313'"))

# /* Eliminacion de D63 por ya estar estimadas desde el origen (EDO), Gobierno Central */
res = work_conn.execute(text("DELETE FROM #ejecucion_cnf_mun_an_aper WHERE C_SCN='D63'"))
_log("CREATE/limpieza #ejecucion_cnf_mun_an_aper", res.rowcount)


In [ ]:
# /* Union de ambos tipos de ejecucion, Gob Central + Municipal */
work_conn.execute(text("DROP TABLE IF EXISTS #ejecucion_cnf_an_aper"))
res = work_conn.execute(text(
    "SELECT * INTO #ejecucion_cnf_an_aper FROM #ejecucion_cnf_cgr_aper_an "
    "UNION ALL SELECT * FROM #ejecucion_cnf_mun_an_aper"
))
_log("CREATE #ejecucion_cnf_an_aper", res.rowcount)

# /* Se eliminan dividendos de CORFO porque se tomaran los del sectorialista de dividendos*/
work_conn.execute(text("DELETE FROM #ejecucion_cnf_an_aper WHERE C_SCN='D421' AND ACTIVIDAD='CORFO'"))
# /* Se eliminan subvenciones porque se toman las calculadas aparte */
work_conn.execute(text("DELETE FROM #ejecucion_cnf_an_aper WHERE C_SCN='D39'"))
# /* Se elimina D41 dolar, porque se toma dado de contragente S2 de Balanza de Pagos */
work_conn.execute(text("DELETE FROM #ejecucion_cnf_an_aper WHERE C_SCN='D41' AND C_TIPO_CUENTA='Y' AND MONEDA='D'"))
# /* Se eliminan datos de producción dado que se ocupan los publicados */
res = work_conn.execute(text("DELETE FROM #ejecucion_cnf_an_aper WHERE C_SCN IN ('D11','D12','P2')"))
_log("DELETE #ejecucion_cnf_an_aper (exclusiones CORFO/D39/D41/producción)", res.rowcount)

# /* Estandarizacion de salida para calculo de cuenta */
for _nuevo, _viejo in [("D11", "D12"), ("D61", "D6111"), ("D75", "D74")]:
    work_conn.execute(text(f"UPDATE #ejecucion_cnf_an_aper SET C_SCN='{_nuevo}' WHERE C_SCN='{_viejo}'"))
work_conn.execute(text("UPDATE #ejecucion_cnf_an_aper SET C_SCN='D99' WHERE C_SCN IN ('D91','D92','D991')"))
work_conn.execute(text("UPDATE #ejecucion_cnf_an_aper SET C_SCN='K2' WHERE C_SCN='P513'"))
work_conn.execute(text("UPDATE #ejecucion_cnf_an_aper SET C_SCN='P51' WHERE C_SCN='P511'"))
res = work_conn.execute(text("UPDATE #ejecucion_cnf_an_aper SET C_SCN='D42' WHERE C_SCN='D421'"))
_log("UPDATE #ejecucion_cnf_an_aper (estandarización C_SCN)", res.rowcount)

# limpieza de tablas intermedias municipales (replica exacta del DROP TABLE del SAS original,
# incluye nombres mal escritos que el SAS nunca creó: DROP TABLE IF EXISTS no falla por eso)
for _t in ["#ejec_cnf_mun_a_an_aper_prelim", "#ejec_cnf_mun_a_an_aper", "#ejec_cnf_mun_b_an_aper",
           "#ejec_cnf_mun_b_an_aper_prelim", "#ejecucion_cnf_mun_an_aper", "#ejecucion_cnf_mun_an_aper_nuevo",
           "#ejec_cnf_mun_a_an_aper_nuevo", "#ejec_cnf_mun_b_an_aper_nuevo"]:
    work_conn.execute(text(f"DROP TABLE IF EXISTS {_t}"))


In [ ]:
# /*SALIDA AGREGADA CNF ANUAL*/
work_conn.execute(text("DROP TABLE IF EXISTS #ejecucion_cnf_an"))
sql_cnf_an = """
SELECT t1.SUBSECTOR, t1.MONEDA, t1.AÑO, t1.C_TIPO_CUENTA, t1.C_CUENTA, t1.C_ITEM, t1.C_SCN, t1.C_CAGENTE,
    t1.C_TRANSACCION, t1.T_TRANSACCION, t1.N_CUENTA, t1.C_ACTIVO, t1.C_CUP, t1.ACTIVIDAD, t1.NIVEL,
    SUM(t1.Dato) AS Dato
INTO #ejecucion_cnf_an
FROM #ejecucion_cnf_an_aper t1
GROUP BY t1.SUBSECTOR, t1.MONEDA, t1.AÑO, t1.C_TIPO_CUENTA, t1.C_CUENTA, t1.C_ITEM, t1.C_SCN, t1.C_CAGENTE,
    t1.C_TRANSACCION, t1.T_TRANSACCION, t1.N_CUENTA, t1.C_ACTIVO, t1.C_CUP, t1.ACTIVIDAD, t1.NIVEL
"""
res = work_conn.execute(text(sql_cnf_an))
_log("CREATE #ejecucion_cnf_an", res.rowcount)


## Salida_COFOG

Arma la base analítica de ejecución presupuestaria con montos promedio por entidad, cuenta y trimestre para el reporte COFOG anual, excluyendo cuentas y entidades de control

*confianza: medium · verificador: approve · SAS: DATA step SET (copia) + PROC SQL CREATE TABLE con múltiples LEFT JOIN, CASE WHEN y GROUP BY agregado*

In [ ]:
# ========= Salida_COFOG =========
# Salida para ANALISIS DESAGREGADO de indicadores GC para COFOG anual
# DATA step: copia directa de EJEC_CGR_Y a EJECUCION (ambos WORK materializados como tablas de sesión)
work_conn.execute(text("DROP TABLE IF EXISTS #ejecucion"))
sql_ejecucion = "SELECT * INTO #ejecucion FROM #ejec_cgr_y"
res = work_conn.execute(text(sql_ejecucion))
_log("SELECT INTO #ejecucion", res.rowcount)


In [ ]:
# /*Analiticos predefinidos */
# Cruza EJECUCION con los catálogos de transacción/cuenta (CR18_T_SCN_1), el promedio
# mensual (PROM_MES) y la apertura/COFOG, excluyendo cuentas y entidades de control
work_conn.execute(text("DROP TABLE IF EXISTS #ejecucion_a_prelim_aper_an"))
sql_ejecucion_a_prelim_aper_an = """
SELECT
    t1.SUBSECTOR,
    t1.MONEDA,
    t1.[AÑO],
    (CASE
        WHEN t1.MES IN (1,2,3) THEN 1
        WHEN t1.MES IN (4,5,6) THEN 2
        WHEN t1.MES IN (7,8,9) THEN 3
        ELSE 4
    END) AS [TRIM],
    t1.MES,
    t1.C_ENTIDAD,
    t1.C_PARTIDA,
    t1.C_CAPITULO,
    t1.C_PROGRAMA,
    t1.C_TIPO_CUENTA,
    t1.C_CUENTA,
    t1.C_ITEM,
    t1.C_ASIGNACION,
    t1.C_ANALITICO,
    t1.N_ANALITICO,
    t2.C_SCN,
    t2.C_CAGENTE,
    t2.C_TRANSACCION,
    t2.T_TRANSACCION,
    t2.N_CUENTA,
    t2.C_ACTIVO,
    t2.C_CUP,
    t4.ACTIVIDAD,
    t4.NIVEL,
    t5.C_COFOG_1,
    t5.N_COFOG_1,
    SUM(t1.DEVENG * t3.DATO / 1000000) AS DEVENGO
INTO #ejecucion_a_prelim_aper_an
FROM GOBGENER.dbo.EJECUCION t1
LEFT JOIN GOBGENER.dbo.CR18_T_SCN_1 t2
    ON t1.C_TIPO_CUENTA = t2.T_CUENTA
   AND t1.C_CUENTA = t2.C_CUENTA
   AND t1.C_ANALITICO = t2.C_ANALITICO
LEFT JOIN #prom_mes t3
    ON t1.MONEDA = t3.MONEDA
   AND t1.MES = t3.MES
   AND t1.[AÑO] = t3.[AÑO]
LEFT JOIN GOBGENER.dbo.CR18_APERTURA t4
    ON t1.C_ENTIDAD = t4.ENTIDAD
LEFT JOIN GOBGENER.dbo.COFOG t5
    ON t1.C_ENTIDAD = t5.ENTIDAD
   AND t1.C_PARTIDA = t5.C_PARTIDA
   AND t1.C_CAPITULO = t5.C_CAPITULO
   AND t1.C_PROGRAMA = t5.C_PROGRAMA
WHERE t1.C_CUENTA NOT IN ('05','13','24','33')
  AND t1.C_ENTIDAD NOT IN (5601,10201)
GROUP BY
    t1.SUBSECTOR,
    t1.MONEDA,
    t1.[AÑO],
    (CASE
        WHEN t1.MES IN (1,2,3) THEN 1
        WHEN t1.MES IN (4,5,6) THEN 2
        WHEN t1.MES IN (7,8,9) THEN 3
        ELSE 4
    END),
    t1.MES,
    t1.C_ENTIDAD,
    t1.C_PARTIDA,
    t1.C_CAPITULO,
    t1.C_PROGRAMA,
    t1.C_TIPO_CUENTA,
    t1.C_CUENTA,
    t1.C_ITEM,
    t1.C_ASIGNACION,
    t1.C_ANALITICO,
    t1.N_ANALITICO,
    t2.C_SCN,
    t2.C_CAGENTE,
    t2.C_TRANSACCION,
    t2.T_TRANSACCION,
    t2.N_CUENTA,
    t2.C_ACTIVO,
    t2.C_CUP,
    t4.ACTIVIDAD,
    t4.NIVEL,
    t5.C_COFOG_1,
    t5.N_COFOG_1
"""
res = work_conn.execute(text(sql_ejecucion_a_prelim_aper_an))
_log("SELECT INTO #ejecucion_a_prelim_aper_an", res.rowcount)


## Salida_COFOG_Muni

Ejecución municipal clasificada según el marco SCN por tipo de cuenta (predefinidos vs. variables), consolidada a nivel municipal con su etiqueta de nivel, para el año vigente y su versión preliminar

*confianza: medium · verificador: approve · SAS: PROC SQL: dos CREATE TABLE con LEFT JOIN + GROUP BY (predefinidos/variables), UNION ALL y UPDATE, repetido para la versión del año y su preliminar (P)*

In [ ]:
# ========= Salida_COFOG_Muni =========
# /* Salida Ejecucion Municipal predefinidos */
work_conn.execute(text("DROP TABLE IF EXISTS #ejec_cnf_mun_a_an_aper"))
sql_ejec_cnf_mun_a_an_aper = f"""
SELECT t1.SUBSECTOR,
       t1.MONEDA,
       t1.AÑO, t1.C_T_AREA,
       t1.C_TIPO_CUENTA,
       t2.C_SCN,
       t2.C_CAGENTE,
       t2.C_TRANSACCION,
       t2.T_TRANSACCION,
       t2.N_CUENTA,
       t2.C_ACTIVO,
       t2.C_CUP,
       CASE WHEN t1.C_T_AREA = 'GM' THEN 'AAPP'
            WHEN t1.C_T_AREA = 'ED' THEN 'EEPP'
            WHEN t1.C_T_AREA = 'SL' THEN 'SSPP' END AS ACTIVIDAD,
       t2.T_TRANSACCION AS NIVEL,
       SUM(t1.DEVENG / 1000000.0) AS Dato
INTO #ejec_cnf_mun_a_an_aper
FROM GOBGENER.dbo.EJECUCION_MUN t1
LEFT JOIN GOBGENER.dbo.CR18_T_SCN_1_MUN t2
  ON t1.C_TIPO_CUENTA = t2.C_TIPO_CUENTA
  AND t1.C_CUENTA = t2.C_CUENTA
  AND t1.C_ANALITICO = t2.C_ANALITICO
WHERE t1.C_CUENTA NOT IN ('05','13','24','33')
  AND t1.AÑO = {int(Year)}
  AND (t2.C_SCN IS NULL OR t2.C_SCN NOT IN ('F22','F29','F3','F32','F41','F42','F5','F612','F7','F71','F79','K10','K11','K9'))
GROUP BY t1.SUBSECTOR, t1.MONEDA, t1.AÑO, t1.C_T_AREA, t1.C_TIPO_CUENTA,
         t2.C_SCN, t2.C_CAGENTE, t2.C_TRANSACCION, t2.T_TRANSACCION,
         t2.N_CUENTA, t2.C_ACTIVO, t2.C_CUP,
         CASE WHEN t1.C_T_AREA = 'GM' THEN 'AAPP'
              WHEN t1.C_T_AREA = 'ED' THEN 'EEPP'
              WHEN t1.C_T_AREA = 'SL' THEN 'SSPP' END
"""
work_conn.execute(text(sql_ejec_cnf_mun_a_an_aper))


In [ ]:
# /* Salida Ejecucion Municipal variables */
work_conn.execute(text("DROP TABLE IF EXISTS #ejec_cnf_mun_b_an_aper"))
sql_ejec_cnf_mun_b_an_aper = f"""
SELECT t1.SUBSECTOR,
       t1.MONEDA,
       t1.AÑO, t1.C_T_AREA,
       t1.C_TIPO_CUENTA,
       t2.C_SCN,
       t2.C_CAGENTE,
       t2.C_TRANSACCION,
       t2.T_TRANSACCION,
       t2.N_CUENTA,
       t2.C_ACTIVO,
       t2.C_CUP,
       CASE WHEN t1.C_T_AREA = 'GM' THEN 'AAPP'
            WHEN t1.C_T_AREA = 'ED' THEN 'EEPP'
            WHEN t1.C_T_AREA = 'SL' THEN 'SSPP' END AS ACTIVIDAD,
       t2.T_TRANSACCION AS NIVEL,
       SUM(t1.DEVENG / 1000000.0) AS Dato
INTO #ejec_cnf_mun_b_an_aper
FROM GOBGENER.dbo.EJECUCION_MUN t1
LEFT JOIN GOBGENER.dbo.CR18_T_SCN_2_MUN t2
  ON t1.C_CUENTA = t2.C_CUENTA
  AND t1.C_ANALITICO = t2.C_ANALITICO
  AND t1.C_T_AREA = t2.C_T_AREA
WHERE t1.C_CUENTA IN ('05','13','24','33')
  AND t1.AÑO = {int(Year)}
  AND (t2.C_SCN IS NULL OR t2.C_SCN NOT IN ('F22','F29','F3','F32','F41','F42','F5','F612','F7','F71','F79','K10','K11','K9'))
GROUP BY t1.SUBSECTOR, t1.MONEDA, t1.AÑO, t1.C_T_AREA, t1.C_TIPO_CUENTA,
         t2.C_SCN, t2.C_CAGENTE, t2.C_TRANSACCION, t2.T_TRANSACCION,
         t2.N_CUENTA, t2.C_ACTIVO, t2.C_CUP,
         CASE WHEN t1.C_T_AREA = 'GM' THEN 'AAPP'
              WHEN t1.C_T_AREA = 'ED' THEN 'EEPP'
              WHEN t1.C_T_AREA = 'SL' THEN 'SSPP' END
"""
work_conn.execute(text(sql_ejec_cnf_mun_b_an_aper))


In [ ]:
# /* Union de ambos tipos de ejecucion municipal */
work_conn.execute(text("DROP TABLE IF EXISTS #ejec_mun_niveles_dist_cfg"))
sql_union_mun = """
SELECT * INTO #ejec_mun_niveles_dist_cfg
FROM #ejec_cnf_mun_a_an_aper
UNION ALL
SELECT * FROM #ejec_cnf_mun_b_an_aper
"""
work_conn.execute(text(sql_union_mun))


In [ ]:
res = work_conn.execute(text("UPDATE #ejec_mun_niveles_dist_cfg SET NIVEL = 'MUNICIPAL'"))
_log("UPDATE #ejec_mun_niveles_dist_cfg", res.rowcount)


In [ ]:
# /* Salida Ejecucion Municipal predefinidos */
work_conn.execute(text("DROP TABLE IF EXISTS #ejec_cnf_mun_a_an_aper_p"))
sql_ejec_cnf_mun_a_an_aper_p = f"""
SELECT t1.SUBSECTOR,
       t1.MONEDA,
       t1.AÑO, t1.C_SERVICIO,
       t1.C_T_CUENTA,
       t2.C_SCN,
       t2.C_CAGENTE,
       t2.C_TRANSACCION,
       t2.T_TRANSACCION,
       t2.N_CUENTA,
       t2.C_ACTIVO,
       t2.C_CUP,
       CASE WHEN t1.C_SERVICIO = 1 THEN 'AAPP'
            WHEN t1.C_SERVICIO = 2 THEN 'EEPP'
            WHEN t1.C_SERVICIO = 3 THEN 'SSPP' END AS ACTIVIDAD,
       t2.T_TRANSACCION AS NIVEL,
       SUM(t1.DEVENGO / 1000000.0) AS Dato
INTO #ejec_cnf_mun_a_an_aper_p
FROM GOBGENER.dbo.EJECUCION_MUN_P t1
LEFT JOIN GOBGENER.dbo.CR18_T_SCN_1_MUN_P t2
  ON t1.C_T_CUENTA = t2.C_T_CUENTA
  AND t1.C_CUENTA = t2.C_CUENTA
  AND t1.C_ANALITICO = t2.C_ANALITICO
WHERE t1.C_CUENTA NOT IN ('05','13','24','33')
  AND t1.AÑO = {int(Year)}
  AND (t2.C_SCN IS NULL OR t2.C_SCN NOT IN ('F22','F29','F3','F32','F41','F42','F5','F612','F7','F71','F79','K10','K11','K9'))
GROUP BY t1.SUBSECTOR, t1.MONEDA, t1.AÑO, t1.C_SERVICIO, t1.C_T_CUENTA,
         t2.C_SCN, t2.C_CAGENTE, t2.C_TRANSACCION, t2.T_TRANSACCION,
         t2.N_CUENTA, t2.C_ACTIVO, t2.C_CUP,
         CASE WHEN t1.C_SERVICIO = 1 THEN 'AAPP'
              WHEN t1.C_SERVICIO = 2 THEN 'EEPP'
              WHEN t1.C_SERVICIO = 3 THEN 'SSPP' END
"""
work_conn.execute(text(sql_ejec_cnf_mun_a_an_aper_p))


In [ ]:
# /* Salida Ejecucion Municipal variables */
work_conn.execute(text("DROP TABLE IF EXISTS #ejec_cnf_mun_b_an_aper_p"))
sql_ejec_cnf_mun_b_an_aper_p = f"""
SELECT t1.SUBSECTOR,
       t1.MONEDA,
       t1.AÑO, t1.C_SERVICIO,
       t1.C_T_CUENTA,
       t2.C_SCN,
       t2.C_CAGENTE,
       t2.C_TRANSACCION,
       t2.T_TRANSACCION,
       t2.N_CUENTA,
       t2.C_ACTIVO,
       t2.C_CUP,
       CASE WHEN t1.C_SERVICIO = 1 THEN 'AAPP'
            WHEN t1.C_SERVICIO = 2 THEN 'EEPP'
            WHEN t1.C_SERVICIO = 3 THEN 'SSPP' END AS ACTIVIDAD,
       t2.T_TRANSACCION AS NIVEL,
       SUM(t1.DEVENGO / 1000000.0) AS Dato
INTO #ejec_cnf_mun_b_an_aper_p
FROM GOBGENER.dbo.EJECUCION_MUN_P t1
LEFT JOIN GOBGENER.dbo.CR18_T_SCN_2_MUN_P t2
  ON t1.C_CUENTA = t2.C_CUENTA
  AND t1.C_ANALITICO = t2.C_ANALITICO
  AND t1.C_SERVICIO = t2.C_SERVICIO
WHERE t1.C_CUENTA IN ('05','13','24','33')
  AND t1.AÑO = {int(Year)}
  AND (t2.C_SCN IS NULL OR t2.C_SCN NOT IN ('F22','F29','F3','F32','F41','F42','F5','F612','F7','F71','F79','K10','K11','K9'))
GROUP BY t1.SUBSECTOR, t1.MONEDA, t1.AÑO, t1.C_SERVICIO, t1.C_T_CUENTA,
         t2.C_SCN, t2.C_CAGENTE, t2.C_TRANSACCION, t2.T_TRANSACCION,
         t2.N_CUENTA, t2.C_ACTIVO, t2.C_CUP,
         CASE WHEN t1.C_SERVICIO = 1 THEN 'AAPP'
              WHEN t1.C_SERVICIO = 2 THEN 'EEPP'
              WHEN t1.C_SERVICIO = 3 THEN 'SSPP' END
"""
work_conn.execute(text(sql_ejec_cnf_mun_b_an_aper_p))


In [ ]:
# /* Union de ambos tipos de ejecucion municipal */
work_conn.execute(text("DROP TABLE IF EXISTS #ejec_mun_niveles_dist_cfg_p"))
sql_union_mun_p = """
SELECT * INTO #ejec_mun_niveles_dist_cfg_p
FROM #ejec_cnf_mun_a_an_aper_p
UNION ALL
SELECT * FROM #ejec_cnf_mun_b_an_aper_p
"""
work_conn.execute(text(sql_union_mun_p))


In [ ]:
res_p = work_conn.execute(text("UPDATE #ejec_mun_niveles_dist_cfg_p SET NIVEL = 'MUNICIPAL'"))
_log("UPDATE #ejec_mun_niveles_dist_cfg_p", res_p.rowcount)


## APER_COFOG_anual_mes

Sin código SAS no hay regla de negocio que documentar en este nodo

*confianza: low · verificador: approve*

In [ ]:
# ========= APER_COFOG_anual_mes =========
raise NotImplementedError("Nodo 'APER_COFOG_anual_mes' no trae código SAS traducible: el bloque fuente llegó vacío. No hay construct que mapear a pandas/SQL ni insumo declarado en input_datasets; se desconoce si es un placeholder intencional o un nodo deshabilitado del flujo original. Completar con el código SAS real antes de habilitar este nodo en el notebook.")

## CFG_niveles_dist_FNDR_P5

Distribución regional del gasto FNDR (P51/P513) en la función COFOG 11, combinando el devengo ponderado del año con los montos imputados permanentes del mismo periodo

*confianza: medium · verificador: approve · SAS: PROC SQL: agregación con SUM+GROUP BY, GROUP BY sin función de agregación (equivalente a ORDER BY), UNION ALL y DROP de tablas intermedias*

In [ ]:
# ========= CFG_niveles_dist_FNDR_P5 =========
# Distribución regional del FNDR (P51/P513) en función COFOG 11, ponderada por PROM_MES
work_conn.execute(text("DROP TABLE IF EXISTS #nivel_fndr_a_distrib"))
sql_nivel_fndr_a_distrib = f"""
SELECT
    t1.SUBSECTOR,
    t1.MONEDA,
    t1.AÑO,
    t1.C_TIPO_CUENTA,
    t2.C_SCN,
    t2.C_CAGENTE,
    t2.C_TRANSACCION,
    t2.T_TRANSACCION,
    t2.N_CUENTA,
    t4.ACTIVIDAD,
    t4.NIVEL,
    t5.C_COFOG_1,
    t5.N_COFOG_1,
    SUM(t1.DEVENG * t3.DATO / 1000000) AS DEVENGO
INTO #nivel_fndr_a_distrib
FROM GOBGENER.dbo.EJECUCION t1
LEFT JOIN GOBGENER.dbo.CR18_T_SCN_1 t2
    ON t1.C_TIPO_CUENTA = t2.T_CUENTA
    AND t1.C_CUENTA = t2.C_CUENTA
    AND t1.C_ANALITICO = t2.C_ANALITICO
LEFT JOIN #prom_mes t3
    ON t1.MONEDA = t3.MONEDA
    AND t1.MES = t3.MES
    AND t1.AÑO = t3.AÑO
LEFT JOIN GOBGENER.dbo.CR18_APERTURA t4
    ON t1.C_ENTIDAD = t4.ENTIDAD
LEFT JOIN GOBGENER.dbo.COFOG t5
    ON t1.C_ENTIDAD = t5.ENTIDAD
    AND t1.C_PARTIDA = t5.C_PARTIDA
    AND t1.C_CAPITULO = t5.C_CAPITULO
    AND t1.C_PROGRAMA = t5.C_PROGRAMA
WHERE t1.C_CUENTA NOT IN ('05','13','24','33')
    AND t4.NIVEL = 'REGIONAL'
    AND t2.C_SCN IN ('P51','P513')
    AND t5.C_COFOG_1 = '11.'
    AND t1.C_ENTIDAD NOT IN (5601,10201)
    AND t1.AÑO = {int(Year)}
GROUP BY
    t1.SUBSECTOR,
    t1.MONEDA,
    t1.AÑO,
    t1.C_TIPO_CUENTA,
    t2.C_SCN,
    t2.C_CAGENTE,
    t2.C_TRANSACCION,
    t2.T_TRANSACCION,
    t2.N_CUENTA,
    t4.ACTIVIDAD,
    t4.NIVEL,
    t5.C_COFOG_1,
    t5.N_COFOG_1
"""
res = work_conn.execute(text(sql_nivel_fndr_a_distrib))
_log("CREATE #nivel_fndr_a_distrib", res.rowcount)


In [ ]:
work_conn.execute(text("DROP TABLE IF EXISTS #cofog_pacapr"))
sql_cofog_pacapr = """
SELECT DISTINCT
    t1.C_COFOG,
    t1.N_COFOG,
    t1.C_COFOG_1,
    t1.N_COFOG_1,
    t2.C_PARTIDA,
    t2.C_CAPITULO,
    t2.C_PROGRAMA
INTO #cofog_pacapr
FROM GOBGENER.dbo.COFOG t1
LEFT JOIN GOBGENER.dbo.ENTIDADES t2 ON t1.ENTIDAD = t2.C_ENTIDAD
"""
res = work_conn.execute(text(sql_cofog_pacapr))
_log("CREATE #cofog_pacapr", res.rowcount)


In [ ]:
# El GROUP BY de esta consulta no tiene ninguna función de agregación en el
# SELECT (t1.Dato AS DEVENGO es columna directa): PROC SQL lo trata como un
# ORDER BY y devuelve TODAS las filas que pasan el WHERE, sin deduplicar por
# esas columnas (a diferencia de la consulta anterior, que sí agrega con SUM)
work_conn.execute(text("DROP TABLE IF EXISTS #nivel_fndr_a_dist_imput_perm_an"))
sql_nivel_fndr_a_dist_imput_perm_an = f"""
SELECT
    t1.SUBSECTOR,
    t1.MONEDA,
    t1.AÑO,
    t1.C_TIPO_CUENTA,
    t1.C_SCN,
    t1.C_CAGENTE,
    t1.C_TRANSACCION,
    t1.T_TRANSACCION,
    t1.N_CUENTA,
    t2.ACTIVIDAD,
    t2.NIVEL,
    t5.C_COFOG_1,
    t5.N_COFOG_1,
    t1.Dato AS DEVENGO
INTO #nivel_fndr_a_dist_imput_perm_an
FROM GOBGENER.dbo.IMPUT_GC_DIP_APER t1
LEFT JOIN #cofog_pacapr t5
    ON t1.C_PARTIDA = t5.C_PARTIDA
    AND t1.C_CAPITULO = t5.C_CAPITULO
    AND t1.C_PROGRAMA = t5.C_PROGRAMA
LEFT JOIN #cr18_apertura_pacapr t2
    ON t1.C_PARTIDA = t2.C_PARTIDA
    AND t1.C_CAPITULO = t2.C_CAPITULO
    AND t1.C_PROGRAMA = t2.C_PROGRAMA
WHERE t1.AÑO = {int(Year)}
    AND t1.C_SCN NOT IN ('F22','F29','F3','F32','F41','F42','F5','F612','F7','F71','F79','K10','K11','K9')
    AND t2.NIVEL = 'REGIONAL'
    AND t1.C_SCN IN ('P51','P513')
    AND t5.C_COFOG_1 = '11.'
ORDER BY
    t1.SUBSECTOR,
    t1.MONEDA,
    t1.AÑO,
    t1.C_TIPO_CUENTA,
    t1.C_SCN,
    t1.C_CAGENTE,
    t1.C_TRANSACCION,
    t1.T_TRANSACCION,
    t1.N_CUENTA,
    t2.ACTIVIDAD,
    t2.NIVEL,
    t5.C_COFOG_1,
    t5.N_COFOG_1
"""
res = work_conn.execute(text(sql_nivel_fndr_a_dist_imput_perm_an))
_log("CREATE #nivel_fndr_a_dist_imput_perm_an", res.rowcount)


In [ ]:
work_conn.execute(text("DROP TABLE IF EXISTS #nivel_fndr_a_distrib_cfg_p5"))
sql_nivel_fndr_a_distrib_cfg_p5 = """
SELECT * INTO #nivel_fndr_a_distrib_cfg_p5 FROM #nivel_fndr_a_distrib
UNION ALL
SELECT * FROM #nivel_fndr_a_dist_imput_perm_an
"""
res = work_conn.execute(text(sql_nivel_fndr_a_distrib_cfg_p5))
_log("CREATE #nivel_fndr_a_distrib_cfg_p5", res.rowcount)


In [ ]:
# DROP TABLE NIVEL_FNDR_A_DISTRIB, NIVEL_FNDR_A_DIST_IMPUT_PERM_AN (limpieza de intermedias)
for t in ["#nivel_fndr_a_distrib", "#nivel_fndr_a_dist_imput_perm_an"]:
    work_conn.execute(text(f"DROP TABLE IF EXISTS {t}"))
